
# Bike Sharing Analytics

## Dashboard global reproducible

Universidad de La Salle  
Maestría en Inteligencia Artificial

Versión adaptada del dashboard académico original.

- Utiliza el dataset local del repositorio.
- Recupera las evidencias académicas originales.
- No requiere cargar archivos ZIP manualmente.
- No reentrena los modelos.
- Guarda resultados en results/reproduccion/dashboard.
- No modifica el dashboard publicado en GitHub Pages.


In [ ]:

# Localizar la raíz del repositorio.

from pathlib import Path
import sys
import hashlib

actual = Path.cwd().resolve()

candidatos = [actual, *actual.parents]

# Alternativa para Google Colab.

candidatos.append(
    Path("/content/Bike-Sharing-Analytics")
)

ROOT = next(
    (
        p for p in candidatos
        if (p / "data" / "day.csv").is_file()
        and (p / "requirements.txt").is_file()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Descarga o clona el repositorio antes de ejecutar."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.config import DATASET, RESULTS_DIR

HASH_ESPERADO = (
    "a6bcf826782d3c0fbfdcbeead17cd0884"
    "185a0dafe8ff10cd48a874ee7ba18be"
)

assert hashlib.sha256(
    DATASET.read_bytes()
).hexdigest() == HASH_ESPERADO

print("Repositorio:", ROOT)
print("Dataset:", DATASET)
print("Integridad del dataset: OK")



# Dashboard Global – Proyecto Integrador de Ciencia de Datos

## Bike Sharing Analytics

**Universidad de La Salle**

**Maestría en Inteligencia Artificial**

**Asignatura:** Ciencia de Datos

**Estudiante:** Adriana Patricia Fernández Infante

### Objetivo general

Desarrollar un dashboard interactivo que integre
los resultados de las cuatro fases del Proyecto
Integrador, facilitando la exploración, interpretación
y comunicación de los hallazgos relacionados con
la demanda de bicicletas.

### Alcance

- Fase 1: Planteamiento del problema, descripción y EDA.
- Fase 2: Análisis de regresión y visualización.
- Fase 3: Modelado avanzado, clustering, optimización
  e interpretabilidad.
- Fase 4: Preprocesamiento avanzado y modelos baseline.

### Tecnologías

- Python
- Pandas
- NumPy
- SciPy
- Scikit-learn
- Plotly
- Dash

### Fuente de datos

Bike Sharing Dataset – UCI Machine Learning Repository.

https://doi.org/10.24432/C5W894


In [ ]:

# ==========================================================
# DASHBOARD GLOBAL - CELDA 2
# Instalación de dependencias
# ==========================================================

# Dependencias instaladas mediante requirements.txt

# Importación de bibliotecas

import sys
import pandas as pd
import numpy as np
import scipy
import sklearn
import plotly
import dash
import dash_bootstrap_components as dbc

from pathlib import Path
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go

# Configuración de reproducibilidad

SEMILLA = 42
np.random.seed(SEMILLA)

# Directorios del proyecto

BASE = RESULTS_DIR / "reproduccion" / "dashboard"

DATOS = BASE / "datos"
EVIDENCIAS = BASE / "evidencias"
RESULTADOS = BASE / "resultados"

for carpeta in [DATOS, EVIDENCIAS, RESULTADOS]:
    carpeta.mkdir(parents=True, exist_ok=True)

# Verificación del entorno

print("=" * 60)
print("DASHBOARD GLOBAL - ENTORNO DE EJECUCIÓN")
print("=" * 60)

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Plotly:", plotly.__version__)
print("Dash:", dash.__version__)
print("Dash Bootstrap:", dbc.__version__)
print("Semilla:", SEMILLA)

print("\n=== DIRECTORIOS DEL PROYECTO ===")

for carpeta in [BASE, DATOS, EVIDENCIAS, RESULTADOS]:
    print("OK:", carpeta)

print("\nEntorno configurado correctamente.")


In [ ]:

# ==========================================================
# DASHBOARD GLOBAL - CELDA 3
# Carga y validación del dataset original
# ==========================================================

import urllib.request
import zipfile
import hashlib

# 1. Fuente oficial del dataset

URL_DATASET = (
    "https://archive.ics.uci.edu/static/public/"
    "275/bike+sharing+dataset.zip"
)

RUTA_ZIP = DATOS / "bike_sharing_dataset.zip"
RUTA_CSV = DATOS / "day.csv"


# Recuperar el dataset y su documentación local.

import shutil

shutil.copy2(DATASET, RUTA_CSV)

documentaciones = list(
    (ROOT / "docs").rglob("documentacion_original.txt")
)

assert len(documentaciones) == 1, (
    "No se encontró una documentación original única."
)

ruta_documentacion = documentaciones[0]

# Reconstruir un ZIP local compatible con las
# celdas originales de documentación del dashboard.
# No corresponde al ZIP original descargado de UCI.

with zipfile.ZipFile(
    RUTA_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archivo:

    archivo.write(RUTA_CSV, arcname="day.csv")

    archivo.write(
        ruta_documentacion,
        arcname="Readme.txt"
    )

print("Dataset local disponible.")
print("ZIP auxiliar reconstruido localmente.")

# 3. Verificar identidad del archivo

sha256 = hashlib.sha256(
    RUTA_CSV.read_bytes()
).hexdigest()

SHA256_ESPERADO = (
    "a6bcf826782d3c0fbfdcbeead17cd0884"
    "185a0dafe8ff10cd48a874ee7ba18be"
)

assert sha256 == SHA256_ESPERADO, (
    "El dataset no coincide con el utilizado en la Fase 4."
)

# 4. Cargar los datos originales

df_original = pd.read_csv(RUTA_CSV)

# Crear copia de trabajo para el dashboard

df = df_original.copy()

df["dteday"] = pd.to_datetime(df["dteday"])

# 5. Validar estructura

assert df.shape == (731, 16), (
    "Dimensiones inesperadas del dataset."
)

columnas_esperadas = [
    "instant", "dteday", "season", "yr",
    "mnth", "holiday", "weekday", "workingday",
    "weathersit", "temp", "atemp", "hum",
    "windspeed", "casual", "registered", "cnt"
]

assert list(df.columns) == columnas_esperadas

# 6. Diagnóstico inicial de calidad

total_faltantes = df.isna().sum().sum()

total_duplicados = df.duplicated().sum()

fechas_duplicadas = df["dteday"].duplicated().sum()

inconsistencias = (
    df["cnt"] != df["casual"] + df["registered"]
).sum()

# 7. Presentar resultados de validación

print("\n" + "=" * 60)
print("DASHBOARD GLOBAL - VALIDACIÓN DEL DATASET")
print("=" * 60)

print("Dataset: Bike Sharing")
print("Fuente: UCI Machine Learning Repository")
print("Archivo:", RUTA_CSV.name)
print("SHA-256:", sha256)
print("Identidad con Fase 4: OK")

print("\n=== DIMENSIONES ===")
print("Registros:", df.shape[0])
print("Variables:", df.shape[1])

print("\n=== PERIODO DE ANÁLISIS ===")
print("Fecha inicial:", df["dteday"].min().date())
print("Fecha final:", df["dteday"].max().date())

print("\n=== CALIDAD DE DATOS ===")
print("Valores faltantes:", total_faltantes)
print("Registros duplicados:", total_duplicados)
print("Fechas duplicadas:", fechas_duplicadas)
print("Inconsistencias en demanda:", inconsistencias)

print("\n=== TIPOS DE DATOS ===")
display(df.dtypes.to_frame("Tipo de dato"))

print("\n=== PRIMEROS CINCO REGISTROS ===")
display(df.head())

print("\nDataset cargado y validado correctamente.")


In [ ]:

# ==========================================================
# DASHBOARD GLOBAL - CELDA 4
# Recuperación local de evidencias originales
# ==========================================================

import shutil
import zipfile
import io

inventario = []

# Recuperar la estructura original de evidencias
# distribuida entre notebooks, docs y results.

for grupo in ["Fases_2_3", "Fase_4"]:

    destino = EVIDENCIAS / grupo

    destino.mkdir(
        parents=True,
        exist_ok=True
    )

    for carpeta in ["notebooks", "docs", "results"]:

        origen = (
            ROOT / carpeta /
            "evidencias" / grupo
        )

        if not origen.exists():
            continue

        for archivo in sorted(origen.rglob("*")):

            if not archivo.is_file():
                continue

            ruta_relativa = archivo.relative_to(origen)

            ruta_destino = destino / ruta_relativa

            ruta_destino.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            if ruta_destino.exists():

                assert (
                    ruta_destino.read_bytes()
                    == archivo.read_bytes()
                ), f"Conflicto de evidencias: {ruta_destino}"

            else:

                shutil.copy2(
                    archivo,
                    ruta_destino
                )

    for ruta in sorted(destino.rglob("*")):

        if ruta.is_file():

            inventario.append({
                "Grupo": grupo,
                "Archivo": ruta.name,
                "Formato": ruta.suffix.lower(),
                "Tamaño_KB": round(
                    ruta.stat().st_size / 1024,
                    2
                ),
                "Ruta": str(ruta)
            })

df_inventario = pd.DataFrame(inventario)

assert not df_inventario.empty

# Verificar los notebooks originales necesarios.

for nombre in [
    "Proyecto_Integrador_Bicicletas_Final.ipynb",
    "Fase_3_Modelado_Avanzado_Bicicletas.ipynb"
]:

    encontrados = list(
        EVIDENCIAS.rglob(nombre)
    )

    assert len(encontrados) == 1, nombre

# Verificar evidencias de Fase 4.

origen_f4 = EVIDENCIAS / "Fase_4"

assert len(list(origen_f4.rglob("*.csv"))) == 19
assert len(list(origen_f4.rglob("*.png"))) == 12
assert len(list(origen_f4.rglob("*.json"))) == 2

resumen_evidencias = pd.crosstab(
    df_inventario["Grupo"],
    df_inventario["Formato"],
    margins=True,
    margins_name="Total"
)

df_inventario.to_csv(
    RESULTADOS / "inventario_evidencias.csv",
    index=False
)

display(resumen_evidencias)

print("Evidencias locales recuperadas:", len(df_inventario))
print("Notebooks originales: OK")
print("Evidencias Fase 4: OK")
print("Carga manual de archivos: no requerida")


In [ ]:

# ==========================================================
# DASHBOARD GLOBAL - CELDA 5
# Construcción de indicadores globales (KPI)
# ==========================================================

# 1. Indicadores generales del dataset

total_registros = len(df)
total_variables = df.shape[1]

total_alquileres = int(df["cnt"].sum())

promedio_diario = df["cnt"].mean()

mediana_diaria = df["cnt"].median()

demanda_maxima = int(df["cnt"].max())

demanda_minima = int(df["cnt"].min())

fecha_maxima = df.loc[
    df["cnt"].idxmax(), "dteday"
].date()

fecha_minima = df.loc[
    df["cnt"].idxmin(), "dteday"
].date()

# 2. Indicadores de calidad de datos

total_celdas = df.size

total_faltantes = int(df.isna().sum().sum())

porcentaje_completitud = (
    1 - total_faltantes / total_celdas
) * 100

total_duplicados = int(df.duplicated().sum())

# 3. Distribución de usuarios

total_casuales = int(df["casual"].sum())

total_registrados = int(df["registered"].sum())

porcentaje_casuales = (
    total_casuales / total_alquileres
) * 100

porcentaje_registrados = (
    total_registrados / total_alquileres
) * 100

# 4. Evolución anual de la demanda

demanda_anual = (
    df.assign(Año=df["dteday"].dt.year)
    .groupby("Año", as_index=False)
    .agg(
        Alquileres=("cnt", "sum"),
        Promedio_diario=("cnt", "mean"),
        Usuarios_casuales=("casual", "sum"),
        Usuarios_registrados=("registered", "sum"),
        Dias_observados=("cnt", "size")
    )
)

demanda_2011 = demanda_anual.loc[
    demanda_anual["Año"] == 2011,
    "Alquileres"
].iloc[0]

demanda_2012 = demanda_anual.loc[
    demanda_anual["Año"] == 2012,
    "Alquileres"
].iloc[0]

crecimiento_anual = (
    (demanda_2012 / demanda_2011) - 1
) * 100

# 5. Evolución mensual de la demanda

demanda_mensual = (
    df.assign(
        Mes=df["dteday"].dt.to_period("M")
    )
    .groupby("Mes", as_index=False)
    .agg(
        Alquileres=("cnt", "sum"),
        Promedio_diario=("cnt", "mean"),
        Casuales=("casual", "sum"),
        Registrados=("registered", "sum")
    )
)

demanda_mensual["Mes"] = (
    demanda_mensual["Mes"].dt.to_timestamp()
)

# 6. Consolidar indicadores globales

kpi_globales = {
    "Registros": total_registros,
    "Variables originales": total_variables,
    "Total de alquileres": total_alquileres,
    "Promedio diario": round(promedio_diario, 2),
    "Mediana diaria": round(mediana_diaria, 2),
    "Demanda máxima diaria": demanda_maxima,
    "Fecha demanda máxima": str(fecha_maxima),
    "Demanda mínima diaria": demanda_minima,
    "Fecha demanda mínima": str(fecha_minima),
    "Completitud (%)": round(
        porcentaje_completitud, 2
    ),
    "Valores faltantes": total_faltantes,
    "Registros duplicados": total_duplicados,
    "Alquileres casuales": total_casuales,
    "Alquileres registrados": total_registrados,
    "Participación casuales (%)": round(
        porcentaje_casuales, 2
    ),
    "Participación registrados (%)": round(
        porcentaje_registrados, 2
    ),
    "Crecimiento anual (%)": round(
        crecimiento_anual, 2
    )
}

df_kpi_globales = pd.DataFrame(
    list(kpi_globales.items()),
    columns=["Indicador", "Valor"]
)

# 7. Validaciones de integridad

assert total_alquileres == (
    total_casuales + total_registrados
)

assert demanda_anual["Alquileres"].sum() == (
    total_alquileres
)

assert demanda_mensual["Alquileres"].sum() == (
    total_alquileres
)

assert len(demanda_mensual) == 24

assert np.isclose(
    porcentaje_casuales + porcentaje_registrados,
    100
)

# 8. Guardar resultados para el dashboard

df_kpi_globales.to_csv(
    RESULTADOS / "kpi_globales.csv",
    index=False
)

demanda_anual.to_csv(
    RESULTADOS / "demanda_anual.csv",
    index=False
)

demanda_mensual.to_csv(
    RESULTADOS / "demanda_mensual.csv",
    index=False
)

# 9. Presentar resultados

print("=" * 60)
print("DASHBOARD GLOBAL - INDICADORES PRINCIPALES")
print("=" * 60)

display(df_kpi_globales)

print("\n=== EVOLUCIÓN ANUAL DE LA DEMANDA ===")

display(demanda_anual.round(2))

print("\n=== EVOLUCIÓN MENSUAL DE LA DEMANDA ===")

display(demanda_mensual.round(2))

print("\n=== VALIDACIONES ===")

print("Integridad de alquileres: OK")
print("Consolidación anual: OK")
print("Consolidación mensual: OK")
print("Distribución de usuarios: OK")

print("\nIndicadores globales generados correctamente.")


In [ ]:

# ==========================================================
# DASHBOARD GLOBAL - CELDA 6
# FASE 1: Análisis Exploratorio de Datos (EDA)
# ==========================================================

from scipy.stats import chi2_contingency
from itertools import combinations

# 1. Clasificación de variables según su naturaleza

variables_cuantitativas = [
    "temp",
    "atemp",
    "hum",
    "windspeed",
    "casual",
    "registered",
    "cnt"
]

variables_cualitativas = [
    "season",
    "yr",
    "mnth",
    "holiday",
    "weekday",
    "workingday",
    "weathersit"
]

# instant es un identificador.
# dteday es una variable temporal.
# Se excluyen de las estadísticas cuantitativas.

# ==========================================================
# 2. ANÁLISIS UNIVARIADO CUANTITATIVO
# ==========================================================

datos_numericos = df[variables_cuantitativas]

estadisticas_eda = pd.DataFrame({
    "Observaciones": datos_numericos.count(),
    "Media": datos_numericos.mean(),
    "Mediana": datos_numericos.median(),
    "Moda": datos_numericos.mode().iloc[0],
    "Desviacion_estandar": datos_numericos.std(),
    "Varianza": datos_numericos.var(),
    "Minimo": datos_numericos.min(),
    "Maximo": datos_numericos.max(),
    "Rango": (
        datos_numericos.max() -
        datos_numericos.min()
    ),
    "Q1": datos_numericos.quantile(0.25),
    "Q3": datos_numericos.quantile(0.75),
    "IQR": (
        datos_numericos.quantile(0.75) -
        datos_numericos.quantile(0.25)
    ),
    "Asimetria": datos_numericos.skew(),
    "Curtosis": datos_numericos.kurt()
})

estadisticas_eda.index.name = "Variable"

estadisticas_eda = (
    estadisticas_eda.reset_index()
)

# ==========================================================
# 3. ANÁLISIS UNIVARIADO CUALITATIVO
# ==========================================================

frecuencias = []

for variable in variables_cualitativas:

    conteos = (
        df[variable]
        .value_counts()
        .sort_index()
    )

    for categoria, cantidad in conteos.items():

        frecuencias.append({
            "Variable": variable,
            "Categoria": categoria,
            "Frecuencia": int(cantidad),
            "Porcentaje": round(
                cantidad / len(df) * 100, 4
            )
        })

df_frecuencias = pd.DataFrame(frecuencias)

# ==========================================================
# 4. ANÁLISIS DE CORRELACIONES
# ==========================================================

correlacion_pearson = (
    datos_numericos.corr(method="pearson")
)

correlacion_spearman = (
    datos_numericos.corr(method="spearman")
)

# Correlaciones de factores meteorológicos
# con la demanda total.

correlacion_demanda = pd.DataFrame({
    "Pearson": correlacion_pearson["cnt"],
    "Spearman": correlacion_spearman["cnt"]
})

correlacion_demanda = (
    correlacion_demanda
    .drop(index=["cnt", "casual", "registered"])
    .sort_values("Pearson", ascending=False)
)

# ==========================================================
# 5. PRUEBAS DE INDEPENDENCIA CHI-CUADRADO
# ==========================================================

# Se seleccionan pares con relevancia para el dominio.
# No se prueban relaciones deterministas del calendario,
# como mes frente a estación.

pares_chi2 = [
    ("season", "weathersit"),
    ("workingday", "weathersit"),
    ("yr", "weathersit"),
    ("weekday", "weathersit")
]

resultados_chi2 = []

for variable_1, variable_2 in pares_chi2:

    tabla_contingencia = pd.crosstab(
        df[variable_1],
        df[variable_2]
    )

    chi2, p_valor, gl, esperadas = (
        chi2_contingency(
            tabla_contingencia,
            correction=False
        )
    )

    n = tabla_contingencia.to_numpy().sum()

    filas, columnas = tabla_contingencia.shape

    # V de Cramér: magnitud de la asociación.

    v_cramer = np.sqrt(
        chi2 / (
            n * min(filas - 1, columnas - 1)
        )
    )

    porcentaje_esperadas_menores_5 = (
        np.mean(esperadas < 5) * 100
    )

    supuestos_validos = (
        esperadas.min() >= 1
        and porcentaje_esperadas_menores_5 <= 20
    )

    resultados_chi2.append({
        "Variable_1": variable_1,
        "Variable_2": variable_2,
        "Chi2": chi2,
        "Grados_libertad": gl,
        "P_valor": p_valor,
        "V_Cramer": v_cramer,
        "Esperada_minima": esperadas.min(),
        "Esperadas_menores_5_pct":
            porcentaje_esperadas_menores_5,
        "Frecuencias_esperadas_validas":
            supuestos_validos
    })

df_chi2 = pd.DataFrame(resultados_chi2)

# Corrección de Bonferroni por pruebas múltiples.

df_chi2["P_ajustado"] = (
    df_chi2["P_valor"] * len(pares_chi2)
).clip(upper=1)

df_chi2["Conclusion"] = np.where(
    ~df_chi2["Frecuencias_esperadas_validas"],
    "Interpretar con precaución",
    np.where(
        df_chi2["P_ajustado"] < 0.05,
        "Evidencia de asociación",
        "No se rechaza independencia"
    )
)

# ==========================================================
# 6. DETECCIÓN DE OUTLIERS MEDIANTE IQR
# ==========================================================

resultados_outliers = []

for variable in variables_cuantitativas:

    q1 = df[variable].quantile(0.25)
    q3 = df[variable].quantile(0.75)

    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    mascara = (
        (df[variable] < limite_inferior) |
        (df[variable] > limite_superior)
    )

    cantidad = int(mascara.sum())

    resultados_outliers.append({
        "Variable": variable,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Limite_inferior": limite_inferior,
        "Limite_superior": limite_superior,
        "Outliers": cantidad,
        "Porcentaje": cantidad / len(df) * 100
    })

df_outliers_eda = pd.DataFrame(
    resultados_outliers
)

# ==========================================================
# 7. DIAGNÓSTICO DE CALIDAD POR VARIABLE
# ==========================================================

df_calidad_eda = pd.DataFrame({
    "Variable": df.columns,
    "Tipo_dato": df.dtypes.astype(str).values,
    "Valores_faltantes": df.isna().sum().values,
    "Porcentaje_faltantes": (
        df.isna().mean().values * 100
    ),
    "Valores_unicos": df.nunique().values
})

# ==========================================================
# 8. VALIDACIONES DE INTEGRIDAD
# ==========================================================

assert len(df) == 731
assert df.shape[1] == 16

assert len(estadisticas_eda) == 7

assert (
    df_frecuencias.groupby("Variable")
    ["Frecuencia"].sum() == len(df)
).all()

assert len(df_calidad_eda) == 16

assert np.isfinite(
    correlacion_pearson.to_numpy()
).all()

assert np.isfinite(
    correlacion_spearman.to_numpy()
).all()

# ==========================================================
# 9. GUARDAR RESULTADOS DE LA FASE 1
# ==========================================================

FASE1 = RESULTADOS / "Fase_1"
FASE1.mkdir(parents=True, exist_ok=True)

tablas_fase1 = {
    "estadisticas_descriptivas": estadisticas_eda,
    "frecuencias_categoricas": df_frecuencias,
    "pruebas_chi2": df_chi2,
    "outliers_eda": df_outliers_eda,
    "calidad_datos": df_calidad_eda
}

for nombre, tabla in tablas_fase1.items():

    tabla.to_csv(
        FASE1 / f"{nombre}.csv",
        index=False
    )

correlacion_pearson.to_csv(
    FASE1 / "correlacion_pearson.csv"
)

correlacion_spearman.to_csv(
    FASE1 / "correlacion_spearman.csv"
)

correlacion_demanda.to_csv(
    FASE1 / "correlacion_demanda.csv"
)

# ==========================================================
# 10. PRESENTACIÓN DE RESULTADOS
# ==========================================================

print("=" * 60)
print("FASE 1 - ANÁLISIS EXPLORATORIO DE DATOS")
print("=" * 60)

print("\n=== ESTADÍSTICAS DESCRIPTIVAS ===")
display(estadisticas_eda.round(4))

print("\n=== FRECUENCIAS CATEGÓRICAS ===")
display(df_frecuencias)

print("\n=== CORRELACIÓN DE PEARSON ===")
display(correlacion_pearson.round(4))

print("\n=== CORRELACIÓN DE SPEARMAN ===")
display(correlacion_spearman.round(4))

print("\n=== FACTORES ASOCIADOS A LA DEMANDA ===")
display(correlacion_demanda.round(4))

print("\n=== PRUEBAS CHI-CUADRADO ===")
display(df_chi2.round(6))

print("\n=== DETECCIÓN DE OUTLIERS ===")
display(df_outliers_eda.round(4))

print("\n=== CALIDAD DE DATOS ===")
display(df_calidad_eda)

print("\n=== VALIDACIONES ===")
print("Estadísticas descriptivas: OK")
print("Frecuencias categóricas: OK")
print("Matrices de correlación: OK")
print("Integridad del dataset: OK")

print("\nArchivos generados:",
      len(list(FASE1.glob("*.csv"))))

print("Fase 1 - Resultados EDA generados correctamente.")


In [ ]:

# ==========================================================
# DASHBOARD GLOBAL - CELDA 7
# FASE 1: Visualizaciones interactivas con Plotly
# ==========================================================

# 1. Configuración visual del dashboard

import plotly.io as pio

pio.templates.default = "plotly_white"

COLORES = {
    "principal": "#2563EB",
    "secundario": "#0D9488",
    "acento": "#F59E0B",
    "negativo": "#DC2626",
    "fondo": "#F8FAFC",
    "texto": "#1E293B"
}

# Diccionario de etiquetas para facilitar la interpretación

ETIQUETAS = {
    "temp": "Temperatura normalizada",
    "atemp": "Sensación térmica normalizada",
    "hum": "Humedad normalizada",
    "windspeed": "Velocidad del viento normalizada",
    "casual": "Alquileres casuales",
    "registered": "Alquileres registrados",
    "cnt": "Demanda total",
    "dteday": "Fecha",
    "season": "Estación",
    "weathersit": "Condición meteorológica"
}

# Copia exclusiva para visualización.
# No se modifica el dataset original.

df_visual = df.copy()

df_visual["Estacion"] = df_visual["season"].map({
    1: "Invierno",
    2: "Primavera",
    3: "Verano",
    4: "Otoño"
})

df_visual["Clima"] = df_visual["weathersit"].map({
    1: "Despejado o parcialmente nublado",
    2: "Niebla o nublado",
    3: "Lluvia o nieve ligera",
    4: "Condiciones severas"
})

df_visual["Año"] = (
    df_visual["dteday"].dt.year.astype(str)
)

# Diccionario que almacenará las figuras de la Fase 1

figuras_fase1 = {}

# ==========================================================
# 2. HISTOGRAMA DE LA DEMANDA
# ==========================================================

figuras_fase1["histograma_demanda"] = px.histogram(
    df_visual,
    x="cnt",
    nbins=30,
    marginal="box",
    title="Distribución de la demanda diaria",
    labels=ETIQUETAS,
    color_discrete_sequence=[COLORES["principal"]]
)

figuras_fase1["histograma_demanda"].add_vline(
    x=df["cnt"].mean(),
    line_dash="dash",
    line_color=COLORES["negativo"],
    annotation_text="Media"
)

# ==========================================================
# 3. BOXPLOT DE VARIABLES METEOROLÓGICAS
# ==========================================================

variables_clima = [
    "temp", "atemp", "hum", "windspeed"
]

df_clima_largo = df_visual.melt(
    value_vars=variables_clima,
    var_name="Variable",
    value_name="Valor"
)

df_clima_largo["Variable"] = (
    df_clima_largo["Variable"].map(ETIQUETAS)
)

figuras_fase1["boxplot_clima"] = px.box(
    df_clima_largo,
    x="Variable",
    y="Valor",
    color="Variable",
    points="outliers",
    title="Distribución y outliers meteorológicos"
)

figuras_fase1["boxplot_clima"].update_layout(
    showlegend=False
)

# ==========================================================
# 4. MAPAS DE CALOR DE CORRELACIONES
# ==========================================================

figuras_fase1["correlacion_pearson"] = px.imshow(
    correlacion_pearson,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="Matriz de correlación de Pearson"
)

figuras_fase1["correlacion_spearman"] = px.imshow(
    correlacion_spearman,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="Matriz de correlación de Spearman"
)

# ==========================================================
# 5. DIAGRAMA DE DISPERSIÓN
# ==========================================================

figuras_fase1["dispersion_temperatura"] = px.scatter(
    df_visual,
    x="temp",
    y="cnt",
    color="Año",
    hover_data=[
        "dteday",
        "hum",
        "windspeed",
        "Estacion"
    ],
    labels=ETIQUETAS,
    opacity=0.65,
    title="Relación entre temperatura y demanda"
)

# ==========================================================
# 6. DEMANDA POR ESTACIÓN
# ==========================================================

figuras_fase1["demanda_estacion"] = px.box(
    df_visual,
    x="Estacion",
    y="cnt",
    color="Estacion",
    points="outliers",
    labels=ETIQUETAS,
    category_orders={
        "Estacion": [
            "Invierno",
            "Primavera",
            "Verano",
            "Otoño"
        ]
    },
    title="Distribución de la demanda por estación"
)

figuras_fase1["demanda_estacion"].update_layout(
    showlegend=False
)

# ==========================================================
# 7. DEMANDA SEGÚN CONDICIÓN METEOROLÓGICA
# ==========================================================

demanda_clima = (
    df_visual.groupby("Clima", as_index=False)
    .agg(
        Demanda_promedio=("cnt", "mean"),
        Demanda_total=("cnt", "sum"),
        Dias_observados=("cnt", "size")
    )
    .sort_values(
        "Demanda_promedio",
        ascending=False
    )
)

figuras_fase1["demanda_clima"] = px.bar(
    demanda_clima,
    x="Clima",
    y="Demanda_promedio",
    color="Clima",
    text_auto=".0f",
    hover_data=[
        "Demanda_total",
        "Dias_observados"
    ],
    title="Demanda diaria promedio según el clima"
)

figuras_fase1["demanda_clima"].update_layout(
    showlegend=False
)

# ==========================================================
# 8. COMPARACIÓN DE OUTLIERS
# ==========================================================

df_outliers_visual = (
    df_outliers_eda.copy()
    .sort_values("Outliers", ascending=True)
)

figuras_fase1["outliers"] = px.bar(
    df_outliers_visual,
    x="Outliers",
    y="Variable",
    orientation="h",
    text="Outliers",
    hover_data=["Porcentaje"],
    color_discrete_sequence=[COLORES["acento"]],
    title="Valores atípicos detectados mediante IQR"
)

# ==========================================================
# 9. FACTORES ASOCIADOS A LA DEMANDA
# ==========================================================

df_correlacion_visual = (
    correlacion_demanda
    .reset_index()
    .melt(
        id_vars="Variable",
        var_name="Metodo",
        value_name="Correlacion"
    )
)

figuras_fase1["factores_demanda"] = px.bar(
    df_correlacion_visual,
    x="Variable",
    y="Correlacion",
    color="Metodo",
    barmode="group",
    text_auto=".3f",
    title="Correlación de factores meteorológicos"
)

figuras_fase1["factores_demanda"].add_hline(
    y=0,
    line_color="#64748B"
)

# ==========================================================
# 10. PRUEBAS DE INDEPENDENCIA CHI-CUADRADO
# ==========================================================

df_chi2_visual = df_chi2.copy()

df_chi2_visual["Relacion"] = (
    df_chi2_visual["Variable_1"]
    + " vs. "
    + df_chi2_visual["Variable_2"]
)

figuras_fase1["chi_cuadrado"] = px.bar(
    df_chi2_visual,
    x="Relacion",
    y="P_ajustado",
    color="Conclusion",
    text_auto=".3f",
    hover_data=[
        "P_valor",
        "Chi2",
        "V_Cramer",
        "Frecuencias_esperadas_validas"
    ],
    title="Pruebas Chi-cuadrado: p-valores ajustados"
)

figuras_fase1["chi_cuadrado"].add_hline(
    y=0.05,
    line_dash="dash",
    line_color=COLORES["negativo"],
    annotation_text="Significancia: 0.05"
)

# ==========================================================
# 11. CALIDAD Y COMPLETITUD DE DATOS
# ==========================================================

df_calidad_visual = df_calidad_eda.copy()

df_calidad_visual["Completitud"] = (
    100 - df_calidad_visual["Porcentaje_faltantes"]
)

figuras_fase1["calidad_datos"] = px.bar(
    df_calidad_visual,
    x="Completitud",
    y="Variable",
    orientation="h",
    text_auto=".0f",
    color_discrete_sequence=[COLORES["secundario"]],
    title="Porcentaje de completitud por variable"
)

figuras_fase1["calidad_datos"].update_xaxes(
    range=[0, 110],
    ticksuffix="%"
)

# ==========================================================
# 12. APLICAR ESTILO VISUAL UNIFICADO
# ==========================================================

for nombre, figura in figuras_fase1.items():

    figura.update_layout(
        template="plotly_white",
        font=dict(
            family="Arial",
            size=12,
            color=COLORES["texto"]
        ),
        title_font_size=17,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(
            l=45,
            r=35,
            t=70,
            b=65
        ),
        height=450,
        hoverlabel=dict(
            bgcolor="white",
            font_size=12
        )
    )

# ==========================================================
# 13. GUARDAR FIGURAS INTERACTIVAS
# ==========================================================

CARPETA_GRAFICAS_F1 = FASE1 / "graficas"

CARPETA_GRAFICAS_F1.mkdir(
    parents=True,
    exist_ok=True
)

for nombre, figura in figuras_fase1.items():

    figura.write_html(
        str(CARPETA_GRAFICAS_F1 / f"{nombre}.html"),
        include_plotlyjs=True
    )

    figura.write_json(
        str(CARPETA_GRAFICAS_F1 / f"{nombre}.json")
    )

# ==========================================================
# 14. VALIDACIONES
# ==========================================================

assert len(figuras_fase1) == 11

assert all(
    len(figura.data) > 0
    for figura in figuras_fase1.values()
)

assert len(
    list(CARPETA_GRAFICAS_F1.glob("*.html"))
) == 11

assert len(
    list(CARPETA_GRAFICAS_F1.glob("*.json"))
) == 11

# ==========================================================
# 15. PRESENTACIÓN DE RESULTADOS
# ==========================================================

print("=" * 60)
print("FASE 1 - VISUALIZACIONES INTERACTIVAS")
print("=" * 60)

print("\nGráficas generadas:", len(figuras_fase1))

for numero, nombre in enumerate(figuras_fase1, 1):

    print(f"{numero:02d}. {nombre}: OK")

print("\nArchivos HTML:", len(
    list(CARPETA_GRAFICAS_F1.glob("*.html"))
))

print("Archivos JSON:", len(
    list(CARPETA_GRAFICAS_F1.glob("*.json"))
))

print("\n=== DEMANDA PROMEDIO SEGÚN EL CLIMA ===")

display(demanda_clima.round(2))

print("\n=== VISTA PREVIA DE GRÁFICAS ===")

# Mostrar tres figuras para verificar su funcionamiento.

figuras_fase1["histograma_demanda"].show()

figuras_fase1["correlacion_pearson"].show()

figuras_fase1["dispersion_temperatura"].show()

print("\nFase 1 - Visualizaciones generadas correctamente.")


In [ ]:
# ==========================================================
# DASHBOARD GLOBAL - CELDA 8
# FASE 2: Recuperación de resultados originales
# ==========================================================

import json
import re
from pathlib import Path

BASE_DASHBOARD = BASE

# 1. Localizar el notebook original de la Fase 2

nombre_notebook_f2 = "Proyecto_Integrador_Bicicletas_Final.ipynb"

archivos_f2 = list(
    BASE_DASHBOARD.rglob(nombre_notebook_f2)
)

assert len(archivos_f2) == 1, (
    f"Se esperaba un notebook de Fase 2. "
    f"Encontrados: {len(archivos_f2)}"
)

ruta_notebook_f2 = archivos_f2[0]

# 2. Crear directorio para resultados de la Fase 2

FASE2 = BASE_DASHBOARD / "resultados" / "Fase_2"

FASE2.mkdir(parents=True, exist_ok=True)

# 3. Leer el notebook sin ejecutar su código

with open(ruta_notebook_f2, "r", encoding="utf-8") as archivo:
    notebook_f2 = json.load(archivo)

celdas_f2 = notebook_f2.get("cells", [])

# 4. Identificar las celdas y sus salidas

inventario_f2 = []
salidas_f2 = []

for numero, celda in enumerate(celdas_f2, start=1):

    tipo = celda.get("cell_type", "")

    fuente = celda.get("source", "")

    if isinstance(fuente, list):
        fuente = "".join(fuente)

    outputs = celda.get("outputs", [])

    inventario_f2.append({
        "Celda": numero,
        "Tipo": tipo,
        "Ejecucion": celda.get("execution_count"),
        "Cantidad_salidas": len(outputs),
        "Contenido": fuente[:180].replace("\n", " ")
    })

    # Recuperar resultados textuales almacenados

    for salida in outputs:

        texto = salida.get("text", "")

        if not texto:
            texto = salida.get("data", {}).get(
                "text/plain", ""
            )

        if isinstance(texto, list):
            texto = "".join(texto)

        if texto:
            salidas_f2.append({
                "Celda": numero,
                "Tipo_salida": salida.get("output_type"),
                "Resultado": texto
            })

# 5. Consolidar los resultados

df_inventario_f2 = pd.DataFrame(inventario_f2)

df_salidas_f2 = pd.DataFrame(
    salidas_f2,
    columns=["Celda", "Tipo_salida", "Resultado"]
)

# 6. Identificar resultados relevantes para regresión

patron_metricas = (
    r"\b(?:MAE|MSE|RMSE|R2|R²|R\^2|MAPE|"
    r"coeficiente|regresión|regresion|regression|"
    r"residuos|residuals|intercepto|accuracy)\b"
)

df_metricas_detectadas_f2 = df_salidas_f2[
    df_salidas_f2["Resultado"].str.contains(
        patron_metricas,
        case=False,
        regex=True,
        na=False
    )
].copy()

# 7. Guardar evidencias recuperadas

df_inventario_f2.to_csv(
    FASE2 / "inventario_notebook_fase2.csv",
    index=False,
    encoding="utf-8-sig"
)

df_salidas_f2.to_csv(
    FASE2 / "salidas_originales_fase2.csv",
    index=False,
    encoding="utf-8-sig"
)

df_metricas_detectadas_f2.to_csv(
    FASE2 / "metricas_detectadas_fase2.csv",
    index=False,
    encoding="utf-8-sig"
)

# 8. Presentar resultados

print("=" * 60)
print("FASE 2 - RECUPERACIÓN DE RESULTADOS ORIGINALES")
print("=" * 60)

print("\nNotebook:", ruta_notebook_f2.name)

print("Total de celdas:", len(celdas_f2))

print("Celdas de código:", sum(
    c.get("cell_type") == "code"
    for c in celdas_f2
))

print("Celdas Markdown:", sum(
    c.get("cell_type") == "markdown"
    for c in celdas_f2
))

print("Salidas textuales recuperadas:", len(df_salidas_f2))

print("Salidas con posibles métricas:",
      len(df_metricas_detectadas_f2))

print("\n=== INVENTARIO DE CELDAS ===")

display(df_inventario_f2)

print("\n=== RESULTADOS DE REGRESIÓN DETECTADOS ===")

for _, fila in df_metricas_detectadas_f2.iterrows():

    print("\n" + "-" * 60)
    print("Celda:", fila["Celda"])
    print("-" * 60)
    print(fila["Resultado"][:6000])

# 9. Validaciones

assert len(celdas_f2) > 0

assert len(df_salidas_f2) > 0, (
    "El notebook no contiene resultados textuales guardados."
)

print("\n=== VALIDACIONES ===")
print("Lectura del notebook: OK")
print("Recuperación de salidas: OK")
print("Archivos CSV generados: 3")
print("Modelos reentrenados: 0")

print("\nFase 2 - Recuperación de evidencias finalizada.")

In [ ]:
# ==========================================================
# DASHBOARD GLOBAL - CELDA 9
# FASE 2: Consolidación y visualizaciones
# ==========================================================

import base64
import re

# ==========================================================
# 1. EXTRAER MÉTRICAS DESDE LAS SALIDAS ORIGINALES
# ==========================================================

def recuperar_texto_f2(numero_celda):

    return "\n".join(
        df_salidas_f2.loc[
            df_salidas_f2["Celda"] == numero_celda,
            "Resultado"
        ].tolist()
    )

texto_validacion_f2 = recuperar_texto_f2(19)
texto_prueba_f2 = recuperar_texto_f2(21)
texto_r_f2 = recuperar_texto_f2(30)

# Extraer resultados de regresión mediante expresiones
# regulares, evitando introducir métricas manualmente.

numero = r"(-?\d+(?:\.\d+)?)"

patron_modelos = (
    r"(Regresión lineal (?:simple|múltiple))"
    + r"\s+" + numero
    + r"\s+" + numero
    + r"\s+" + numero
)

coincidencias_modelos = re.findall(
    patron_modelos,
    texto_validacion_f2
)

assert len(coincidencias_modelos) == 2, (
    "No se recuperaron los dos modelos de validación."
)

df_validacion_f2 = pd.DataFrame(
    coincidencias_modelos,
    columns=["Modelo", "MAE", "RMSE", "R2"]
)

for columna in ["MAE", "RMSE", "R2"]:

    df_validacion_f2[columna] = (
        df_validacion_f2[columna].astype(float)
    )

# Recuperar evaluación final del modelo seleccionado.

patron_evaluacion = (
    r"(Validación|Prueba)"
    + r"\s+" + numero
    + r"\s+" + numero
    + r"\s+" + numero
)

coincidencias_evaluacion = re.findall(
    patron_evaluacion,
    texto_prueba_f2
)

assert len(coincidencias_evaluacion) == 2

df_evaluacion_f2 = pd.DataFrame(
    coincidencias_evaluacion,
    columns=["Conjunto", "MAE", "RMSE", "R2"]
)

for columna in ["MAE", "RMSE", "R2"]:

    df_evaluacion_f2[columna] = (
        df_evaluacion_f2[columna].astype(float)
    )

# ==========================================================
# 2. IDENTIFICAR EL MODELO SELECCIONADO
# ==========================================================

modelo_seleccionado_f2 = (
    df_validacion_f2.loc[
        df_validacion_f2["MAE"].idxmin(),
        "Modelo"
    ]
)

mae_simple_f2 = df_validacion_f2.loc[
    df_validacion_f2["Modelo"] ==
    "Regresión lineal simple",
    "MAE"
].iloc[0]

mae_multiple_f2 = df_validacion_f2.loc[
    df_validacion_f2["Modelo"] ==
    "Regresión lineal múltiple",
    "MAE"
].iloc[0]

mejora_mae_f2 = (
    1 - mae_multiple_f2 / mae_simple_f2
) * 100

metricas_test_f2 = (
    df_evaluacion_f2
    .set_index("Conjunto")
    .loc["Prueba"]
)

# ==========================================================
# 3. RECUPERAR RESULTADOS COMPLEMENTARIOS DE R
# ==========================================================

r2_r = re.search(
    r"Multiple R-squared:\s*([\d.]+)",
    texto_r_f2
)

r2_ajustado_r = re.search(
    r"Adjusted R-squared:\s*([\d.]+)",
    texto_r_f2
)

diferencia_r_python = re.search(
    r"Diferencia máxima R vs Python:\s*"
    r"([\d.eE+-]+)",
    texto_r_f2
)

singularidades_r = re.search(
    r"(\d+) not defined because of singularities",
    texto_r_f2
)

assert all([
    r2_r,
    r2_ajustado_r,
    diferencia_r_python,
    singularidades_r
]), "No se recuperaron todos los resultados de R."

df_resultados_r_f2 = pd.DataFrame({
    "Indicador": [
        "R2 entrenamiento R",
        "R2 ajustado entrenamiento R",
        "Diferencia máxima R vs Python",
        "Coeficientes no estimables"
    ],
    "Valor": [
        float(r2_r.group(1)),
        float(r2_ajustado_r.group(1)),
        float(diferencia_r_python.group(1)),
        int(singularidades_r.group(1))
    ]
})

# ==========================================================
# 4. PARTICIÓN CRONOLÓGICA
# ==========================================================

# Recuperar tamaños desde el resumen reproducible.

texto_resumen_f2 = recuperar_texto_f2(32)

particion_f2 = re.search(
    r"Partición:\s*(\d+)\s+entrenamiento,\s*"
    r"(\d+)\s+validación,\s*(\d+)\s+prueba",
    texto_resumen_f2
)

assert particion_f2 is not None

df_particion_f2 = pd.DataFrame({
    "Conjunto": [
        "Entrenamiento",
        "Validación",
        "Prueba"
    ],
    "Registros": [
        int(particion_f2.group(1)),
        int(particion_f2.group(2)),
        int(particion_f2.group(3))
    ]
})

df_particion_f2["Porcentaje"] = (
    df_particion_f2["Registros"]
    / df_particion_f2["Registros"].sum()
    * 100
)

# ==========================================================
# 5. VISUALIZACIONES INTERACTIVAS
# ==========================================================

figuras_fase2 = {}

# 5.1 Comparación de errores en validación

df_errores_f2 = df_validacion_f2.melt(
    id_vars="Modelo",
    value_vars=["MAE", "RMSE"],
    var_name="Metrica",
    value_name="Valor"
)

figuras_fase2["comparacion_modelos"] = px.bar(
    df_errores_f2,
    x="Modelo",
    y="Valor",
    color="Metrica",
    barmode="group",
    text_auto=".2f",
    title="Comparación de modelos en validación",
    color_discrete_sequence=[
        COLORES["principal"],
        COLORES["secundario"]
    ]
)

# 5.2 Comparación del coeficiente R2

figuras_fase2["comparacion_r2"] = px.bar(
    df_validacion_f2,
    x="Modelo",
    y="R2",
    color="Modelo",
    text_auto=".4f",
    title="Coeficiente R² en validación"
)

figuras_fase2["comparacion_r2"].add_hline(
    y=0,
    line_dash="dash",
    line_color=COLORES["negativo"]
)

figuras_fase2["comparacion_r2"].update_layout(
    showlegend=False
)

# 5.3 Desempeño del modelo seleccionado

df_evaluacion_largo_f2 = df_evaluacion_f2.melt(
    id_vars="Conjunto",
    value_vars=["MAE", "RMSE"],
    var_name="Metrica",
    value_name="Valor"
)

figuras_fase2["evaluacion_final"] = px.bar(
    df_evaluacion_largo_f2,
    x="Conjunto",
    y="Valor",
    color="Metrica",
    barmode="group",
    text_auto=".2f",
    title="Regresión múltiple: validación y prueba",
    color_discrete_sequence=[
        COLORES["principal"],
        COLORES["secundario"]
    ]
)

# 5.4 R2 del modelo seleccionado

figuras_fase2["r2_evaluacion"] = px.bar(
    df_evaluacion_f2,
    x="Conjunto",
    y="R2",
    color="Conjunto",
    text_auto=".4f",
    title="Regresión múltiple: evolución del R²"
)

figuras_fase2["r2_evaluacion"].add_hline(
    y=0,
    line_dash="dash",
    line_color=COLORES["negativo"]
)

figuras_fase2["r2_evaluacion"].update_layout(
    showlegend=False
)

# 5.5 Distribución de los conjuntos

figuras_fase2["particion_datos"] = px.pie(
    df_particion_f2,
    names="Conjunto",
    values="Registros",
    hole=0.55,
    title="Partición cronológica del dataset",
    color_discrete_sequence=[
        COLORES["principal"],
        COLORES["secundario"],
        COLORES["acento"]
    ]
)

figuras_fase2["particion_datos"].update_traces(
    textinfo="label+percent"
)

# 5.6 Mejora relativa del MAE

figuras_fase2["mejora_mae"] = go.Figure(
    go.Indicator(
        mode="number",
        value=mejora_mae_f2,
        number={
            "suffix": "%",
            "valueformat": ".2f",
            "font": {
                "size": 65,
                "color": COLORES["secundario"]
            }
        },
        title={
            "text": "Reducción del MAE en validación"
        }
    )
)

# ==========================================================
# 6. APLICAR ESTILO VISUAL
# ==========================================================

for nombre, figura in figuras_fase2.items():

    figura.update_layout(
        template="plotly_white",
        font=dict(
            family="Arial",
            size=12,
            color=COLORES["texto"]
        ),
        title_font_size=17,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=45, r=35, t=70, b=65),
        height=420
    )

# ==========================================================
# 7. RECUPERAR IMÁGENES ORIGINALES DEL NOTEBOOK
# ==========================================================

CARPETA_IMAGENES_F2 = FASE2 / "imagenes_originales"

CARPETA_IMAGENES_F2.mkdir(
    parents=True,
    exist_ok=True
)

imagenes_originales_f2 = []

for numero_celda, celda in enumerate(
    celdas_f2, start=1
):

    for numero_salida, salida in enumerate(
        celda.get("outputs", []), start=1
    ):

        datos_salida = salida.get("data", {})

        for mime, extension in [
            ("image/png", ".png"),
            ("image/jpeg", ".jpg")
        ]:

            if mime not in datos_salida:
                continue

            contenido = datos_salida[mime]

            if isinstance(contenido, list):
                contenido = "".join(contenido)

            nombre = (
                f"celda_{numero_celda:02d}"
                f"_salida_{numero_salida:02d}"
                f"{extension}"
            )

            ruta_imagen = (
                CARPETA_IMAGENES_F2 / nombre
            )

            ruta_imagen.write_bytes(
                base64.b64decode(contenido)
            )

            imagenes_originales_f2.append({
                "Celda": numero_celda,
                "Archivo": nombre,
                "Ruta": str(ruta_imagen),
                "Tamaño_KB": round(
                    ruta_imagen.stat().st_size / 1024,
                    2
                )
            })

df_imagenes_f2 = pd.DataFrame(
    imagenes_originales_f2,
    columns=[
        "Celda",
        "Archivo",
        "Ruta",
        "Tamaño_KB"
    ]
)

# ==========================================================
# 8. GUARDAR TABLAS Y FIGURAS
# ==========================================================

tablas_fase2 = {
    "comparacion_modelos": df_validacion_f2,
    "evaluacion_final": df_evaluacion_f2,
    "resultados_r": df_resultados_r_f2,
    "particion_datos": df_particion_f2,
    "inventario_imagenes": df_imagenes_f2
}

for nombre, tabla in tablas_fase2.items():

    tabla.to_csv(
        FASE2 / f"{nombre}.csv",
        index=False,
        encoding="utf-8-sig"
    )

CARPETA_GRAFICAS_F2 = FASE2 / "graficas"

CARPETA_GRAFICAS_F2.mkdir(
    parents=True,
    exist_ok=True
)

for nombre, figura in figuras_fase2.items():

    figura.write_html(
        str(CARPETA_GRAFICAS_F2 / f"{nombre}.html"),
        include_plotlyjs=True
    )

    figura.write_json(
        str(CARPETA_GRAFICAS_F2 / f"{nombre}.json")
    )

# ==========================================================
# 9. VALIDACIONES
# ==========================================================

assert df_particion_f2["Registros"].sum() == len(df)

assert modelo_seleccionado_f2 == (
    "Regresión lineal múltiple"
)

assert np.isclose(
    metricas_test_f2["MAE"],
    937.7764,
    atol=0.0001
)

assert np.isclose(
    metricas_test_f2["RMSE"],
    1231.9848,
    atol=0.0001
)

assert np.isclose(
    metricas_test_f2["R2"],
    0.6218,
    atol=0.0001
)

assert float(
    diferencia_r_python.group(1)
) < 1e-8

assert len(figuras_fase2) == 6

assert len(list(
    CARPETA_GRAFICAS_F2.glob("*.html")
)) == 6

assert len(list(
    CARPETA_GRAFICAS_F2.glob("*.json")
)) == 6

# ==========================================================
# 10. PRESENTACIÓN DE RESULTADOS
# ==========================================================

print("=" * 60)
print("FASE 2 - CONSOLIDACIÓN Y VISUALIZACIONES")
print("=" * 60)

print("\n=== COMPARACIÓN DE MODELOS ===")
display(df_validacion_f2)

print("\nModelo seleccionado:", modelo_seleccionado_f2)

print(f"Reducción del MAE: {mejora_mae_f2:.2f}%")

print("\n=== EVALUACIÓN FINAL ===")
display(df_evaluacion_f2)

print("\n=== RESULTADOS COMPLEMENTARIOS EN R ===")
display(df_resultados_r_f2)

print("\n=== PARTICIÓN CRONOLÓGICA ===")
display(df_particion_f2.round(2))

print("\n=== IMÁGENES ORIGINALES RECUPERADAS ===")
display(df_imagenes_f2)

print("\n=== GRÁFICAS INTERACTIVAS ===")

for nombre in figuras_fase2:
    print(f"{nombre}: OK")

print("\n=== VALIDACIONES ===")
print("Integridad de métricas originales: OK")
print("Partición del dataset: OK")
print("Equivalencia R y Python: OK")
print("Modelos reentrenados: 0")
print("Gráficas interactivas:", len(figuras_fase2))
print("Imágenes originales:", len(df_imagenes_f2))

print("\n=== VISTA PREVIA ===")

figuras_fase2["comparacion_modelos"].show()

figuras_fase2["r2_evaluacion"].show()

print("\nFase 2 - Consolidación finalizada correctamente.")

In [ ]:
# ==========================================================
# DASHBOARD GLOBAL - CELDA 10
# FASE 3: Recuperación de resultados originales
# ==========================================================

import json
import re
import base64
from pathlib import Path

# ==========================================================
# 1. LOCALIZAR EL NOTEBOOK ORIGINAL
# ==========================================================

nombre_notebook_f3 = (
    "Fase_3_Modelado_Avanzado_Bicicletas.ipynb"
)

archivos_f3 = list(
    BASE_DASHBOARD.rglob(nombre_notebook_f3)
)

assert len(archivos_f3) == 1, (
    f"Se esperaba un notebook de Fase 3. "
    f"Encontrados: {len(archivos_f3)}"
)

ruta_notebook_f3 = archivos_f3[0]

# Crear directorios para almacenar las evidencias.

FASE3 = BASE_DASHBOARD / "resultados" / "Fase_3"

FASE3.mkdir(parents=True, exist_ok=True)

CARPETA_IMAGENES_F3 = FASE3 / "imagenes_originales"

CARPETA_IMAGENES_F3.mkdir(
    parents=True,
    exist_ok=True
)

# ==========================================================
# 2. LEER EL NOTEBOOK SIN EJECUTAR SU CÓDIGO
# ==========================================================

with open(
    ruta_notebook_f3,
    "r",
    encoding="utf-8"
) as archivo:

    notebook_f3 = json.load(archivo)

celdas_f3 = notebook_f3.get("cells", [])

assert len(celdas_f3) > 0, (
    "El notebook no contiene celdas."
)

# ==========================================================
# 3. RECUPERAR INVENTARIO, SALIDAS E IMÁGENES
# ==========================================================

inventario_f3 = []
salidas_f3 = []
imagenes_originales_f3 = []
errores_f3 = []

for numero_celda, celda in enumerate(
    celdas_f3,
    start=1
):

    tipo = celda.get("cell_type", "")

    fuente = celda.get("source", "")

    if isinstance(fuente, list):
        fuente = "".join(fuente)

    outputs = celda.get("outputs", [])

    inventario_f3.append({
        "Celda": numero_celda,
        "Tipo": tipo,
        "Ejecucion": celda.get("execution_count"),
        "Cantidad_salidas": len(outputs),
        "Contenido": fuente[:200].replace("\n", " ")
    })

    for numero_salida, salida in enumerate(
        outputs,
        start=1
    ):

        datos_salida = salida.get("data", {})

        # Recuperar resultados textuales originales.

        texto = salida.get("text", "")

        if not texto:
            texto = datos_salida.get("text/plain", "")

        if isinstance(texto, list):
            texto = "".join(texto)

        if texto:

            salidas_f3.append({
                "Celda": numero_celda,
                "Salida": numero_salida,
                "Tipo_salida": salida.get("output_type"),
                "Resultado": texto
            })

        # Identificar posibles errores históricos.

        if salida.get("output_type") == "error":

            errores_f3.append({
                "Celda": numero_celda,
                "Error": salida.get("ename"),
                "Descripcion": salida.get("evalue")
            })

        # Recuperar imágenes originales.

        for mime, extension in [
            ("image/png", ".png"),
            ("image/jpeg", ".jpg")
        ]:

            if mime not in datos_salida:
                continue

            contenido = datos_salida[mime]

            if isinstance(contenido, list):
                contenido = "".join(contenido)

            nombre = (
                f"celda_{numero_celda:02d}"
                f"_salida_{numero_salida:02d}"
                f"{extension}"
            )

            ruta_imagen = (
                CARPETA_IMAGENES_F3 / nombre
            )

            ruta_imagen.write_bytes(
                base64.b64decode(contenido)
            )

            imagenes_originales_f3.append({
                "Celda": numero_celda,
                "Archivo": nombre,
                "Ruta": str(ruta_imagen),
                "Tamaño_KB": round(
                    ruta_imagen.stat().st_size / 1024,
                    2
                )
            })

# ==========================================================
# 4. CONSOLIDAR LAS EVIDENCIAS
# ==========================================================

df_inventario_f3 = pd.DataFrame(inventario_f3)

df_salidas_f3 = pd.DataFrame(
    salidas_f3,
    columns=[
        "Celda",
        "Salida",
        "Tipo_salida",
        "Resultado"
    ]
)

df_imagenes_f3 = pd.DataFrame(
    imagenes_originales_f3,
    columns=[
        "Celda",
        "Archivo",
        "Ruta",
        "Tamaño_KB"
    ]
)

df_errores_f3 = pd.DataFrame(
    errores_f3,
    columns=[
        "Celda",
        "Error",
        "Descripcion"
    ]
)

assert not df_salidas_f3.empty, (
    "No se encontraron resultados textuales guardados."
)

# ==========================================================
# 5. IDENTIFICAR MÉTRICAS Y RESULTADOS RELEVANTES
# ==========================================================

patrones_f3 = {

    "Regresion": (
        r"\b(?:MAE|RMSE|MSE|R2|R²|Ridge|"
        r"Regressor|regresión|regresion)\b"
    ),

    "Clasificacion": (
        r"\b(?:Accuracy|Precision|Recall|"
        r"F1|F1_score|F1_macro|AUC|AUC_ROC|"
        r"ROC|KNN|Classifier|clasificación|"
        r"clasificacion|umbral)\b"
    ),

    "Clustering": (
        r"\b(?:Silhouette|KMeans|K-Means|"
        r"Agglomerative|clustering|"
        r"ARI|inercia|Davies|Calinski)\b"
    ),

    "Validacion": (
        r"\b(?:validación|validacion|"
        r"TimeSeriesSplit|fold|folds|"
        r"hiperparámetros|hiperparametros|"
        r"seleccionado|partición|particion)\b"
    )
}

resultados_detectados_f3 = {}

for categoria, patron in patrones_f3.items():

    resultados_detectados_f3[categoria] = (
        df_salidas_f3.loc[
            df_salidas_f3["Resultado"].str.contains(
                patron,
                case=False,
                regex=True,
                na=False
            )
        ].copy()
    )

# Consolidar sin duplicar salidas que pertenecen
# a más de una categoría.

df_metricas_detectadas_f3 = (
    pd.concat(
        resultados_detectados_f3.values(),
        ignore_index=True
    )
    .drop_duplicates(
        subset=["Celda", "Salida"]
    )
    .sort_values(["Celda", "Salida"])
    .reset_index(drop=True)
)

# ==========================================================
# 6. GUARDAR EVIDENCIAS RECUPERADAS
# ==========================================================

tablas_recuperadas_f3 = {

    "inventario_notebook_fase3":
        df_inventario_f3,

    "salidas_originales_fase3":
        df_salidas_f3,

    "metricas_detectadas_fase3":
        df_metricas_detectadas_f3,

    "inventario_imagenes_fase3":
        df_imagenes_f3,

    "errores_historicos_fase3":
        df_errores_f3
}

for nombre, tabla in tablas_recuperadas_f3.items():

    tabla.to_csv(
        FASE3 / f"{nombre}.csv",
        index=False,
        encoding="utf-8-sig"
    )

# Guardar resultados por categoría.

for categoria, tabla in resultados_detectados_f3.items():

    tabla.to_csv(
        FASE3 / f"resultados_{categoria.lower()}.csv",
        index=False,
        encoding="utf-8-sig"
    )

# ==========================================================
# 7. PRESENTAR RESUMEN DE RECUPERACIÓN
# ==========================================================

print("=" * 60)
print("FASE 3 - RECUPERACIÓN DE RESULTADOS ORIGINALES")
print("=" * 60)

print("\nNotebook:", ruta_notebook_f3.name)

print("Total de celdas:", len(celdas_f3))

print(
    "Celdas de código:",
    sum(
        c.get("cell_type") == "code"
        for c in celdas_f3
    )
)

print(
    "Celdas Markdown:",
    sum(
        c.get("cell_type") == "markdown"
        for c in celdas_f3
    )
)

print(
    "Salidas textuales recuperadas:",
    len(df_salidas_f3)
)

print(
    "Imágenes originales recuperadas:",
    len(df_imagenes_f3)
)

print(
    "Errores históricos detectados:",
    len(df_errores_f3)
)

print("\n=== RESULTADOS POR CATEGORÍA ===")

for categoria, tabla in resultados_detectados_f3.items():

    print(f"{categoria}: {len(tabla)} salidas")

# ==========================================================
# 8. MOSTRAR INVENTARIO DE CELDAS
# ==========================================================

print("\n=== INVENTARIO DE CELDAS ===")

print(
    df_inventario_f3[
        [
            "Celda",
            "Tipo",
            "Ejecucion",
            "Cantidad_salidas",
            "Contenido"
        ]
    ].to_string(index=False)
)

# ==========================================================
# 9. MOSTRAR RESULTADOS ORIGINALES
# ==========================================================

print("\n=== MÉTRICAS Y RESULTADOS DETECTADOS ===")

for _, fila in df_metricas_detectadas_f3.iterrows():

    print("\n" + "-" * 60)

    print(
        f"Celda: {fila['Celda']} | "
        f"Salida: {fila['Salida']}"
    )

    print("-" * 60)

    print(fila["Resultado"][:8000])

# ==========================================================
# 10. MOSTRAR IMÁGENES Y ERRORES HISTÓRICOS
# ==========================================================

print("\n=== IMÁGENES ORIGINALES RECUPERADAS ===")

print(
    df_imagenes_f3[
        ["Celda", "Archivo", "Tamaño_KB"]
    ].to_string(index=False)
)

if not df_errores_f3.empty:

    print("\n=== ERRORES HISTÓRICOS ===")

    print(
        df_errores_f3.to_string(index=False)
    )

# ==========================================================
# 11. VALIDACIONES FINALES
# ==========================================================

assert len(df_inventario_f3) == len(celdas_f3)

assert len(df_metricas_detectadas_f3) > 0, (
    "No se detectaron métricas relevantes."
)

assert all(
    Path(ruta).exists()
    for ruta in df_imagenes_f3["Ruta"]
)

assert all(
    (FASE3 / f"{nombre}.csv").exists()
    for nombre in tablas_recuperadas_f3
)

print("\n=== VALIDACIONES ===")

print("Lectura del notebook: OK")
print("Recuperación de salidas: OK")
print("Recuperación de imágenes: OK")
print("Archivos CSV generados: 9")
print("Modelos reentrenados: 0")

print(
    "\nFase 3 - Recuperación de evidencias finalizada."
)

In [ ]:
# ==========================================================
# DASHBOARD GLOBAL - CELDA 11
# FASE 3: Consolidación y visualizaciones
# ==========================================================

import re

# ==========================================================
# 1. FUNCIONES PARA RECUPERAR RESULTADOS ORIGINALES
# ==========================================================

def recuperar_texto_f3(celda, salida=None):

    filtro = df_salidas_f3["Celda"] == celda

    if salida is not None:
        filtro &= df_salidas_f3["Salida"] == salida

    return "\n".join(
        df_salidas_f3.loc[filtro, "Resultado"].tolist()
    )


def extraer_tabla_f3(texto, modelos, columnas):

    # Extraer filas numéricas de las tablas originales.
    numero = r"(-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)"

    patron = (
        r"(?m)^\s*\d+\s+("
        + "|".join(re.escape(m) for m in modelos)
        + r")"
        + (r"\s+" + numero) * (len(columnas) - 1)
        + r"\s*$"
    )

    coincidencias = re.findall(patron, texto)

    assert len(coincidencias) == len(modelos), (
        f"No se recuperaron todos los modelos: {modelos}"
    )

    tabla = pd.DataFrame(
        coincidencias,
        columns=columnas
    )

    for columna in columnas[1:]:
        tabla[columna] = pd.to_numeric(
            tabla[columna],
            errors="raise"
        )

    assert tabla[columnas[0]].is_unique

    return tabla


# ==========================================================
# 2. MÉTRICAS DE REGRESIÓN
# ==========================================================

# Resultados originales de validación: celda 13.

df_regresion_val_f3 = extraer_tabla_f3(
    recuperar_texto_f3(13, 1),
    ["Ridge", "Random Forest"],
    ["Modelo", "MAE", "RMSE", "R2"]
)

df_regresion_val_f3["Conjunto"] = "Validación"

# Resultados originales de prueba: celda 44.

df_regresion_test_f3 = extraer_tabla_f3(
    recuperar_texto_f3(44, 2),
    ["Ridge", "Random Forest"],
    ["Modelo", "MAE", "RMSE", "R2"]
)

df_regresion_test_f3["Conjunto"] = "Prueba"

df_regresion_f3 = pd.concat(
    [
        df_regresion_val_f3,
        df_regresion_test_f3
    ],
    ignore_index=True
)

# Seleccionar exclusivamente según validación.

mejor_regresion_f3 = (
    df_regresion_val_f3.loc[
        df_regresion_val_f3["RMSE"].idxmin(),
        "Modelo"
    ]
)

# ==========================================================
# 3. MÉTRICAS DE CLASIFICACIÓN
# ==========================================================

df_clasificacion_test_f3 = extraer_tabla_f3(
    recuperar_texto_f3(44, 4),
    ["Random Forest", "KNN"],
    [
        "Modelo",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "AUC_ROC"
    ]
)

# Recuperar F1 promedio de validación temporal.

texto_knn_f3 = recuperar_texto_f3(22, 2)
texto_rf_f3 = recuperar_texto_f3(24, 2)

f1_knn_f3 = re.search(
    r"F1 promedio:\s*([\d.]+)",
    texto_knn_f3
)

f1_rf_f3 = re.search(
    r"F1 promedio:\s*([\d.]+)",
    texto_rf_f3
)

assert f1_knn_f3 is not None
assert f1_rf_f3 is not None

df_clasificacion_cv_f3 = pd.DataFrame({
    "Modelo": ["KNN", "Random Forest"],
    "F1_promedio": [
        float(f1_knn_f3.group(1)),
        float(f1_rf_f3.group(1))
    ]
})

mejor_clasificacion_f3 = (
    df_clasificacion_cv_f3.loc[
        df_clasificacion_cv_f3[
            "F1_promedio"
        ].idxmax(),
        "Modelo"
    ]
)

# ==========================================================
# 4. RESULTADOS DE CLUSTERING
# ==========================================================

df_clustering_f3 = extraer_tabla_f3(
    recuperar_texto_f3(44, 6),
    ["K-Means", "Agglomerative"],
    ["Algoritmo", "Clusters", "Silueta"]
)

df_clustering_f3["Clusters"] = (
    df_clustering_f3["Clusters"].astype(int)
)

texto_ari_f3 = recuperar_texto_f3(34, 3)

resultado_ari_f3 = re.search(
    r"ARI\):\s*([\d.]+)",
    texto_ari_f3
)

assert resultado_ari_f3 is not None

ari_f3 = float(resultado_ari_f3.group(1))

# ==========================================================
# 5. HIPERPARÁMETROS SELECCIONADOS
# ==========================================================

texto_config_f3 = recuperar_texto_f3(44, 7)

patrones_config_f3 = {
    "Ridge": r"Ridge alpha:\s*([^\n]+)",
    "Random Forest Regressor":
        r"Random Forest Regressor:\s*([^\n]+)",
    "KNN": r"KNN k:\s*([^\n]+)",
    "Random Forest Classifier":
        r"Random Forest Classifier:\s*([^\n]+)",
    "K-Means": r"K-Means K:\s*([^\n]+)"
}

nombres_parametros_f3 = {
    "Ridge": "alpha",
    "Random Forest Regressor":
        "n_estimators, max_depth, min_samples_leaf",
    "KNN": "n_neighbors",
    "Random Forest Classifier":
        "n_estimators, max_depth, min_samples_leaf",
    "K-Means": "n_clusters"
}

configuraciones_f3 = []

for modelo, patron in patrones_config_f3.items():

    resultado = re.search(
        patron,
        texto_config_f3
    )

    assert resultado is not None, (
        f"Configuración no encontrada: {modelo}"
    )

    configuraciones_f3.append({
        "Modelo": modelo,
        "Hiperparametros": nombres_parametros_f3[modelo],
        "Configuracion": resultado.group(1).strip()
    })

df_configuraciones_f3 = pd.DataFrame(
    configuraciones_f3
)

# ==========================================================
# 6. DISTRIBUCIÓN TEMPORAL DE CLASES
# ==========================================================

texto_clases_f3 = recuperar_texto_f3(16, 2)

filas_clases_f3 = re.findall(
    r"(Demanda baja|Demanda alta)"
    r"\s+(\d+)\s+(\d+)\s+(\d+)",
    texto_clases_f3
)

assert len(filas_clases_f3) == 2

df_clases_f3 = pd.DataFrame(
    filas_clases_f3,
    columns=[
        "Clase",
        "Entrenamiento",
        "Validación",
        "Prueba"
    ]
)

for columna in [
    "Entrenamiento",
    "Validación",
    "Prueba"
]:

    df_clases_f3[columna] = (
        df_clases_f3[columna].astype(int)
    )

df_clases_largo_f3 = df_clases_f3.melt(
    id_vars="Clase",
    var_name="Conjunto",
    value_name="Registros"
)

# Recuperar umbral original de clasificación.

texto_umbral_f3 = recuperar_texto_f3(16, 1)

resultado_umbral_f3 = re.search(
    r"Umbral de demanda:\s*([\d.]+)",
    texto_umbral_f3
)

assert resultado_umbral_f3 is not None

umbral_demanda_f3 = float(
    resultado_umbral_f3.group(1)
)

# ==========================================================
# 7. VALIDACIÓN CRUZADA TEMPORAL
# ==========================================================

texto_folds_f3 = recuperar_texto_f3(20, 1)

filas_folds_f3 = re.findall(
    r"(?m)^\s*\d+\s+(\d+)\s+(\d+)"
    r"\s+(\d+)\s+(\d+)\s*\\?\s*$",
    texto_folds_f3
)

estados_folds_f3 = re.findall(
    r"(?m)^\s*\d+\s+(\d+)"
    r"\s+(True|False)\s+(True|False)\s*$",
    texto_folds_f3
)

assert len(filas_folds_f3) == 5
assert len(estados_folds_f3) == 5

df_folds_f3 = pd.DataFrame(
    filas_folds_f3,
    columns=[
        "Fold",
        "N_entrenamiento",
        "N_validacion",
        "Clases_entrenamiento"
    ]
).astype(int)

df_folds_f3["Clases_validacion"] = [
    int(fila[0])
    for fila in estados_folds_f3
]

df_folds_f3["AUC_calculable"] = [
    fila[1] == "True"
    for fila in estados_folds_f3
]

df_folds_f3["Fold_utilizable"] = [
    fila[2] == "True"
    for fila in estados_folds_f3
]

folds_validos_f3 = int(
    df_folds_f3["Fold_utilizable"].sum()
)

# ==========================================================
# 8. VISUALIZACIONES INTERACTIVAS
# ==========================================================

figuras_fase3 = {}

# 8.1. Comparación de errores de regresión.

df_regresion_largo_f3 = df_regresion_f3.melt(
    id_vars=["Modelo", "Conjunto"],
    value_vars=["MAE", "RMSE"],
    var_name="Metrica",
    value_name="Valor"
)

figuras_fase3["regresion_errores"] = px.bar(
    df_regresion_largo_f3,
    x="Modelo",
    y="Valor",
    color="Metrica",
    facet_col="Conjunto",
    barmode="group",
    text_auto=".2f",
    title="Regresión avanzada: MAE y RMSE",
    color_discrete_sequence=[
        COLORES["principal"],
        COLORES["secundario"]
    ]
)

# 8.2. Comparación de R².

figuras_fase3["regresion_r2"] = px.bar(
    df_regresion_f3,
    x="Modelo",
    y="R2",
    color="Conjunto",
    barmode="group",
    text_auto=".4f",
    title="Regresión avanzada: coeficiente R²"
)

figuras_fase3["regresion_r2"].add_hline(
    y=0,
    line_dash="dash",
    line_color=COLORES["negativo"]
)

# 8.3. Clasificación en prueba.

df_clasificacion_largo_f3 = (
    df_clasificacion_test_f3.melt(
        id_vars="Modelo",
        var_name="Metrica",
        value_name="Valor"
    )
)

figuras_fase3["clasificacion_prueba"] = px.bar(
    df_clasificacion_largo_f3,
    x="Metrica",
    y="Valor",
    color="Modelo",
    barmode="group",
    text_auto=".3f",
    title="Clasificación: desempeño en prueba",
    range_y=[0, 1.12],
    color_discrete_sequence=[
        COLORES["principal"],
        COLORES["secundario"]
    ]
)

# 8.4. Validación temporal de clasificadores.

figuras_fase3["clasificacion_cv"] = px.bar(
    df_clasificacion_cv_f3,
    x="Modelo",
    y="F1_promedio",
    color="Modelo",
    text_auto=".4f",
    range_y=[0, 1],
    title="F1 promedio en validación temporal"
)

figuras_fase3["clasificacion_cv"].update_layout(
    showlegend=False
)

# 8.5. Distribución temporal de clases.

figuras_fase3["distribucion_clases"] = px.bar(
    df_clases_largo_f3,
    x="Conjunto",
    y="Registros",
    color="Clase",
    barmode="stack",
    text_auto=True,
    title="Distribución temporal de la demanda",
    category_orders={
        "Conjunto": [
            "Entrenamiento",
            "Validación",
            "Prueba"
        ]
    },
    color_discrete_sequence=[
        COLORES["principal"],
        COLORES["secundario"]
    ]
)

# 8.6. Comparación de algoritmos de clustering.

figuras_fase3["clustering_silueta"] = px.bar(
    df_clustering_f3,
    x="Algoritmo",
    y="Silueta",
    color="Algoritmo",
    text_auto=".4f",
    range_y=[0, 1],
    title="Clustering: coeficiente de silueta"
)

figuras_fase3["clustering_silueta"].update_layout(
    showlegend=False
)

# 8.7. Concordancia entre algoritmos.

figuras_fase3["clustering_ari"] = go.Figure(
    go.Indicator(
        mode="number+gauge",
        value=ari_f3,
        number={"valueformat": ".4f"},
        title={"text": "Índice Rand ajustado"},
        gauge={
            "axis": {"range": [-1, 1]},
            "bar": {
                "color": COLORES["principal"]
            }
        }
    )
)

# 8.8. Validación cruzada temporal.

df_folds_grafico_f3 = df_folds_f3.copy()

df_folds_grafico_f3["Estado"] = np.where(
    df_folds_grafico_f3["Fold_utilizable"],
    "Utilizable",
    "No utilizable"
)

figuras_fase3["validacion_temporal"] = px.bar(
    df_folds_grafico_f3,
    x="Fold",
    y="N_entrenamiento",
    color="Estado",
    text_auto=True,
    title="Particiones de validación temporal",
    hover_data=[
        "N_validacion",
        "Clases_entrenamiento",
        "Clases_validacion"
    ],
    color_discrete_map={
        "Utilizable": COLORES["secundario"],
        "No utilizable": COLORES["negativo"]
    }
)

figuras_fase3["validacion_temporal"].update_xaxes(
    dtick=1
)

# ==========================================================
# 9. APLICAR ESTILO VISUAL UNIFICADO
# ==========================================================

for nombre, figura in figuras_fase3.items():

    figura.update_layout(
        template="plotly_white",
        font=dict(
            family="Arial",
            size=12,
            color=COLORES["texto"]
        ),
        title_font_size=17,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=45, r=35, t=70, b=65),
        height=420
    )

# ==========================================================
# 10. GUARDAR TABLAS Y GRÁFICAS
# ==========================================================

df_indicadores_f3 = pd.DataFrame({
    "Indicador": [
        "Modelo regresión seleccionado en validación",
        "Clasificador seleccionado en validación temporal",
        "Umbral de demanda",
        "Folds totales",
        "Folds utilizables",
        "ARI"
    ],
    "Valor": [
        mejor_regresion_f3,
        mejor_clasificacion_f3,
        umbral_demanda_f3,
        len(df_folds_f3),
        folds_validos_f3,
        ari_f3
    ]
})

tablas_fase3 = {
    "metricas_regresion": df_regresion_f3,
    "metricas_clasificacion_prueba":
        df_clasificacion_test_f3,
    "metricas_clasificacion_cv":
        df_clasificacion_cv_f3,
    "metricas_clustering": df_clustering_f3,
    "hiperparametros": df_configuraciones_f3,
    "distribucion_clases": df_clases_f3,
    "validacion_temporal": df_folds_f3,
    "indicadores_fase3": df_indicadores_f3
}

for nombre, tabla in tablas_fase3.items():

    tabla.to_csv(
        FASE3 / f"{nombre}.csv",
        index=False,
        encoding="utf-8-sig"
    )

CARPETA_GRAFICAS_F3 = FASE3 / "graficas"

CARPETA_GRAFICAS_F3.mkdir(
    parents=True,
    exist_ok=True
)

for nombre, figura in figuras_fase3.items():

    figura.write_html(
        str(CARPETA_GRAFICAS_F3 / f"{nombre}.html"),
        include_plotlyjs=True
    )

    figura.write_json(
        str(CARPETA_GRAFICAS_F3 / f"{nombre}.json")
    )

# ==========================================================
# 11. VALIDACIONES DE INTEGRIDAD
# ==========================================================

assert mejor_regresion_f3 == "Random Forest"
assert mejor_clasificacion_f3 == "KNN"

assert len(df_regresion_f3) == 4
assert len(df_clasificacion_test_f3) == 2
assert len(df_clustering_f3) == 2

assert np.isclose(umbral_demanda_f3, 3974)
assert np.isclose(ari_f3, 0.4276)

assert folds_validos_f3 == 4

assert df_clases_f3[
    ["Entrenamiento", "Validación", "Prueba"]
].sum().tolist() == [511, 109, 111]

assert df_clases_f3["Validación"].tolist() == [
    0, 109
]

assert np.isclose(
    df_regresion_test_f3.set_index(
        "Modelo"
    ).loc["Ridge", "R2"],
    0.6215
)

assert np.isclose(
    df_clasificacion_test_f3.set_index(
        "Modelo"
    ).loc["Random Forest", "AUC_ROC"],
    0.9011
)

assert len(figuras_fase3) == 8

assert all(
    (CARPETA_GRAFICAS_F3 / f"{nombre}.html").exists()
    and
    (CARPETA_GRAFICAS_F3 / f"{nombre}.json").exists()
    for nombre in figuras_fase3
)

# ==========================================================
# 12. PRESENTACIÓN DE RESULTADOS
# ==========================================================

print("=" * 60)
print("FASE 3 - CONSOLIDACIÓN Y VISUALIZACIONES")
print("=" * 60)

print("\n=== REGRESIÓN ===")
print(df_regresion_f3.to_string(index=False))

print("\nModelo seleccionado en validación:",
      mejor_regresion_f3)

print("\n=== CLASIFICACIÓN EN PRUEBA ===")
print(df_clasificacion_test_f3.to_string(index=False))

print("\n=== CLASIFICACIÓN - VALIDACIÓN TEMPORAL ===")
print(df_clasificacion_cv_f3.to_string(index=False))

print("\nClasificador seleccionado:",
      mejor_clasificacion_f3)

print("\n=== CLUSTERING ===")
print(df_clustering_f3.to_string(index=False))

print("\nARI:", ari_f3)

print("\n=== HIPERPARÁMETROS ===")
print(df_configuraciones_f3.to_string(index=False))

print("\n=== DISTRIBUCIÓN DE CLASES ===")
print(df_clases_f3.to_string(index=False))

print("\nUmbral de demanda:", umbral_demanda_f3)

print("\n=== VALIDACIÓN CRUZADA TEMPORAL ===")
print(df_folds_f3.to_string(index=False))

print("\nFolds utilizables:", folds_validos_f3)

print("\n=== GRÁFICAS INTERACTIVAS ===")

for nombre in figuras_fase3:
    print(f"{nombre}: OK")

print("\n=== VALIDACIONES ===")
print("Integridad de métricas originales: OK")
print("Distribución de clases: OK")
print("Validación temporal: OK")
print("Selección de modelos: OK")
print("Modelos reentrenados: 0")
print("Tablas consolidadas:", len(tablas_fase3))
print("Gráficas interactivas:", len(figuras_fase3))
print("Imágenes originales:", len(df_imagenes_f3))

print("\nFase 3 - Consolidación finalizada correctamente.")

In [ ]:
# ==========================================================
# DASHBOARD GLOBAL - CELDA 12
# FASE 4: Recuperación y consolidación de evidencias
# ==========================================================

import json
import shutil
from pathlib import Path

# ==========================================================
# 1. CONFIGURAR DIRECTORIOS
# ==========================================================

ORIGEN_FASE4 = EVIDENCIAS / "Fase_4"

FASE4 = BASE_DASHBOARD / "resultados" / "Fase_4"

FASE4.mkdir(parents=True, exist_ok=True)

CARPETA_IMAGENES_F4 = FASE4 / "imagenes_originales"

CARPETA_IMAGENES_F4.mkdir(
    parents=True,
    exist_ok=True
)

assert ORIGEN_FASE4.exists(), (
    "No se encontró el directorio de evidencias de Fase 4."
)

# ==========================================================
# 2. RECUPERAR ARCHIVOS CSV ORIGINALES
# ==========================================================

# Se cargan directamente los resultados generados durante
# la ejecución original de la Fase 4.

tablas_fase4 = {}

inventario_csv_f4 = []

for ruta in sorted(ORIGEN_FASE4.rglob("*.csv")):

    nombre = ruta.stem

    assert nombre not in tablas_fase4, (
        f"Nombre de archivo duplicado: {nombre}"
    )

    tabla = pd.read_csv(
        ruta,
        encoding="utf-8-sig"
    )

    tablas_fase4[nombre] = tabla

    inventario_csv_f4.append({
        "Archivo": ruta.name,
        "Filas": tabla.shape[0],
        "Columnas": tabla.shape[1],
        "Variables": ", ".join(
            tabla.columns.astype(str)
        ),
        "Ruta_original": str(ruta)
    })

    # Conservar una copia exacta del archivo original.

    shutil.copy2(
        ruta,
        FASE4 / ruta.name
    )

df_inventario_csv_f4 = pd.DataFrame(
    inventario_csv_f4
)

assert len(tablas_fase4) > 0, (
    "No se recuperaron archivos CSV."
)

# ==========================================================
# 3. VALIDAR TABLAS PRINCIPALES
# ==========================================================

tablas_requeridas_f4 = [

    "metricas_baseline_regresion",
    "metricas_baseline_clasificacion",
    "resumen_final_regresion",
    "resumen_final_clasificacion",
    "resumen_mejoras",
    "resumen_preparacion",
    "cumplimiento_requerimientos",
    "detalle_outliers",
    "limites_winsorization",
    "resultados_winsorization",
    "resultados_codificacion",
    "verificacion_estandarizacion",
    "estrategia_balanceo",
    "matrices_finales"
]

tablas_faltantes_f4 = [

    nombre
    for nombre in tablas_requeridas_f4
    if nombre not in tablas_fase4
]

assert not tablas_faltantes_f4, (
    f"Tablas faltantes: {tablas_faltantes_f4}"
)

# ==========================================================
# 4. ASIGNAR TABLAS PRINCIPALES
# ==========================================================

df_regresion_f4 = tablas_fase4[
    "metricas_baseline_regresion"
].copy()

df_clasificacion_f4 = tablas_fase4[
    "metricas_baseline_clasificacion"
].copy()

df_resumen_regresion_f4 = tablas_fase4[
    "resumen_final_regresion"
].copy()

df_resumen_clasificacion_f4 = tablas_fase4[
    "resumen_final_clasificacion"
].copy()

df_mejoras_f4 = tablas_fase4[
    "resumen_mejoras"
].copy()

df_preparacion_f4 = tablas_fase4[
    "resumen_preparacion"
].copy()

df_cumplimiento_f4 = tablas_fase4[
    "cumplimiento_requerimientos"
].copy()

df_outliers_f4 = tablas_fase4[
    "detalle_outliers"
].copy()

df_limites_f4 = tablas_fase4[
    "limites_winsorization"
].copy()

df_winsorization_f4 = tablas_fase4[
    "resultados_winsorization"
].copy()

df_codificacion_f4 = tablas_fase4[
    "resultados_codificacion"
].copy()

df_estandarizacion_f4 = tablas_fase4[
    "verificacion_estandarizacion"
].copy()

df_balanceo_f4 = tablas_fase4[
    "estrategia_balanceo"
].copy()

df_matrices_f4 = tablas_fase4[
    "matrices_finales"
].copy()

# ==========================================================
# 5. RECUPERAR IMÁGENES ORIGINALES
# ==========================================================

imagenes_originales_f4 = []

for ruta in sorted(ORIGEN_FASE4.rglob("*")):

    if not ruta.is_file():
        continue

    if ruta.suffix.lower() not in [
        ".png", ".jpg", ".jpeg"
    ]:
        continue

    destino = CARPETA_IMAGENES_F4 / ruta.name

    shutil.copy2(
        ruta,
        destino
    )

    imagenes_originales_f4.append({
        "Archivo": ruta.name,
        "Formato": ruta.suffix.lower(),
        "Tamaño_KB": round(
            destino.stat().st_size / 1024,
            2
        ),
        "Ruta": str(destino)
    })

df_imagenes_f4 = pd.DataFrame(
    imagenes_originales_f4,
    columns=[
        "Archivo",
        "Formato",
        "Tamaño_KB",
        "Ruta"
    ]
)

# ==========================================================
# 6. RECUPERAR METADATOS DE REPRODUCIBILIDAD
# ==========================================================

metadatos_f4 = {}

for ruta in sorted(ORIGEN_FASE4.rglob("*.json")):

    with open(
        ruta,
        "r",
        encoding="utf-8-sig"
    ) as archivo:

        metadatos_f4[ruta.stem] = json.load(
            archivo
        )

    shutil.copy2(
        ruta,
        FASE4 / ruta.name
    )

# ==========================================================
# 7. GUARDAR INVENTARIOS
# ==========================================================

df_inventario_csv_f4.to_csv(
    FASE4 / "inventario_csv_fase4.csv",
    index=False,
    encoding="utf-8-sig"
)

df_imagenes_f4.to_csv(
    FASE4 / "inventario_imagenes_fase4.csv",
    index=False,
    encoding="utf-8-sig"
)

# ==========================================================
# 8. PRESENTAR RESULTADOS RECUPERADOS
# ==========================================================

print("=" * 60)
print("FASE 4 - RECUPERACIÓN Y CONSOLIDACIÓN")
print("=" * 60)

print("\nArchivos CSV recuperados:", len(tablas_fase4))

print("Imágenes originales:", len(df_imagenes_f4))

print("Archivos JSON:", len(metadatos_f4))

print("\n=== INVENTARIO DE TABLAS ===")

print(
    df_inventario_csv_f4[
        ["Archivo", "Filas", "Columnas"]
    ].to_string(index=False)
)

# Mostrar las tablas principales sin generar
# código HTML adicional en la salida de Colab.

tablas_revision_f4 = {

    "MÉTRICAS BASELINE DE REGRESIÓN":
        df_regresion_f4,

    "MÉTRICAS BASELINE DE CLASIFICACIÓN":
        df_clasificacion_f4,

    "RESUMEN FINAL DE REGRESIÓN":
        df_resumen_regresion_f4,

    "RESUMEN FINAL DE CLASIFICACIÓN":
        df_resumen_clasificacion_f4,

    "MEJORAS DE LOS MODELOS":
        df_mejoras_f4,

    "RESUMEN DE PREPARACIÓN":
        df_preparacion_f4,

    "CUMPLIMIENTO DE REQUERIMIENTOS":
        df_cumplimiento_f4,

    "DETECCIÓN DE OUTLIERS":
        df_outliers_f4,

    "LÍMITES DE WINSORIZATION":
        df_limites_f4,

    "RESULTADOS DE WINSORIZATION":
        df_winsorization_f4,

    "CODIFICACIÓN DE VARIABLES":
        df_codificacion_f4,

    "ESTANDARIZACIÓN":
        df_estandarizacion_f4,

    "ESTRATEGIA DE BALANCEO":
        df_balanceo_f4,

    "MATRICES DE CONFUSIÓN":
        df_matrices_f4
}

for titulo, tabla in tablas_revision_f4.items():

    print(f"\n=== {titulo} ===")

    print(tabla.to_string(index=False))

print("\n=== IMÁGENES ORIGINALES ===")

print(
    df_imagenes_f4[
        ["Archivo", "Tamaño_KB"]
    ].to_string(index=False)
)

print("\n=== METADATOS DE REPRODUCIBILIDAD ===")

for nombre, contenido in metadatos_f4.items():

    print(f"\nArchivo: {nombre}.json")

    print(
        json.dumps(
            contenido,
            indent=2,
            ensure_ascii=False
        )
    )

# ==========================================================
# 9. VALIDACIONES FINALES
# ==========================================================

assert len(tablas_fase4) == 19, (
    "La cantidad de CSV no coincide con el inventario."
)

assert len(df_imagenes_f4) == 12, (
    "La cantidad de imágenes no coincide con el inventario."
)

assert len(metadatos_f4) == 2

assert (
    len(tablas_fase4)
    + len(df_imagenes_f4)
    + len(metadatos_f4)
) == 33

assert all(
    not tabla.empty
    for tabla in tablas_fase4.values()
)

assert all(
    Path(ruta).exists()
    for ruta in df_imagenes_f4["Ruta"]
)

assert all(
    (FASE4 / f"{nombre}.csv").exists()
    for nombre in tablas_fase4
)

print("\n=== VALIDACIONES ===")

print("Recuperación de archivos CSV: OK")
print("Recuperación de imágenes: OK")
print("Recuperación de metadatos: OK")
print("Tablas principales disponibles: OK")
print("Integridad del inventario: OK")
print("Total de evidencias originales: 33")
print("Modelos reentrenados: 0")

print(
    "\nFase 4 - Recuperación de evidencias finalizada."
)

In [ ]:
# ==========================================================
# DASHBOARD GLOBAL - CELDA 13
# FASE 4: Consolidación y visualizaciones interactivas
# ==========================================================

import hashlib
import plotly.express as px
import plotly.graph_objects as go

# ==========================================================
# 1. VALIDAR IDENTIDAD DEL DATASET
# ==========================================================

hash_actual_f4 = hashlib.sha256(
    RUTA_CSV.read_bytes()
).hexdigest()

hash_original_f4 = metadatos_f4[
    "identificacion_dataset"
]["sha256"]

assert hash_actual_f4 == hash_original_f4, (
    "El dataset no coincide con el utilizado en Fase 4."
)

print("Identidad del dataset: OK")

# ==========================================================
# 2. CONSOLIDAR INDICADORES PRINCIPALES
# ==========================================================

indicadores_preparacion_f4 = (
    df_preparacion_f4
    .set_index("Indicador")["Resultado"]
    .to_dict()
)

regresion_test_f4 = (
    df_regresion_f4.loc[
        df_regresion_f4["Conjunto"] == "Prueba"
    ]
    .set_index("Modelo")
)

clasificacion_test_f4 = (
    df_clasificacion_f4.loc[
        df_clasificacion_f4["Conjunto"] == "Prueba"
    ]
    .set_index("Modelo")
)

rmse_lineal_f4 = regresion_test_f4.loc[
    "Regresión Lineal Simple", "RMSE"
]

rmse_dummy_f4 = regresion_test_f4.loc[
    "Dummy Regressor", "RMSE"
]

accuracy_logistica_f4 = clasificacion_test_f4.loc[
    "Regresión Logística", "Accuracy"
]

accuracy_dummy_f4 = clasificacion_test_f4.loc[
    "Dummy Classifier", "Accuracy"
]

auc_logistica_f4 = clasificacion_test_f4.loc[
    "Regresión Logística", "AUC_ROC"
]

mejora_rmse_f4 = (
    1 - rmse_lineal_f4 / rmse_dummy_f4
) * 100

mejora_accuracy_f4 = (
    accuracy_logistica_f4 - accuracy_dummy_f4
) * 100

mejora_f1_macro_f4 = (
    clasificacion_test_f4.loc[
        "Regresión Logística", "F1_Macro"
    ]
    -
    clasificacion_test_f4.loc[
        "Dummy Classifier", "F1_Macro"
    ]
) * 100

# ==========================================================
# 3. CUMPLIMIENTO DE REQUERIMIENTOS
# ==========================================================

df_estados_f4 = (
    df_cumplimiento_f4["Estado"]
    .value_counts()
    .rename_axis("Estado")
    .reset_index(name="Cantidad")
)

cumplidos_f4 = int(
    (df_cumplimiento_f4["Estado"] == "Cumplido").sum()
)

no_aplica_f4 = int(
    (df_cumplimiento_f4["Estado"] == "No aplica").sum()
)

total_requerimientos_f4 = len(df_cumplimiento_f4)

aplicables_f4 = (
    total_requerimientos_f4 - no_aplica_f4
)

cumplimiento_pct_f4 = (
    cumplidos_f4 / aplicables_f4 * 100
)

# ==========================================================
# 4. CONFIGURAR VISUALIZACIONES
# ==========================================================

figuras_fase4 = {}

orden_conjuntos_f4 = [
    "Entrenamiento",
    "Validación",
    "Prueba"
]

colores_modelos_f4 = {
    "Regresión Lineal Simple": COLORES["principal"],
    "Regresión Logística": COLORES["principal"],
    "Dummy Regressor": "#94A3B8",
    "Dummy Classifier": "#94A3B8"
}

# ==========================================================
# 5. BASELINE DE REGRESIÓN
# ==========================================================

# 5.1 Comparación de MAE y RMSE.

df_regresion_largo_f4 = df_regresion_f4.melt(
    id_vars=["Modelo", "Conjunto"],
    value_vars=["MAE", "RMSE"],
    var_name="Metrica",
    value_name="Valor"
)

figuras_fase4["regresion_errores"] = px.bar(
    df_regresion_largo_f4,
    x="Conjunto",
    y="Valor",
    color="Modelo",
    facet_col="Metrica",
    barmode="group",
    text_auto=".1f",
    category_orders={
        "Conjunto": orden_conjuntos_f4
    },
    color_discrete_map=colores_modelos_f4,
    title="Baseline de regresión: MAE y RMSE"
)

# 5.2 Comparación de R².

figuras_fase4["regresion_r2"] = px.bar(
    df_regresion_f4,
    x="Conjunto",
    y="R2",
    color="Modelo",
    barmode="group",
    text_auto=".3f",
    category_orders={
        "Conjunto": orden_conjuntos_f4
    },
    color_discrete_map=colores_modelos_f4,
    title="Baseline de regresión: coeficiente R²"
)

figuras_fase4["regresion_r2"].add_hline(
    y=0,
    line_dash="dash",
    line_color=COLORES["negativo"]
)

# ==========================================================
# 6. BASELINE DE CLASIFICACIÓN
# ==========================================================

# 6.1 Comparación de métricas en prueba.

metricas_clasificacion_f4 = [
    "Accuracy",
    "Balanced_Accuracy",
    "Precision",
    "Recall",
    "F1",
    "F1_Macro",
    "AUC_ROC"
]

df_clasificacion_largo_f4 = (
    df_resumen_clasificacion_f4.melt(
        id_vars="Modelo",
        value_vars=metricas_clasificacion_f4,
        var_name="Metrica",
        value_name="Valor"
    )
)

figuras_fase4["clasificacion_prueba"] = px.bar(
    df_clasificacion_largo_f4,
    x="Metrica",
    y="Valor",
    color="Modelo",
    barmode="group",
    text_auto=".3f",
    range_y=[0, 1.12],
    color_discrete_map=colores_modelos_f4,
    title="Baseline de clasificación: métricas en prueba"
)

# 6.2 Accuracy por conjunto.

figuras_fase4["clasificacion_accuracy"] = px.bar(
    df_clasificacion_f4,
    x="Conjunto",
    y="Accuracy",
    color="Modelo",
    barmode="group",
    text_auto=".1%",
    range_y=[0, 1.12],
    category_orders={
        "Conjunto": orden_conjuntos_f4
    },
    color_discrete_map=colores_modelos_f4,
    title="Accuracy por conjunto de evaluación"
)

figuras_fase4[
    "clasificacion_accuracy"
].update_yaxes(tickformat=".0%")

# 6.3 F1 Macro por conjunto.

figuras_fase4["clasificacion_f1_macro"] = px.bar(
    df_clasificacion_f4,
    x="Conjunto",
    y="F1_Macro",
    color="Modelo",
    barmode="group",
    text_auto=".3f",
    range_y=[0, 1.12],
    category_orders={
        "Conjunto": orden_conjuntos_f4
    },
    color_discrete_map=colores_modelos_f4,
    title="F1 Macro por conjunto de evaluación"
)

# 6.4 AUC-ROC por conjunto.

figuras_fase4["clasificacion_auc"] = px.bar(
    df_clasificacion_f4,
    x="Conjunto",
    y="AUC_ROC",
    color="Modelo",
    barmode="group",
    text_auto=".3f",
    range_y=[0, 1.12],
    category_orders={
        "Conjunto": orden_conjuntos_f4
    },
    color_discrete_map=colores_modelos_f4,
    title="AUC-ROC por conjunto de evaluación"
)

figuras_fase4["clasificacion_auc"].add_hline(
    y=0.5,
    line_dash="dash",
    annotation_text="Referencia aleatoria"
)

# ==========================================================
# 7. PREPARACIÓN Y CALIDAD DE DATOS
# ==========================================================

# 7.1 Completitud de los datos.

total_celdas_f4 = (
    indicadores_preparacion_f4["Registros originales"]
    *
    indicadores_preparacion_f4["Variables originales"]
)

faltantes_f4 = indicadores_preparacion_f4[
    "Valores faltantes"
]

completitud_f4 = (
    1 - faltantes_f4 / total_celdas_f4
) * 100

figuras_fase4["completitud"] = go.Figure(
    go.Indicator(
        mode="number+gauge",
        value=completitud_f4,
        number={
            "suffix": "%",
            "valueformat": ".1f"
        },
        title={
            "text": "Completitud del dataset"
        },
        gauge={
            "axis": {"range": [0, 100]},
            "bar": {
                "color": COLORES["secundario"]
            }
        }
    )
)

# 7.2 Outliers detectados por variable.

df_conteo_outliers_f4 = (
    df_outliers_f4["Variable_atipica"]
    .value_counts()
    .rename_axis("Variable")
    .reset_index(name="Outliers")
)

df_conteo_outliers_f4["Porcentaje"] = (
    df_conteo_outliers_f4["Outliers"]
    / len(df) * 100
)

figuras_fase4["outliers"] = px.bar(
    df_conteo_outliers_f4,
    x="Variable",
    y="Outliers",
    color="Variable",
    text_auto=True,
    hover_data=["Porcentaje"],
    title="Valores atípicos detectados mediante IQR"
)

figuras_fase4["outliers"].update_layout(
    showlegend=False
)

# 7.3 Valores modificados mediante winsorization.

figuras_fase4["winsorization"] = px.bar(
    df_winsorization_f4,
    x="Conjunto",
    y="Valores_modificados",
    color="Variable",
    facet_col="Problema",
    barmode="group",
    text_auto=True,
    hover_data=["Porcentaje", "Registros"],
    category_orders={
        "Conjunto": orden_conjuntos_f4
    },
    title="Valores modificados mediante winsorization"
)

# ==========================================================
# 8. INGENIERÍA DE CARACTERÍSTICAS
# ==========================================================

# 8.1 Composición de las características finales.

df_caracteristicas_f4 = pd.DataFrame({
    "Tipo": [
        "Variables numéricas estandarizadas",
        "Columnas One-Hot Encoding"
    ],
    "Cantidad": [
        indicadores_preparacion_f4[
            "Variables numéricas estandarizadas"
        ],
        indicadores_preparacion_f4[
            "Columnas One-Hot Encoding"
        ]
    ]
})

figuras_fase4["caracteristicas"] = px.pie(
    df_caracteristicas_f4,
    names="Tipo",
    values="Cantidad",
    hole=0.55,
    color_discrete_sequence=[
        COLORES["principal"],
        COLORES["secundario"]
    ],
    title="Composición de las características predictoras"
)

figuras_fase4["caracteristicas"].update_traces(
    textinfo="label+value"
)

figuras_fase4["caracteristicas"].add_annotation(
    text="29<br>Características",
    x=0.5,
    y=0.5,
    showarrow=False,
    font_size=17
)

# 8.2 Verificación de StandardScaler.

df_scaler_grafico_f4 = (
    df_estandarizacion_f4.melt(
        id_vars=["Problema", "Variable"],
        value_vars=[
            "Media_transformada",
            "Desviacion_transformada"
        ],
        var_name="Estadistico",
        value_name="Valor"
    )
)

figuras_fase4["estandarizacion"] = px.scatter(
    df_scaler_grafico_f4,
    x="Variable",
    y="Valor",
    color="Estadistico",
    facet_col="Problema",
    title="Verificación de StandardScaler en entrenamiento"
)

figuras_fase4["estandarizacion"].update_traces(
    marker_size=10
)

figuras_fase4["estandarizacion"].update_xaxes(
    tickangle=-45
)

figuras_fase4["estandarizacion"].add_hline(
    y=0,
    line_dash="dot",
    line_color="#94A3B8"
)

figuras_fase4["estandarizacion"].add_hline(
    y=1,
    line_dash="dot",
    line_color="#94A3B8"
)

# ==========================================================
# 9. PARTICIONES DEL DATASET
# ==========================================================

figuras_fase4["particiones"] = px.bar(
    df_matrices_f4,
    x="Problema",
    y="Registros",
    color="Conjunto",
    barmode="stack",
    text_auto=True,
    category_orders={
        "Conjunto": orden_conjuntos_f4
    },
    color_discrete_sequence=[
        COLORES["principal"],
        COLORES["secundario"],
        COLORES["acento"]
    ],
    title="Particiones de regresión y clasificación",
    hover_data=[
        "Caracteristicas",
        "Valores_faltantes"
    ]
)

# ==========================================================
# 10. CUMPLIMIENTO DE REQUERIMIENTOS
# ==========================================================

figuras_fase4["cumplimiento"] = px.pie(
    df_estados_f4,
    names="Estado",
    values="Cantidad",
    hole=0.55,
    color="Estado",
    color_discrete_map={
        "Cumplido": "#16A34A",
        "No aplica": "#94A3B8"
    },
    title="Cumplimiento de requerimientos técnicos"
)

figuras_fase4["cumplimiento"].update_traces(
    textinfo="label+value"
)

figuras_fase4["cumplimiento"].add_annotation(
    text=f"{cumplimiento_pct_f4:.0f}%<br>Aplicables",
    x=0.5,
    y=0.5,
    showarrow=False,
    font_size=17
)

# ==========================================================
# 11. ESTILO VISUAL UNIFICADO
# ==========================================================

for nombre, figura in figuras_fase4.items():

    figura.update_layout(
        template="plotly_white",
        font=dict(
            family="Arial",
            size=12,
            color=COLORES["texto"]
        ),
        title_font_size=17,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(
            l=45, r=35, t=70, b=90
        ),
        height=450
    )

# ==========================================================
# 12. CONSOLIDAR INDICADORES DE FASE 4
# ==========================================================

df_indicadores_f4 = pd.DataFrame({
    "Indicador": [
        "Completitud del dataset (%)",
        "Características predictoras",
        "Variables derivadas",
        "RMSE regresión lineal en prueba",
        "RMSE Dummy Regressor en prueba",
        "Reducción relativa RMSE (%)",
        "Accuracy regresión logística",
        "AUC-ROC regresión logística",
        "Mejora Accuracy (pp)",
        "Mejora F1 Macro (pp)",
        "Requerimientos cumplidos",
        "Requerimientos no aplicables",
        "Cumplimiento aplicable (%)"
    ],
    "Valor": [
        completitud_f4,
        indicadores_preparacion_f4[
            "Características predictoras finales"
        ],
        indicadores_preparacion_f4[
            "Variables derivadas"
        ],
        rmse_lineal_f4,
        rmse_dummy_f4,
        mejora_rmse_f4,
        accuracy_logistica_f4,
        auc_logistica_f4,
        mejora_accuracy_f4,
        mejora_f1_macro_f4,
        cumplidos_f4,
        no_aplica_f4,
        cumplimiento_pct_f4
    ]
})

# ==========================================================
# 13. EXPORTAR TABLAS Y GRÁFICAS
# ==========================================================

tablas_consolidadas_f4 = {
    "indicadores_fase4": df_indicadores_f4,
    "conteo_outliers": df_conteo_outliers_f4,
    "composicion_caracteristicas": df_caracteristicas_f4,
    "resumen_cumplimiento": df_estados_f4
}

for nombre, tabla in tablas_consolidadas_f4.items():

    tabla.to_csv(
        FASE4 / f"{nombre}.csv",
        index=False,
        encoding="utf-8-sig"
    )

CARPETA_GRAFICAS_F4 = FASE4 / "graficas"

CARPETA_GRAFICAS_F4.mkdir(
    parents=True,
    exist_ok=True
)

for nombre, figura in figuras_fase4.items():

    figura.write_html(
        str(CARPETA_GRAFICAS_F4 / f"{nombre}.html"),
        include_plotlyjs=True
    )

    figura.write_json(
        str(CARPETA_GRAFICAS_F4 / f"{nombre}.json")
    )

# ==========================================================
# 14. VALIDACIONES DE INTEGRIDAD
# ==========================================================

assert len(df_regresion_f4) == 6
assert len(df_clasificacion_f4) == 6

assert np.isclose(
    rmse_lineal_f4, 2568.032012
)

assert np.isclose(
    accuracy_logistica_f4, 0.90
)

assert np.isclose(
    auc_logistica_f4, 0.975140,
    atol=0.000001
)

assert np.isclose(
    mejora_rmse_f4, 4.150970,
    atol=0.0001
)

assert np.isclose(
    mejora_accuracy_f4, 28.181818,
    atol=0.0001
)

assert np.isclose(
    mejora_f1_macro_f4, 51.156648,
    atol=0.0001
)

assert completitud_f4 == 100

assert df_caracteristicas_f4[
    "Cantidad"
].sum() == 29

assert (
    df_matrices_f4["Caracteristicas"] == 29
).all()

assert (
    df_matrices_f4["Valores_faltantes"] == 0
).all()

assert (
    df_matrices_f4.groupby("Problema")[
        "Registros"
    ].sum() == 731
).all()

assert np.allclose(
    df_estandarizacion_f4["Media_transformada"],
    0,
    atol=1e-10
)

assert np.allclose(
    df_estandarizacion_f4[
        "Desviacion_transformada"
    ],
    1,
    atol=1e-10
)

assert cumplidos_f4 == 12
assert no_aplica_f4 == 1
assert cumplimiento_pct_f4 == 100

assert len(figuras_fase4) == 13

assert all(
    (CARPETA_GRAFICAS_F4 / f"{nombre}.html").exists()
    and
    (CARPETA_GRAFICAS_F4 / f"{nombre}.json").exists()
    for nombre in figuras_fase4
)

# ==========================================================
# 15. PRESENTACIÓN DE RESULTADOS
# ==========================================================

print("=" * 60)
print("FASE 4 - CONSOLIDACIÓN Y VISUALIZACIONES")
print("=" * 60)

print("\n=== INDICADORES PRINCIPALES ===")

print(
    df_indicadores_f4.round(6).to_string(
        index=False
    )
)

print("\n=== OUTLIERS POR VARIABLE ===")

print(
    df_conteo_outliers_f4.round(4).to_string(
        index=False
    )
)

print("\n=== CARACTERÍSTICAS PREDICTORAS ===")

print(
    df_caracteristicas_f4.to_string(
        index=False
    )
)

print("\n=== DIMENSIONES DE MATRICES FINALES ===")

print(
    df_matrices_f4.to_string(index=False)
)

print("\n=== CUMPLIMIENTO ===")

print(
    df_estados_f4.to_string(index=False)
)

print("\n=== GRÁFICAS INTERACTIVAS ===")

for nombre in figuras_fase4:
    print(f"{nombre}: OK")

print("\n=== VALIDACIONES ===")

print("Identidad del dataset: OK")
print("Integridad de métricas originales: OK")
print("Composición de características: OK")
print("Particiones del dataset: OK")
print("Verificación de StandardScaler: OK")
print("Cumplimiento de requerimientos: OK")
print("Modelos reentrenados: 0")

print(
    "Tablas consolidadas nuevas:",
    len(tablas_consolidadas_f4)
)

print(
    "Gráficas interactivas:",
    len(figuras_fase4)
)

print(
    "Imágenes originales conservadas:",
    len(df_imagenes_f4)
)

print(
    "\nFase 4 - Consolidación finalizada correctamente."
)

In [ ]:
# ==========================================================
# DASHBOARD GLOBAL - CELDA 14
# Integración del resumen ejecutivo
# ==========================================================

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# ==========================================================
# 1. CONFIGURAR DIRECTORIOS
# ==========================================================

RESUMEN = RESULTADOS / "Resumen_Ejecutivo"

RESUMEN.mkdir(
    parents=True,
    exist_ok=True
)

CARPETA_GRAFICAS_RESUMEN = RESUMEN / "graficas"

CARPETA_GRAFICAS_RESUMEN.mkdir(
    parents=True,
    exist_ok=True
)

# ==========================================================
# 2. INVENTARIO GLOBAL DE LAS CUATRO FASES
# ==========================================================

inventario_fases = pd.DataFrame({
    "Fase": [
        "Fase 1",
        "Fase 2",
        "Fase 3",
        "Fase 4"
    ],
    "Componente": [
        "Análisis exploratorio de datos",
        "Regresión simple y múltiple",
        "Modelado avanzado",
        "Preprocesamiento y baseline"
    ],
    "Graficas_interactivas": [
        len(figuras_fase1),
        len(figuras_fase2),
        len(figuras_fase3),
        len(figuras_fase4)
    ],
    "Imagenes_originales": [
        0,
        len(df_imagenes_f2),
        len(df_imagenes_f3),
        len(df_imagenes_f4)
    ],
    "Procedencia": [
        "EDA reconstruido desde el dataset",
        "Notebook original",
        "Notebook original",
        "Archivos CSV originales"
    ],
    "Estado": [
        "Consolidado",
        "Consolidado",
        "Consolidado",
        "Consolidado"
    ]
})

total_graficas_fases = int(
    inventario_fases[
        "Graficas_interactivas"
    ].sum()
)

total_imagenes_fases = int(
    inventario_fases[
        "Imagenes_originales"
    ].sum()
)

# ==========================================================
# 3. CONSOLIDAR MÉTRICAS DE REGRESIÓN
# ==========================================================

# Las métricas se recuperan de las tablas originales.
# Se conservan las diferencias metodológicas entre fases.

regresion_global_f2 = (
    df_evaluacion_f2.loc[
        df_evaluacion_f2["Conjunto"] == "Prueba",
        ["MAE", "RMSE", "R2"]
    ].copy()
)

regresion_global_f2["Modelo"] = (
    modelo_seleccionado_f2
)

regresion_global_f2["Fase"] = "Fase 2"

regresion_global_f3 = (
    df_regresion_f3.loc[
        df_regresion_f3["Conjunto"] == "Prueba",
        ["Modelo", "MAE", "RMSE", "R2"]
    ].copy()
)

regresion_global_f3["Fase"] = "Fase 3"

regresion_global_f4 = (
    df_regresion_f4.loc[
        df_regresion_f4["Conjunto"] == "Prueba",
        ["Modelo", "MAE", "RMSE", "R2"]
    ].copy()
)

regresion_global_f4["Fase"] = "Fase 4"

df_regresion_global = pd.concat(
    [
        regresion_global_f2,
        regresion_global_f3,
        regresion_global_f4
    ],
    ignore_index=True
)

df_regresion_global = df_regresion_global[
    ["Fase", "Modelo", "MAE", "RMSE", "R2"]
]

df_regresion_global["Etiqueta"] = (
    df_regresion_global["Fase"]
    + " | "
    + df_regresion_global["Modelo"]
)

df_regresion_global["Particion"] = (
    "Cronológica"
)

# ==========================================================
# 4. CONSOLIDAR MÉTRICAS DE CLASIFICACIÓN
# ==========================================================

columnas_clasificacion_global = [
    "Modelo",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "AUC_ROC"
]

clasificacion_global_f3 = (
    df_clasificacion_test_f3[
        columnas_clasificacion_global
    ].copy()
)

clasificacion_global_f3["Fase"] = "Fase 3"

clasificacion_global_f3["Particion"] = (
    "Cronológica"
)

clasificacion_global_f4 = (
    df_clasificacion_f4.loc[
        df_clasificacion_f4["Conjunto"] == "Prueba",
        columnas_clasificacion_global
    ].copy()
)

clasificacion_global_f4["Fase"] = "Fase 4"

clasificacion_global_f4["Particion"] = (
    "Aleatoria estratificada"
)

df_clasificacion_global = pd.concat(
    [
        clasificacion_global_f3,
        clasificacion_global_f4
    ],
    ignore_index=True
)

df_clasificacion_global = df_clasificacion_global[
    [
        "Fase",
        "Modelo",
        "Particion",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "AUC_ROC"
    ]
]

# No se establece un ranking global de clasificación
# porque los conjuntos de evaluación son diferentes.

# ==========================================================
# 5. INDICADORES EJECUTIVOS
# ==========================================================

df_indicadores_ejecutivos = pd.DataFrame([
    {
        "Indicador": "Total de alquileres",
        "Valor": total_alquileres,
        "Unidad": "Alquileres",
        "Fuente": "Dataset original"
    },
    {
        "Indicador": "Promedio diario",
        "Valor": promedio_diario,
        "Unidad": "Alquileres/día",
        "Fuente": "Dataset original"
    },
    {
        "Indicador": "Crecimiento anual",
        "Valor": crecimiento_anual,
        "Unidad": "%",
        "Fuente": "Dataset original"
    },
    {
        "Indicador": "Completitud",
        "Valor": porcentaje_completitud,
        "Unidad": "%",
        "Fuente": "Fase 1"
    },
    {
        "Indicador": "Reducción MAE regresión múltiple",
        "Valor": mejora_mae_f2,
        "Unidad": "%",
        "Fuente": "Fase 2 - Validación"
    },
    {
        "Indicador": "R2 regresión múltiple",
        "Valor": metricas_test_f2["R2"],
        "Unidad": "Coeficiente",
        "Fuente": "Fase 2 - Prueba"
    },
    {
        "Indicador": "Silueta K-Means",
        "Valor": df_clustering_f3.loc[
            df_clustering_f3["Algoritmo"] == "K-Means",
            "Silueta"
        ].iloc[0],
        "Unidad": "Coeficiente",
        "Fuente": "Fase 3"
    },
    {
        "Indicador": "Accuracy regresión logística",
        "Valor": accuracy_logistica_f4 * 100,
        "Unidad": "%",
        "Fuente": "Fase 4 - Prueba estratificada"
    },
    {
        "Indicador": "AUC-ROC regresión logística",
        "Valor": auc_logistica_f4,
        "Unidad": "Coeficiente",
        "Fuente": "Fase 4 - Prueba estratificada"
    },
    {
        "Indicador": "Características predictoras",
        "Valor": indicadores_preparacion_f4[
            "Características predictoras finales"
        ],
        "Unidad": "Características",
        "Fuente": "Fase 4"
    },
    {
        "Indicador": "Cumplimiento aplicable",
        "Valor": cumplimiento_pct_f4,
        "Unidad": "%",
        "Fuente": "Fase 4 - Verificación interna"
    }
])

# ==========================================================
# 6. VISUALIZACIONES DEL RESUMEN EJECUTIVO
# ==========================================================

figuras_resumen = {}

# 6.1 Evolución mensual de alquileres.

df_mensual_ejecutivo = demanda_mensual.copy()

df_mensual_ejecutivo["Año"] = (
    df_mensual_ejecutivo["Mes"].dt.year.astype(str)
)

df_mensual_ejecutivo["Numero_mes"] = (
    df_mensual_ejecutivo["Mes"].dt.month
)

nombres_meses = {
    1: "Ene", 2: "Feb", 3: "Mar",
    4: "Abr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Ago", 9: "Sep",
    10: "Oct", 11: "Nov", 12: "Dic"
}

df_mensual_ejecutivo["Nombre_mes"] = (
    df_mensual_ejecutivo["Numero_mes"]
    .map(nombres_meses)
)

figuras_resumen["demanda_mensual"] = px.line(
    df_mensual_ejecutivo,
    x="Nombre_mes",
    y="Alquileres",
    color="Año",
    markers=True,
    category_orders={
        "Nombre_mes": list(nombres_meses.values())
    },
    color_discrete_sequence=[
        COLORES["principal"],
        COLORES["secundario"]
    ],
    title="Evolución mensual de la demanda"
)

figuras_resumen["demanda_mensual"].update_traces(
    line_width=3,
    marker_size=7
)

# 6.2 Demanda anual.

figuras_resumen["demanda_anual"] = px.bar(
    demanda_anual.assign(
        Año=demanda_anual["Año"].astype(str)
    ),
    x="Año",
    y="Alquileres",
    color="Año",
    text_auto=".3s",
    color_discrete_sequence=[
        COLORES["principal"],
        COLORES["secundario"]
    ],
    title="Total de alquileres por año"
)

figuras_resumen["demanda_anual"].update_layout(
    showlegend=False
)

# 6.3 Participación por tipo de usuario.

df_usuarios_global = pd.DataFrame({
    "Tipo": ["Registrados", "Casuales"],
    "Alquileres": [
        total_registrados,
        total_casuales
    ]
})

figuras_resumen["tipo_usuario"] = px.pie(
    df_usuarios_global,
    names="Tipo",
    values="Alquileres",
    hole=0.60,
    color_discrete_sequence=[
        COLORES["principal"],
        COLORES["secundario"]
    ],
    title="Participación por tipo de usuario"
)

figuras_resumen["tipo_usuario"].update_traces(
    textinfo="label+percent"
)

# 6.4 Serie temporal diaria.

figuras_resumen["demanda_diaria"] = px.line(
    df,
    x="dteday",
    y="cnt",
    title="Serie temporal de alquileres diarios",
    labels={
        "dteday": "Fecha",
        "cnt": "Alquileres"
    }
)

figuras_resumen["demanda_diaria"].update_traces(
    line_color=COLORES["principal"],
    line_width=1.5
)

figuras_resumen["demanda_diaria"].update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=[
            dict(
                count=3,
                label="3 meses",
                step="month",
                stepmode="backward"
            ),
            dict(
                count=6,
                label="6 meses",
                step="month",
                stepmode="backward"
            ),
            dict(
                count=1,
                label="1 año",
                step="year",
                stepmode="backward"
            ),
            dict(
                step="all",
                label="Todo"
            )
        ]
    )
)

# 6.5 Comparación contextual de RMSE.

figuras_resumen["regresion_global_rmse"] = px.bar(
    df_regresion_global,
    x="RMSE",
    y="Etiqueta",
    color="Fase",
    orientation="h",
    text_auto=".2f",
    title="Regresión: RMSE en prueba por fase",
    color_discrete_sequence=[
        COLORES["principal"],
        COLORES["secundario"],
        COLORES["acento"]
    ]
)

figuras_resumen[
    "regresion_global_rmse"
].update_yaxes(
    categoryorder="array",
    categoryarray=(
        df_regresion_global["Etiqueta"].tolist()[::-1]
    )
)

# 6.6 Comparación contextual de R2.

figuras_resumen["regresion_global_r2"] = px.bar(
    df_regresion_global,
    x="R2",
    y="Etiqueta",
    color="Fase",
    orientation="h",
    text_auto=".4f",
    title="Regresión: R² en prueba por fase",
    color_discrete_sequence=[
        COLORES["principal"],
        COLORES["secundario"],
        COLORES["acento"]
    ]
)

figuras_resumen["regresion_global_r2"].add_vline(
    x=0,
    line_dash="dash",
    line_color=COLORES["negativo"]
)

# Advertencia metodológica en las comparaciones.

for nombre in [
    "regresion_global_rmse",
    "regresion_global_r2"
]:

    figuras_resumen[nombre].add_annotation(
        text=(
            "Comparación contextual: "
            "diferentes características y preprocesamientos."
        ),
        xref="paper",
        yref="paper",
        x=0,
        y=-0.22,
        showarrow=False,
        font=dict(size=11),
        align="left"
    )

# 6.7 Clasificación separada por protocolo.

df_clasificacion_ejecutivo = (
    df_clasificacion_global.melt(
        id_vars=[
            "Fase",
            "Modelo",
            "Particion"
        ],
        value_vars=[
            "Accuracy",
            "F1",
            "AUC_ROC"
        ],
        var_name="Metrica",
        value_name="Valor"
    )
)

figuras_resumen["clasificacion_por_fase"] = px.bar(
    df_clasificacion_ejecutivo,
    x="Metrica",
    y="Valor",
    color="Modelo",
    facet_col="Fase",
    barmode="group",
    text_auto=".3f",
    range_y=[0, 1.12],
    hover_data=["Particion"],
    title="Clasificación por protocolo de evaluación"
)

figuras_resumen[
    "clasificacion_por_fase"
].add_annotation(
    text=(
        "Fase 3: cronológica. "
        "Fase 4: aleatoria estratificada. "
        "No son directamente comparables."
    ),
    xref="paper",
    yref="paper",
    x=0,
    y=-0.22,
    showarrow=False,
    font=dict(size=11),
    align="left"
)

# 6.8 Inventario de visualizaciones.

figuras_resumen["evidencias_por_fase"] = px.bar(
    inventario_fases,
    x="Fase",
    y=[
        "Graficas_interactivas",
        "Imagenes_originales"
    ],
    barmode="group",
    text_auto=True,
    title="Evidencias visuales por fase",
    color_discrete_sequence=[
        COLORES["principal"],
        COLORES["secundario"]
    ]
)

# ==========================================================
# 7. APLICAR ESTILO VISUAL
# ==========================================================

for nombre, figura in figuras_resumen.items():

    figura.update_layout(
        template="plotly_white",
        font=dict(
            family="Arial",
            size=12,
            color=COLORES["texto"]
        ),
        title_font_size=17,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(
            l=50,
            r=40,
            t=85,
            b=110
        ),
        height=450
    )

# ==========================================================
# 8. HALLAZGOS Y CONCLUSIONES GLOBALES
# ==========================================================

df_hallazgos_globales = pd.DataFrame([
    {
        "Fase": "Global",
        "Hallazgo": (
            f"La demanda aumentó {crecimiento_anual:.2f}% "
            "entre 2011 y 2012."
        ),
        "Implicacion": (
            "Considerar crecimiento y estacionalidad "
            "en la planificación de capacidad."
        )
    },
    {
        "Fase": "Fase 1",
        "Hallazgo": (
            "La temperatura y la sensación térmica "
            "presentan asociación positiva con la demanda."
        ),
        "Implicacion": (
            "Incorporar variables meteorológicas "
            "en los modelos predictivos."
        )
    },
    {
        "Fase": "Fase 2",
        "Hallazgo": (
            f"La regresión múltiple redujo el MAE "
            f"de validación en {mejora_mae_f2:.2f}%."
        ),
        "Implicacion": (
            "Las variables adicionales aportan "
            "capacidad predictiva frente al modelo simple."
        )
    },
    {
        "Fase": "Fase 3",
        "Hallazgo": (
            "Random Forest fue seleccionado en regresión "
            "por RMSE de validación y KNN en clasificación "
            "por F1 de validación temporal."
        ),
        "Implicacion": (
            "La selección debe respetar el protocolo "
            "de validación definido previamente."
        )
    },
    {
        "Fase": "Fase 3",
        "Hallazgo": (
            "Los coeficientes de silueta de ambos "
            "algoritmos de clustering son bajos."
        ),
        "Implicacion": (
            "No se evidencia una separación fuerte "
            "entre los grupos identificados."
        )
    },
    {
        "Fase": "Fase 4",
        "Hallazgo": (
            f"La regresión logística alcanzó "
            f"{accuracy_logistica_f4:.1%} de Accuracy "
            "en prueba estratificada."
        ),
        "Implicacion": (
            "El baseline supervisado supera al "
            "clasificador de referencia en este protocolo."
        )
    }
])

# ==========================================================
# 9. LIMITACIONES METODOLÓGICAS
# ==========================================================

df_limitaciones_globales = pd.DataFrame([
    {
        "Ambito": "Procedencia",
        "Limitacion": (
            "La Fase 1 fue reconstruida a partir "
            "del dataset original."
        )
    },
    {
        "Ambito": "Representatividad",
        "Limitacion": (
            "El dataset corresponde exclusivamente "
            "al periodo 2011-2012."
        )
    },
    {
        "Ambito": "Correlación",
        "Limitacion": (
            "Las asociaciones estadísticas no "
            "demuestran causalidad."
        )
    },
    {
        "Ambito": "Fuga de información",
        "Limitacion": (
            "casual y registered no deben utilizarse "
            "para predecir cnt, porque su suma define "
            "directamente la variable objetivo."
        )
    },
    {
        "Ambito": "Regresión",
        "Limitacion": (
            "Las fases utilizan diferentes características "
            "y estrategias de preprocesamiento."
        )
    },
    {
        "Ambito": "Regresión en R",
        "Limitacion": (
            "Se identificaron cinco coeficientes "
            "no estimables por singularidades."
        )
    },
    {
        "Ambito": "Clasificación temporal",
        "Limitacion": (
            "La validación externa de Fase 3 "
            "contiene exclusivamente demanda alta."
        )
    },
    {
        "Ambito": "Comparabilidad",
        "Limitacion": (
            "La clasificación de Fase 3 es cronológica "
            "y la de Fase 4 aleatoria estratificada."
        )
    },
    {
        "Ambito": "Umbral de clasificación",
        "Limitacion": (
            "El umbral de 3974 alquileres fue heredado "
            "de Fase 3. Existe posible solapamiento "
            "entre su conjunto de estimación y las "
            "particiones aleatorias de Fase 4."
        )
    },
    {
        "Ambito": "Generalización",
        "Limitacion": (
            "El desempeño de clasificación de Fase 4 "
            "no demuestra capacidad de predicción "
            "sobre periodos futuros."
        )
    },
    {
        "Ambito": "Evidencias",
        "Limitacion": (
            "Los modelos de las fases anteriores "
            "no fueron reentrenados en este dashboard."
        )
    }
])

# ==========================================================
# 10. EXPORTAR TABLAS CONSOLIDADAS
# ==========================================================

tablas_resumen = {
    "inventario_fases": inventario_fases,
    "indicadores_ejecutivos": df_indicadores_ejecutivos,
    "regresion_global": df_regresion_global,
    "clasificacion_global": df_clasificacion_global,
    "hallazgos_globales": df_hallazgos_globales,
    "limitaciones_globales": df_limitaciones_globales
}

for nombre, tabla in tablas_resumen.items():

    tabla.to_csv(
        RESUMEN / f"{nombre}.csv",
        index=False,
        encoding="utf-8-sig"
    )

for nombre, figura in figuras_resumen.items():

    figura.write_html(
        str(
            CARPETA_GRAFICAS_RESUMEN /
            f"{nombre}.html"
        ),
        include_plotlyjs=True
    )

    figura.write_json(
        str(
            CARPETA_GRAFICAS_RESUMEN /
            f"{nombre}.json"
        )
    )

# ==========================================================
# 11. VALIDACIONES GLOBALES
# ==========================================================

assert len(inventario_fases) == 4

assert total_graficas_fases == 38

assert total_imagenes_fases == 25

assert len(df_regresion_global) == 5

assert len(df_clasificacion_global) == 4

assert (
    df_clasificacion_global["Particion"].nunique()
    == 2
)

assert total_alquileres == int(df["cnt"].sum())

assert (
    total_casuales + total_registrados
    == total_alquileres
)

assert np.isclose(
    porcentaje_casuales + porcentaje_registrados,
    100
)

assert len(figuras_resumen) == 8

assert all(
    (RESUMEN / f"{nombre}.csv").exists()
    for nombre in tablas_resumen
)

assert all(
    (
        CARPETA_GRAFICAS_RESUMEN /
        f"{nombre}.html"
    ).exists()
    and
    (
        CARPETA_GRAFICAS_RESUMEN /
        f"{nombre}.json"
    ).exists()
    for nombre in figuras_resumen
)

# ==========================================================
# 12. PRESENTACIÓN DE RESULTADOS
# ==========================================================

print("=" * 60)
print("DASHBOARD GLOBAL - RESUMEN EJECUTIVO")
print("=" * 60)

print("\n=== INVENTARIO DE FASES ===")

print(
    inventario_fases.to_string(index=False)
)

print("\n=== INDICADORES EJECUTIVOS ===")

print(
    df_indicadores_ejecutivos.round(4)
    .to_string(index=False)
)

print("\n=== REGRESIÓN GLOBAL ===")

print(
    df_regresion_global[
        ["Fase", "Modelo", "MAE", "RMSE", "R2"]
    ].round(4).to_string(index=False)
)

print("\n=== CLASIFICACIÓN POR PROTOCOLO ===")

print(
    df_clasificacion_global.round(4)
    .to_string(index=False)
)

print("\n=== GRÁFICAS DEL RESUMEN EJECUTIVO ===")

for nombre in figuras_resumen:

    print(f"{nombre}: OK")

print("\n=== VALIDACIONES ===")

print("Integración de las cuatro fases: OK")
print("Integridad de indicadores: OK")
print("Trazabilidad de métricas: OK")
print("Separación de protocolos: OK")
print("Exportación de tablas y gráficas: OK")

print("\n=== INVENTARIO GLOBAL ===")

print("Fases consolidadas:", len(inventario_fases))

print(
    "Gráficas de las cuatro fases:",
    total_graficas_fases
)

print(
    "Gráficas del resumen ejecutivo:",
    len(figuras_resumen)
)

print(
    "Total de gráficas interactivas:",
    total_graficas_fases + len(figuras_resumen)
)

print(
    "Imágenes originales:",
    total_imagenes_fases
)

print("Hallazgos:", len(df_hallazgos_globales))

print(
    "Limitaciones documentadas:",
    len(df_limitaciones_globales)
)

print("Modelos reentrenados: 0")

print(
    "\nResumen ejecutivo consolidado correctamente."
)

In [ ]:
# ==========================================================
# DASHBOARD GLOBAL - CELDA 15
# Construcción de la interfaz gráfica en Dash
# ==========================================================

import base64
import mimetypes
from pathlib import Path

from dash import Dash, dcc, html, dash_table, Input, Output
import dash_bootstrap_components as dbc

# ==========================================================
# 1. CONFIGURACIÓN DE LA APLICACIÓN
# ==========================================================

app = Dash(
    __name__,
    external_stylesheets=[dbc.themes.BOOTSTRAP],
    suppress_callback_exceptions=True,
    title="Bike Sharing Analytics"
)

app.server.config["JSON_AS_ASCII"] = False

ESTILO_FONDO = "#F1F5F9"
ESTILO_ENCABEZADO = "#142C49"
ESTILO_TEXTO = "#334155"

ESTILO_TARJETA = {
    "border": "1px solid #E2E8F0",
    "borderRadius": "12px",
    "boxShadow": "0 2px 6px rgba(0,0,0,0.05)",
    "backgroundColor": "#FFFFFF",
    "height": "100%"
}

CONFIG_GRAFICAS = {
    "displaylogo": False,
    "responsive": True,
    "scrollZoom": False,
    "toImageButtonOptions": {
        "format": "png",
        "scale": 2
    }
}

# ==========================================================
# 2. FUNCIONES AUXILIARES DE FORMATO
# ==========================================================

def formato_entero(valor):

    return f"{int(valor):,}".replace(",", ".")


def formato_decimal(valor, decimales=2):

    return (
        f"{float(valor):,.{decimales}f}"
        .replace(",", "_")
        .replace(".", ",")
        .replace("_", ".")
    )


def formato_porcentaje(valor, decimales=2):

    return f"{formato_decimal(valor, decimales)} %"


# ==========================================================
# 3. COMPONENTES REUTILIZABLES
# ==========================================================

# 3.1 Tarjetas de indicadores KPI.

def tarjeta_kpi(titulo, valor, descripcion="",
                color="#2563EB"):

    return dbc.Card(
        dbc.CardBody([
            html.Div(
                titulo,
                style={
                    "fontSize": "13px",
                    "color": "#64748B",
                    "fontWeight": "600"
                }
            ),

            html.H3(
                valor,
                style={
                    "color": color,
                    "fontWeight": "700",
                    "marginTop": "10px",
                    "marginBottom": "8px"
                }
            ),

            html.Small(
                descripcion,
                style={"color": "#64748B"}
            )
        ]),
        style={
            **ESTILO_TARJETA,
            "borderTop": f"4px solid {color}"
        }
    )


# 3.2 Fila adaptable de indicadores.

def fila_kpis(indicadores):

    return dbc.Row(
        [
            dbc.Col(
                tarjeta_kpi(*indicador),
                xs=12,
                sm=6,
                lg=3,
                className="mb-3"
            )
            for indicador in indicadores
        ],
        className="g-3 mb-3"
    )


# 3.3 Encabezado de sección.

def encabezado_seccion(titulo, descripcion=""):

    return html.Div([
        html.H3(
            titulo,
            style={
                "fontWeight": "700",
                "color": ESTILO_ENCABEZADO
            }
        ),

        html.P(
            descripcion,
            style={"color": "#64748B"}
        )
    ], className="mt-3 mb-3")


# 3.4 Bloque de interpretación metodológica.

def nota_metodologica(titulo, contenido,
                      color="info"):

    return dbc.Alert(
        [
            html.Strong(titulo),
            html.P(
                contenido,
                className="mb-0 mt-2"
            )
        ],
        color=color,
        className="mb-4"
    )


# 3.5 Tarjeta de gráfica interactiva.

def tarjeta_grafica(figura, identificador=None):

    # Crear una copia para conservar la figura original.
    figura_visual = go.Figure(figura)

    figura_visual.update_layout(
        autosize=True,
        height=450,
        margin=dict(
            l=65,
            r=35,
            t=80,
            b=100
        )
    )

    parametros = {
        "figure": figura_visual,
        "config": CONFIG_GRAFICAS,
        "responsive": True,
        "style": {
            "height": "450px",
            "minHeight": "450px",
            "width": "100%"
        }
    }

    if identificador is not None:
        parametros["id"] = identificador

    return dbc.Card(
        dbc.CardBody(
            dcc.Graph(**parametros),
            style={
                "padding": "12px",
                "minHeight": "474px"
            }
        ),
        style={
            **ESTILO_TARJETA,
            "height": "auto",
            "minHeight": "474px",
            "width": "100%",
            "overflow": "hidden"
        },
        className="mb-3"
    )

# 3.6 Cuadrícula de gráficas.

def cuadricula_graficas(figuras, prefijo,
                        columnas=2):

    ancho = 12 if columnas == 1 else 6

    return dbc.Row(
        [
            dbc.Col(
                tarjeta_grafica(
                    figura,
                    f"{prefijo}-{nombre}"
                ),
                xs=12,
                md=12,
                lg=ancho,
                className="mb-3",
                style={
                    "minWidth": "0",
                    "height": "auto"
                }
            )
            for nombre, figura in figuras.items()
        ],
        className="g-3",
        style={
            "alignItems": "flex-start"
        }
    )

# 3.7 Tablas interactivas.

def tabla_interactiva(dataframe, pagina=10):

    datos = dataframe.copy()

    for columna in datos.columns:

        if pd.api.types.is_datetime64_any_dtype(
            datos[columna]
        ):
            datos[columna] = (
                datos[columna].astype(str)
            )

        elif pd.api.types.is_float_dtype(
            datos[columna]
        ):
            datos[columna] = (
                datos[columna].round(6)
            )

    datos = datos.astype(object).where(
        pd.notna(datos),
        None
    )

    return dash_table.DataTable(
        data=datos.to_dict("records"),

        columns=[
            {
                "name": str(columna).replace("_", " "),
                "id": str(columna)
            }
            for columna in datos.columns
        ],

        page_size=pagina,
        page_action="native",
        sort_action="native",
        filter_action="native",

        export_format="csv",

        style_table={
            "overflowX": "auto"
        },

        style_header={
            "backgroundColor": "#142C49",
            "color": "white",
            "fontWeight": "bold",
            "padding": "12px"
        },

        style_cell={
            "fontFamily": "Arial",
            "fontSize": "12px",
            "padding": "10px",
            "textAlign": "left",
            "whiteSpace": "normal",
            "height": "auto",
            "minWidth": "100px",
            "maxWidth": "450px"
        },

        style_data_conditional=[
            {
                "if": {"row_index": "odd"},
                "backgroundColor": "#F8FAFC"
            }
        ]
    )


# 3.8 Sección desplegable de tablas.

def panel_tablas(tablas):

    return dbc.Accordion(
        [
            dbc.AccordionItem(
                tabla_interactiva(tabla),
                title=titulo
            )
            for titulo, tabla in tablas.items()
        ],
        start_collapsed=True,
        always_open=True,
        className="mb-4"
    )


# 3.9 Convertir imágenes originales a Base64.
# Permite visualizarlas sin depender de rutas públicas.

def imagen_base64(ruta):

    ruta = Path(ruta)

    assert ruta.is_file(), (
        f"No se encontró la imagen: {ruta}"
    )

    tipo_mime = (
        mimetypes.guess_type(str(ruta))[0]
        or "image/png"
    )

    contenido = base64.b64encode(
        ruta.read_bytes()
    ).decode("utf-8")

    return f"data:{tipo_mime};base64,{contenido}"


# 3.10 Galería de evidencias originales.

def galeria_imagenes(inventario):

    if inventario.empty:

        return html.P(
            "No se dispone de imágenes originales."
        )

    assert "Ruta" in inventario.columns, (
        "El inventario no contiene la columna Ruta."
    )

    tarjetas = []

    for _, registro in inventario.iterrows():

        ruta = Path(registro["Ruta"])

        nombre = ruta.stem.replace("_", " ")

        tarjetas.append(
            dbc.Col(
                dbc.Card([
                    dbc.CardHeader(
                        nombre,
                        style={
                            "fontWeight": "600",
                            "fontSize": "13px"
                        }
                    ),

                    dbc.CardBody(
                        html.Img(
                            src=imagen_base64(ruta),
                            alt=nombre,
                            style={
                                "width": "100%",
                                "height": "auto",
                                "objectFit": "contain"
                            }
                      )
                    )
                ], style=ESTILO_TARJETA),

                xs=12,
                lg=6,
                className="mb-3"
            )
        )

    return dbc.Row(
        tarjetas,
        className="g-3"
    )


# 3.11 Panel desplegable de evidencias.

def panel_evidencias(inventario):

    return dbc.Accordion([
        dbc.AccordionItem(
            galeria_imagenes(inventario),
            title=(
                f"Evidencias gráficas originales "
                f"({len(inventario)})"
            )
        )
    ], start_collapsed=True, className="mb-4")


# ==========================================================
# 4. PESTAÑA: RESUMEN EJECUTIVO
# ==========================================================

def vista_resumen():

    return html.Div([

        encabezado_seccion(
            "Resumen ejecutivo",
            "Visión integral de la demanda de bicicletas "
            "y los resultados de las cuatro fases."
        ),

        fila_kpis([
            (
                "Total de alquileres",
                formato_entero(total_alquileres),
                "Periodo 2011–2012",
                "#2563EB"
            ),
            (
                "Promedio diario",
                formato_entero(round(promedio_diario)),
                "Alquileres por día",
                "#0D9488"
            ),
            (
                "Crecimiento anual",
                "+" + formato_porcentaje(
                    crecimiento_anual
                ),
                "2012 frente a 2011",
                "#16A34A"
            ),
            (
                "Completitud",
                formato_porcentaje(
                    porcentaje_completitud, 0
                ),
                "Sin valores faltantes",
                "#7C3AED"
            )
        ]),

        # Filtros que se conectarán en la celda 16.

        dbc.Card(
            dbc.CardBody([
                html.H5("Filtros de demanda"),

                dbc.Row([
                    dbc.Col([
                        html.Label("Año"),

                        dcc.Dropdown(
                            id="filtro-anio",
                            options=[
                                {
                                    "label": "Todos",
                                    "value": "todos"
                                },
                                {
                                    "label": "2011",
                                    "value": "2011"
                                },
                                {
                                    "label": "2012",
                                    "value": "2012"
                                }
                            ],
                            value="todos",
                            clearable=False
                        )
                    ], xs=12, md=4),

                    dbc.Col([
                        html.Label("Tipo de usuario"),

                        dcc.Dropdown(
                            id="filtro-usuario",
                            options=[
                                {
                                    "label": "Todos",
                                    "value": "cnt"
                                },
                                {
                                    "label": "Registrados",
                                    "value": "registered"
                                },
                                {
                                    "label": "Casuales",
                                    "value": "casual"
                                }
                            ],
                            value="cnt",
                            clearable=False
                        )
                    ], xs=12, md=4),

                    dbc.Col([
                        html.Label("Granularidad"),

                        dcc.Dropdown(
                            id="filtro-granularidad",
                            options=[
                                {
                                    "label": "Diaria",
                                    "value": "diaria"
                                },
                                {
                                    "label": "Mensual",
                                    "value": "mensual"
                                }
                            ],
                            value="mensual",
                            clearable=False
                        )
                    ], xs=12, md=4)

                ], className="g-3"),

                html.Div(
                    id="kpis-filtrados",
                    className="mt-3"
                )

            ]),

            style=ESTILO_TARJETA,
            className="mb-4"
        ),

        tarjeta_grafica(
            figuras_resumen["demanda_mensual"],
            "grafica-demanda-filtrada"
        ),

        cuadricula_graficas(
            {
                nombre: figura
                for nombre, figura
                in figuras_resumen.items()
                if nombre != "demanda_mensual"
            },
            "resumen"
        ),

        encabezado_seccion(
            "Hallazgos principales"
        ),

        dbc.Row([
            dbc.Col(
                dbc.Card(
                    dbc.CardBody([
                        dbc.Badge(
                            registro["Fase"],
                            color="primary",
                            className="mb-2"
                        ),

                        html.H6(
                            registro["Hallazgo"],
                            style={"fontWeight": "600"}
                        ),

                        html.P(
                            registro["Implicacion"],
                            className="mb-0",
                            style={"fontSize": "14px"}
                        )
                    ]),
                    style=ESTILO_TARJETA
                ),
                xs=12,
                lg=6,
                className="mb-3"
            )
            for _, registro
            in df_hallazgos_globales.iterrows()
        ], className="g-3"),

        encabezado_seccion(
            "Resultados y trazabilidad"
        ),

        panel_tablas({
            "Indicadores ejecutivos":
                df_indicadores_ejecutivos,

            "Resultados globales de regresión":
                df_regresion_global,

            "Clasificación por protocolo":
                df_clasificacion_global,

            "Inventario de fases":
                inventario_fases,

            "Limitaciones metodológicas":
                df_limitaciones_globales
        }),

        nota_metodologica(
            "Interpretación de resultados",
            "Las comparaciones de regresión son "
            "contextuales. Los resultados de clasificación "
            "de las fases 3 y 4 corresponden a protocolos "
            "diferentes y no permiten establecer un "
            "ganador global.",
            "warning"
        )
    ])


# ==========================================================
# 5. PESTAÑA: FASE 1 - EDA
# ==========================================================

def vista_fase1():

    return html.Div([

        encabezado_seccion(
            "Fase 1 — Análisis exploratorio de datos",
            "Caracterización estadística, calidad y "
            "relaciones entre las variables."
        ),

        fila_kpis([
            (
                "Registros",
                formato_entero(len(df)),
                "Observaciones diarias",
                "#2563EB"
            ),
            (
                "Variables originales",
                str(df.shape[1]),
                "Dataset day.csv",
                "#0D9488"
            ),
            (
                "Valores faltantes",
                str(int(df.isna().sum().sum())),
                "Completitud del 100 %",
                "#16A34A"
            ),
            (
                "Duplicados",
                str(int(df.duplicated().sum())),
                "Registros completos duplicados",
                "#7C3AED"
            )
        ]),

        nota_metodologica(
            "Procedencia de la Fase 1",
            "El análisis exploratorio fue reconstruido "
            "a partir del dataset original, debido a "
            "que no se dispone del notebook inicial."
        ),

        cuadricula_graficas(
            figuras_fase1,
            "fase1"
        ),

        encabezado_seccion(
            "Resultados estadísticos"
        ),

        panel_tablas({
            "Estadísticas descriptivas":
                estadisticas_eda,

            "Frecuencias categóricas":
                df_frecuencias,

            "Correlación de Pearson":
                correlacion_pearson.reset_index(),

            "Correlación de Spearman":
                correlacion_spearman.reset_index(),

            "Factores asociados a la demanda":
                correlacion_demanda.reset_index(),

            "Pruebas Chi-cuadrado":
                df_chi2,

            "Detección de valores atípicos":
                df_outliers_eda,

            "Calidad de datos":
                df_calidad_eda
        }),

        nota_metodologica(
            "Consideraciones estadísticas",
            "La correlación no implica causalidad. "
            "Las pruebas Chi-cuadrado son exploratorias "
            "por la posible dependencia temporal. "
            "casual y registered no deben utilizarse "
            "como predictores de cnt.",
            "warning"
        )
    ])


# ==========================================================
# 6. PESTAÑA: FASE 2 - REGRESIÓN
# ==========================================================

def vista_fase2():

    return html.Div([

        encabezado_seccion(
            "Fase 2 — Regresión simple y múltiple",
            "Evaluación predictiva, comparación de "
            "modelos y análisis complementario en R."
        ),

        fila_kpis([
            (
                "Modelo seleccionado",
                "Múltiple",
                "Menor MAE de validación",
                "#2563EB"
            ),
            (
                "Reducción de MAE",
                formato_porcentaje(mejora_mae_f2),
                "Frente al baseline en validación",
                "#16A34A"
            ),
            (
                "RMSE en prueba",
                formato_decimal(
                    metricas_test_f2["RMSE"]
                ),
                "Alquileres",
                "#0D9488"
            ),
            (
                "R² en prueba",
                formato_decimal(
                    metricas_test_f2["R2"], 4
                ),
                "Regresión múltiple",
                "#7C3AED"
            )
        ]),

        cuadricula_graficas(
            figuras_fase2,
            "fase2"
        ),

        encabezado_seccion(
            "Resultados originales"
        ),

        panel_tablas({
            "Evaluación del modelo seleccionado":
                df_evaluacion_f2,

            "Resultados de prueba":
                df_regresion_global.loc[
                    df_regresion_global["Fase"]
                    == "Fase 2"
                ]
        }),

        panel_evidencias(df_imagenes_f2),

        nota_metodologica(
            "Validación Python y R",
            "El notebook original confirmó la "
            "equivalencia de las predicciones de "
            "LinearRegression y lm(). En R se "
            "identificaron cinco coeficientes no "
            "estimables por singularidades.",
            "warning"
        )
    ])


# ==========================================================
# 7. PESTAÑA: FASE 3 - MODELADO AVANZADO
# ==========================================================

def vista_fase3():

    return html.Div([

        encabezado_seccion(
            "Fase 3 — Modelado avanzado",
            "Regresión regularizada, Random Forest, "
            "KNN, clustering y validación temporal."
        ),

        fila_kpis([
            (
                "Regresor seleccionado",
                mejor_regresion_f3,
                "RMSE de validación",
                "#2563EB"
            ),
            (
                "Clasificador seleccionado",
                mejor_clasificacion_f3,
                "F1 de validación temporal",
                "#0D9488"
            ),
            (
                "Silueta K-Means",
                formato_decimal(
                    df_clustering_f3.loc[
                        df_clustering_f3["Algoritmo"]
                        == "K-Means",
                        "Silueta"
                    ].iloc[0],
                    4
                ),
                "Separación de clústeres",
                "#7C3AED"
            ),
            (
                "Folds utilizables",
                f"{folds_validos_f3}/5",
                "Validación cruzada temporal",
                "#16A34A"
            )
        ]),

        cuadricula_graficas(
            figuras_fase3,
            "fase3"
        ),

        encabezado_seccion(
            "Métricas y configuraciones"
        ),

        panel_tablas({
            "Métricas de regresión":
                df_regresion_f3,

            "Clasificación en prueba":
                df_clasificacion_test_f3,

            "Validación temporal":
                df_clasificacion_cv_f3,

            "Clustering":
                df_clustering_f3,

            "Hiperparámetros seleccionados":
                df_configuraciones_f3,

            "Distribución temporal de clases":
                df_clases_f3,

            "Particiones de validación cruzada":
                df_folds_f3
        }),

        panel_evidencias(df_imagenes_f3),

        nota_metodologica(
            "Limitaciones del modelado avanzado",
            "La validación externa contiene únicamente "
            "demanda alta. Se empleó validación cruzada "
            "temporal con cuatro folds utilizables. "
            "Los coeficientes de silueta evidencian "
            "una separación débil de los clústeres.",
            "warning"
        )
    ])


# ==========================================================
# 8. PESTAÑA: FASE 4 - PREPROCESAMIENTO Y BASELINE
# ==========================================================

def vista_fase4():

    return html.Div([

        encabezado_seccion(
            "Fase 4 — Preprocesamiento y baseline",
            "Ingeniería de características, "
            "transformaciones sin fuga de información "
            "y evaluación de modelos de referencia."
        ),

        fila_kpis([
            (
                "Características predictoras",
                "29",
                "7 numéricas y 22 codificadas",
                "#2563EB"
            ),
            (
                "Accuracy logística",
                formato_porcentaje(
                    accuracy_logistica_f4 * 100, 1
                ),
                "Prueba aleatoria estratificada",
                "#16A34A"
            ),
            (
                "AUC-ROC",
                formato_decimal(
                    auc_logistica_f4, 4
                ),
                "Regresión logística",
                "#0D9488"
            ),
            (
                "Cumplimiento aplicable",
                formato_porcentaje(
                    cumplimiento_pct_f4, 0
                ),
                "Verificación interna",
                "#7C3AED"
            )
        ]),

        cuadricula_graficas(
            figuras_fase4,
            "fase4"
        ),

        encabezado_seccion(
            "Preparación y evaluación de datos"
        ),

        panel_tablas({
            "Resumen de preparación":
                df_preparacion_f4,

            "Métricas baseline de regresión":
                df_regresion_f4,

            "Métricas baseline de clasificación":
                df_clasificacion_f4,

            "Mejoras de los modelos":
                df_mejoras_f4,

            "Valores atípicos":
                df_outliers_f4,

            "Límites de winsorization":
                df_limites_f4,

            "Resultados de winsorization":
                df_winsorization_f4,

            "Codificación de variables":
                df_codificacion_f4,

            "Verificación de StandardScaler":
                df_estandarizacion_f4,

            "Estrategia de balanceo":
                df_balanceo_f4,

            "Dimensiones de matrices finales":
                df_matrices_f4,

            "Cumplimiento de requerimientos":
                df_cumplimiento_f4
        }),

        panel_evidencias(df_imagenes_f4),

        nota_metodologica(
            "Protocolo de evaluación",
            "La regresión utiliza una partición "
            "cronológica y la clasificación una "
            "partición aleatoria estratificada. "
            "El umbral de 3974 alquileres fue heredado "
            "de Fase 3 y existe posible solapamiento "
            "entre los datos usados para estimarlo "
            "y las particiones de Fase 4.",
            "warning"
        )
    ])


# ==========================================================
# 9. ENCABEZADO PRINCIPAL
# ==========================================================

encabezado_dashboard = html.Div([

    dbc.Container([

        html.Div(
            "UNIVERSIDAD DE LA SALLE",
            style={
                "fontSize": "12px",
                "letterSpacing": "2px",
                "color": "#BFDBFE"
            }
        ),

        html.H1(
            "Bike Sharing Analytics",
            style={
                "fontWeight": "700",
                "color": "#FFFFFF",
                "marginTop": "10px"
            }
        ),

        html.P(
            "Dashboard global — Proyecto integrador "
            "de Ciencia de Datos",
            style={
                "fontSize": "17px",
                "color": "#DBEAFE"
            }
        ),

        html.Div(
            "Maestría en Inteligencia Artificial | "
            "Adriana Patricia Fernández Infante",
            style={
                "fontSize": "13px",
                "color": "#CBD5E1"
            }
        ),

        html.Div([
            dbc.Badge(
                "Dataset UCI",
                color="primary",
                className="me-2"
            ),

            dbc.Badge(
                "2011–2012",
                color="secondary",
                className="me-2"
            ),

            dbc.Badge(
                "4 fases integradas",
                color="success"
            )
        ], className="mt-3")

    ], fluid=True)

], style={
    "backgroundColor": ESTILO_ENCABEZADO,
    "padding": "30px 20px",
    "borderBottom": "5px solid #2563EB"
})


# ==========================================================
# 10. NAVEGACIÓN PRINCIPAL
# ==========================================================

navegacion_dashboard = dbc.Tabs([

    dbc.Tab(
        label="Resumen ejecutivo",
        tab_id="resumen"
    ),

    dbc.Tab(
        label="Fase 1 · EDA",
        tab_id="fase1"
    ),

    dbc.Tab(
        label="Fase 2 · Regresión",
        tab_id="fase2"
    ),

    dbc.Tab(
        label="Fase 3 · Modelado avanzado",
        tab_id="fase3"
    ),

    dbc.Tab(
        label="Fase 4 · Baseline",
        tab_id="fase4"
    )

], id="tabs-dashboard",
   active_tab="resumen",
   className="mb-4 mt-4")


# ==========================================================
# 11. PIE DE PÁGINA
# ==========================================================

pie_dashboard = html.Footer([

    html.Hr(),

    html.P(
        "Proyecto Integrador de Ciencia de Datos "
        "— Universidad de La Salle",
        className="mb-1"
    ),

    html.P(
        "Dataset: Bike Sharing Dataset — UCI "
        "Machine Learning Repository.",
        className="mb-1"
    ),

    html.A(
        "Referencia del dataset: DOI 10.24432/C5W894",
        href="https://doi.org/10.24432/C5W894",
        target="_blank"
    ),

    html.P(
        "Resultados recuperados de evidencias "
        "originales. No se reentrenaron modelos.",
        className="mt-2"
    )

], style={
    "fontSize": "12px",
    "color": "#64748B",
    "textAlign": "center",
    "padding": "20px"
})


# ==========================================================
# 12. LAYOUT PRINCIPAL
# ==========================================================

app.layout = html.Div([

    encabezado_dashboard,

    dbc.Container([

        navegacion_dashboard,

        dcc.Loading(
            html.Div(
                id="contenido-dashboard",
                children=vista_resumen()
            ),
            type="circle"
        ),

        pie_dashboard

    ], fluid=True, className="px-3 px-lg-4")

], style={
    "backgroundColor": ESTILO_FONDO,
    "minHeight": "100vh",
    "fontFamily": "Arial, sans-serif",
    "color": ESTILO_TEXTO
})


# ==========================================================
# 13. REGISTRO DE VISTAS
# ==========================================================

vistas_dashboard = {
    "resumen": vista_resumen,
    "fase1": vista_fase1,
    "fase2": vista_fase2,
    "fase3": vista_fase3,
    "fase4": vista_fase4
}


# ==========================================================
# 14. VALIDACIÓN DE COMPONENTES
# ==========================================================

def recorrer_componentes(componente):

    if isinstance(componente, (list, tuple)):

        for elemento in componente:
            yield from recorrer_componentes(elemento)

    elif hasattr(componente, "to_plotly_json"):

        yield componente

        hijos = getattr(
            componente,
            "children",
            None
        )

        if hijos is not None:
            yield from recorrer_componentes(hijos)


inventario_vistas = []

for nombre, funcion_vista in vistas_dashboard.items():

    vista = funcion_vista()

    componentes = list(
        recorrer_componentes(vista)
    )

    ids = [
        componente.id
        for componente in componentes
        if isinstance(
            getattr(componente, "id", None),
            str
        )
    ]

    assert len(ids) == len(set(ids)), (
        f"Identificadores duplicados en {nombre}"
    )

    inventario_vistas.append({
        "Vista": nombre,

        "Componentes": len(componentes),

        "Graficas": sum(
            isinstance(c, dcc.Graph)
            for c in componentes
        ),

        "Tablas": sum(
            isinstance(c, dash_table.DataTable)
            for c in componentes
        ),

        "Imagenes": sum(
            isinstance(c, html.Img)
            for c in componentes
        ),

        "Estado": "OK"
    })


df_inventario_vistas = pd.DataFrame(
    inventario_vistas
)

# ==========================================================
# 15. VALIDACIONES GLOBALES
# ==========================================================

assert len(vistas_dashboard) == 5

assert df_inventario_vistas[
    "Graficas"
].sum() == 46

assert df_inventario_vistas[
    "Imagenes"
].sum() == 25

assert (
    df_inventario_vistas["Estado"] == "OK"
).all()

df_inventario_vistas.to_csv(
    RESULTADOS / "inventario_vistas_dashboard.csv",
    index=False,
    encoding="utf-8-sig"
)

# ==========================================================
# 16. PRESENTACIÓN DE RESULTADOS
# ==========================================================

print("=" * 60)
print("DASHBOARD GLOBAL - INTERFAZ GRÁFICA DASH")
print("=" * 60)

print("\n=== INVENTARIO DE VISTAS ===")

print(
    df_inventario_vistas.to_string(index=False)
)

print("\n=== COMPONENTES INTEGRADOS ===")

print("Pestañas:", len(vistas_dashboard))

print(
    "Gráficas interactivas:",
    df_inventario_vistas["Graficas"].sum()
)

print(
    "Tablas interactivas:",
    df_inventario_vistas["Tablas"].sum()
)

print(
    "Imágenes originales:",
    df_inventario_vistas["Imagenes"].sum()
)

print(
    "Componentes de interfaz:",
    df_inventario_vistas["Componentes"].sum()
)

print("\n=== VALIDACIONES ===")

print("Aplicación Dash creada: OK")
print("Layout principal: OK")
print("Cinco vistas construidas: OK")
print("Identificadores únicos por vista: OK")
print("Integración de gráficas: OK")
print("Integración de imágenes: OK")
print("Tablas interactivas: OK")

print("\nCallbacks: pendientes de Celda 16")
print("Servidor Dash: pendiente de ejecución")

print(
    "\nInterfaz gráfica construida correctamente."
)

In [ ]:
# ==========================================================
# DASHBOARD GLOBAL - CELDA 16
# Callbacks, filtros y navegación dinámica
# ==========================================================

from dash import Input, Output
import plotly.express as px
import pandas as pd
import numpy as np

# ==========================================================
# 1. FUNCIÓN DE NAVEGACIÓN ENTRE PESTAÑAS
# ==========================================================

def seleccionar_vista(pestana):

    # Recuperar la función asociada a la pestaña.
    # Si no existe, mostrar el resumen ejecutivo.

    funcion_vista = vistas_dashboard.get(
        pestana,
        vista_resumen
    )

    return funcion_vista()


# ==========================================================
# 2. FUNCIÓN DE FILTRADO DE DATOS
# ==========================================================

def filtrar_demanda(anio, usuario, granularidad):

    # Validar los parámetros recibidos.

    assert anio in ["todos", "2011", "2012"]

    assert usuario in [
        "cnt",
        "registered",
        "casual"
    ]

    assert granularidad in [
        "diaria",
        "mensual"
    ]

    # Trabajar sobre una copia para no modificar
    # el dataset original.

    datos_filtrados = df.copy()

    # Aplicar filtro de año.

    if anio != "todos":

        datos_filtrados = datos_filtrados.loc[
            datos_filtrados["dteday"].dt.year
            == int(anio)
        ].copy()

    # Orden cronológico.

    datos_filtrados = (
        datos_filtrados
        .sort_values("dteday")
        .reset_index(drop=True)
    )

    assert not datos_filtrados.empty, (
        "No existen registros para la selección."
    )

    # ======================================================
    # 3. CALCULAR INDICADORES DINÁMICOS
    # ======================================================

    alquileres_filtrados = int(
        datos_filtrados[usuario].sum()
    )

    promedio_filtrado = float(
        datos_filtrados[usuario].mean()
    )

    dias_filtrados = int(
        datos_filtrados["dteday"].nunique()
    )

    participacion_filtrada = (
        alquileres_filtrados
        / total_alquileres
        * 100
    )

    indicadores = {
        "Alquileres": alquileres_filtrados,
        "Promedio_diario": promedio_filtrado,
        "Dias_observados": dias_filtrados,
        "Participacion_global": participacion_filtrada
    }

    # ======================================================
    # 4. PREPARAR SERIE TEMPORAL
    # ======================================================

    etiquetas_usuario = {
        "cnt": "Todos los usuarios",
        "registered": "Usuarios registrados",
        "casual": "Usuarios casuales"
    }

    etiqueta_usuario = etiquetas_usuario[usuario]

    if granularidad == "mensual":

        serie = (
            datos_filtrados
            .assign(
                Fecha=datos_filtrados["dteday"]
                .dt.to_period("M")
                .dt.to_timestamp()
            )
            .groupby("Fecha", as_index=False)
            .agg(
                Alquileres=(usuario, "sum")
            )
        )

        titulo_granularidad = "mensual"

    else:

        serie = (
            datos_filtrados[
                ["dteday", usuario]
            ]
            .rename(columns={
                "dteday": "Fecha",
                usuario: "Alquileres"
            })
            .copy()
        )

        titulo_granularidad = "diaria"

    serie["Año"] = (
        serie["Fecha"].dt.year.astype(str)
    )

    # Validar que la agregación conserve los totales.

    assert int(
        serie["Alquileres"].sum()
    ) == alquileres_filtrados

    # ======================================================
    # 5. GENERAR GRÁFICA DINÁMICA
    # ======================================================

    figura = px.line(
        serie,
        x="Fecha",
        y="Alquileres",
        color="Año",
        markers=(granularidad == "mensual"),

        color_discrete_map={
            "2011": COLORES["principal"],
            "2012": COLORES["secundario"]
        },

        title=(
            f"Demanda {titulo_granularidad} "
            f"— {etiqueta_usuario}"
        ),

        labels={
            "Fecha": "Periodo",
            "Alquileres": "Cantidad de alquileres"
        }
    )

    figura.update_traces(
        line_width=(
            3 if granularidad == "mensual"
            else 1.5
        ),

        hovertemplate=(
            "Fecha: %{x|%Y-%m-%d}"
            "<br>Alquileres: %{y:,.0f}"
            "<extra></extra>"
        )
    )

    figura.update_layout(
        template="plotly_white",

        font=dict(
            family="Arial",
            size=12,
            color=COLORES["texto"]
        ),

        title_font_size=17,

        hovermode="x unified",

        height=450,

        margin=dict(
            l=50,
            r=35,
            t=75,
            b=60
        ),

        legend_title_text="Año",

        paper_bgcolor="white",
        plot_bgcolor="white"
    )

    figura.update_xaxes(
        rangeslider_visible=(
            granularidad == "diaria"
        )
    )

    figura.update_yaxes(
        rangemode="tozero"
    )

    return indicadores, figura, serie


# ==========================================================
# 6. FUNCIÓN PARA TARJETAS KPI FILTRADAS
# ==========================================================

def construir_kpis_filtrados(indicadores):

    return fila_kpis([
        (
            "Alquileres seleccionados",
            formato_entero(
                indicadores["Alquileres"]
            ),
            "Según los filtros aplicados",
            "#2563EB"
        ),
        (
            "Promedio diario",
            formato_decimal(
                indicadores["Promedio_diario"]
            ),
            "Alquileres por día observado",
            "#0D9488"
        ),
        (
            "Días observados",
            formato_entero(
                indicadores["Dias_observados"]
            ),
            "Registros del periodo seleccionado",
            "#7C3AED"
        ),
        (
            "Participación global",
            formato_porcentaje(
                indicadores["Participacion_global"]
            ),
            "Sobre el total histórico de alquileres",
            "#16A34A"
        )
    ])


# ==========================================================
# 7. CALLBACK DE NAVEGACIÓN
# ==========================================================

@app.callback(
    Output(
        "contenido-dashboard",
        "children"
    ),

    Input(
        "tabs-dashboard",
        "active_tab"
    )
)

def actualizar_pestana(pestana):

    return seleccionar_vista(pestana)


# ==========================================================
# 8. CALLBACK DE FILTROS INTERACTIVOS
# ==========================================================

@app.callback(
    Output(
        "grafica-demanda-filtrada",
        "figure"
    ),

    Output(
        "kpis-filtrados",
        "children"
    ),

    Input(
        "filtro-anio",
        "value"
    ),

    Input(
        "filtro-usuario",
        "value"
    ),

    Input(
        "filtro-granularidad",
        "value"
    )
)

def actualizar_demanda(
    anio,
    usuario,
    granularidad
):

    indicadores, figura, serie = filtrar_demanda(
        anio,
        usuario,
        granularidad
    )

    tarjetas = construir_kpis_filtrados(
        indicadores
    )

    return figura, tarjetas


# ==========================================================
# 9. PRUEBAS FUNCIONALES DE LOS FILTROS
# ==========================================================

# Se evalúan todas las combinaciones posibles:
#
# 3 opciones de año
# 3 tipos de usuario
# 2 granularidades
#
# Total: 18 combinaciones.

from itertools import product

opciones_anio = [
    "todos",
    "2011",
    "2012"
]

opciones_usuario = [
    "cnt",
    "registered",
    "casual"
]

opciones_granularidad = [
    "diaria",
    "mensual"
]

resultados_pruebas_filtros = []

for anio, usuario, granularidad in product(
    opciones_anio,
    opciones_usuario,
    opciones_granularidad
):

    indicadores, figura, serie = filtrar_demanda(
        anio,
        usuario,
        granularidad
    )

    # Verificación independiente de los resultados.

    if anio == "todos":

        datos_referencia = df

    else:

        datos_referencia = df.loc[
            df["dteday"].dt.year == int(anio)
        ]

    total_esperado = int(
        datos_referencia[usuario].sum()
    )

    dias_esperados = len(datos_referencia)

    promedio_esperado = (
        datos_referencia[usuario].mean()
    )

    if granularidad == "diaria":

        puntos_esperados = dias_esperados

    else:

        puntos_esperados = (
            datos_referencia["dteday"]
            .dt.to_period("M")
            .nunique()
        )

    assert indicadores[
        "Alquileres"
    ] == total_esperado

    assert indicadores[
        "Dias_observados"
    ] == dias_esperados

    assert np.isclose(
        indicadores["Promedio_diario"],
        promedio_esperado
    )

    assert len(serie) == puntos_esperados

    assert int(
        serie["Alquileres"].sum()
    ) == total_esperado

    assert len(figura.data) > 0

    resultados_pruebas_filtros.append({
        "Año": anio,
        "Usuario": usuario,
        "Granularidad": granularidad,
        "Alquileres": indicadores["Alquileres"],
        "Promedio_diario": indicadores[
            "Promedio_diario"
        ],
        "Dias": indicadores["Dias_observados"],
        "Puntos_grafica": len(serie),
        "Estado": "OK"
    })

df_pruebas_filtros = pd.DataFrame(
    resultados_pruebas_filtros
)

# ==========================================================
# 10. PRUEBAS DE NAVEGACIÓN
# ==========================================================

resultados_pruebas_navegacion = []

for pestana in vistas_dashboard:

    vista = seleccionar_vista(pestana)

    componentes = list(
        recorrer_componentes(vista)
    )

    cantidad_graficas = sum(
        isinstance(componente, dcc.Graph)
        for componente in componentes
    )

    cantidad_esperada = (
        df_inventario_vistas.loc[
            df_inventario_vistas["Vista"]
            == pestana,
            "Graficas"
        ].iloc[0]
    )

    assert cantidad_graficas == cantidad_esperada

    resultados_pruebas_navegacion.append({
        "Pestana": pestana,
        "Graficas_esperadas": cantidad_esperada,
        "Graficas_recuperadas": cantidad_graficas,
        "Estado": "OK"
    })

df_pruebas_navegacion = pd.DataFrame(
    resultados_pruebas_navegacion
)

# ==========================================================
# 11. VERIFICAR REGISTRO DE CALLBACKS
# ==========================================================

callbacks_registrados = list(
    app.callback_map.keys()
)

assert len(callbacks_registrados) == 2

assert len(df_pruebas_filtros) == 18

assert len(df_pruebas_navegacion) == 5

assert (
    df_pruebas_filtros["Estado"] == "OK"
).all()

assert (
    df_pruebas_navegacion["Estado"] == "OK"
).all()

# ==========================================================
# 12. VALIDAR RESPUESTAS DE LOS CALLBACKS
# ==========================================================

# Ejecutar directamente las funciones para comprobar
# que devuelven los componentes esperados.

figura_prueba, tarjetas_prueba = actualizar_demanda(
    "todos",
    "cnt",
    "mensual"
)

assert len(figura_prueba.data) > 0

assert isinstance(
    tarjetas_prueba,
    dbc.Row
)

vista_prueba = actualizar_pestana("fase3")

assert isinstance(
    vista_prueba,
    html.Div
)

# ==========================================================
# 13. EXPORTAR EVIDENCIAS DE VALIDACIÓN
# ==========================================================

CARPETA_VALIDACIONES = (
    RESULTADOS / "Validaciones_Dashboard"
)

CARPETA_VALIDACIONES.mkdir(
    parents=True,
    exist_ok=True
)

df_pruebas_filtros.to_csv(
    CARPETA_VALIDACIONES /
    "pruebas_filtros.csv",

    index=False,
    encoding="utf-8-sig"
)

df_pruebas_navegacion.to_csv(
    CARPETA_VALIDACIONES /
    "pruebas_navegacion.csv",

    index=False,
    encoding="utf-8-sig"
)

# ==========================================================
# 14. PRESENTACIÓN DE RESULTADOS
# ==========================================================

print("=" * 60)
print("DASHBOARD GLOBAL - CALLBACKS E INTERACTIVIDAD")
print("=" * 60)

print("\n=== CALLBACKS REGISTRADOS ===")

for numero, callback in enumerate(
    callbacks_registrados,
    start=1
):

    print(f"{numero}. {callback}")

print("\n=== PRUEBAS DE FILTROS ===")

print(
    df_pruebas_filtros.round(2)
    .to_string(index=False)
)

print("\n=== PRUEBAS DE NAVEGACIÓN ===")

print(
    df_pruebas_navegacion.to_string(
        index=False
    )
)

print("\n=== VALIDACIONES ===")

print("Registro de callbacks: OK")
print("Filtrado por año: OK")
print("Filtrado por tipo de usuario: OK")
print("Granularidad diaria: OK")
print("Granularidad mensual: OK")
print("Indicadores dinámicos: OK")
print("Conservación de totales: OK")
print("Navegación entre pestañas: OK")
print("Exportación de evidencias: OK")

print("\n=== RESUMEN ===")

print(
    "Callbacks registrados:",
    len(callbacks_registrados)
)

print(
    "Pruebas de filtros superadas:",
    len(df_pruebas_filtros),
    "/ 18"
)

print(
    "Pruebas de navegación superadas:",
    len(df_pruebas_navegacion),
    "/ 5"
)

print(
    "Total de pruebas funcionales:",
    len(df_pruebas_filtros)
    + len(df_pruebas_navegacion)
)

print("\nServidor Dash: pendiente de ejecución")

print(
    "\nCallbacks e interactividad "
    "implementados correctamente."
)

In [ ]:
# ==========================================================
# DASHBOARD GLOBAL - CELDA 17
# Ejecución del dashboard en Google Colab
# ==========================================================

import dash

# ==========================================================
# 1. VERIFICACIONES PREVIAS
# ==========================================================

assert app.layout is not None, (
    "El layout de la aplicación no está disponible."
)

assert len(app.callback_map) == 2, (
    "Los callbacks no están registrados correctamente."
)

assert len(vistas_dashboard) == 5

assert len(df_pruebas_filtros) == 18

assert len(df_pruebas_navegacion) == 5

# ==========================================================
# 2. VALIDACIÓN DE ENDPOINTS DEL SERVIDOR
# ==========================================================

# Comprobar que Dash puede serializar el layout
# y entregar las dependencias de los callbacks.

with app.server.test_client() as cliente:

    respuesta_inicio = cliente.get("/")

    respuesta_layout = cliente.get(
        "/_dash-layout"
    )

    respuesta_dependencias = cliente.get(
        "/_dash-dependencies"
    )

    assert respuesta_inicio.status_code == 200

    assert respuesta_layout.status_code == 200

    assert respuesta_dependencias.status_code == 200

# ==========================================================
# 3. INFORMACIÓN DE LA APLICACIÓN
# ==========================================================

print("=" * 60)
print("BIKE SHARING ANALYTICS")
print("DASHBOARD GLOBAL - PROYECTO INTEGRADOR")
print("=" * 60)

print("\nVersión de Dash:", dash.__version__)

print("\n=== VALIDACIONES PREVIAS ===")

print("Layout principal: OK")
print("Callbacks registrados: OK")
print("Vistas disponibles: OK")
print("Pruebas funcionales: OK")

print("\n=== ENDPOINTS DEL SERVIDOR ===")

print(
    "Página principal:",
    respuesta_inicio.status_code
)

print(
    "Layout Dash:",
    respuesta_layout.status_code
)

print(
    "Dependencias de callbacks:",
    respuesta_dependencias.status_code
)

print("\n=== COMPONENTES DISPONIBLES ===")

print("Pestañas: 5")
print("Gráficas interactivas: 46")
print("Tablas interactivas: 34")
print("Imágenes originales: 25")
print("Pruebas funcionales superadas: 23")

print("\nIniciando servidor Dash...")
print("Modo de visualización: inline")
print("Puerto: 8050")

# ==========================================================
# 4. EJECUTAR DASHBOARD
# ==========================================================

print("Servidor validado. Inicio aplazado.")


In [ ]:
# ==========================================================
# DASHBOARD GLOBAL - CELDA 18
# Complementos académicos de la Fase 1
# ==========================================================

import pandas as pd
import numpy as np
import zipfile
from scipy.stats import zscore

# ==========================================================
# 1. FORMULACIÓN DEL PROBLEMA
# ==========================================================

contexto_fase1 = """
Los sistemas de bicicletas compartidas requieren una
planificación eficiente de sus recursos para responder
a las variaciones de la demanda.

La demanda de alquileres puede estar asociada con factores
meteorológicos, estacionales y temporales. Comprender estas
relaciones permite apoyar la planificación operativa,
la disponibilidad de bicicletas y la toma de decisiones
basada en datos.

El proyecto analiza 731 observaciones diarias del sistema
Capital Bikeshare durante los años 2011 y 2012.
"""

preguntas_fase1 = pd.DataFrame([
    {
        "Tipo": "Principal",
        "Pregunta": (
            "¿Cómo se relacionan las condiciones "
            "meteorológicas y temporales con la demanda "
            "diaria de bicicletas compartidas?"
        )
    },
    {
        "Tipo": "Secundaria 1",
        "Pregunta": (
            "¿Qué patrones temporales y estacionales "
            "presenta la demanda durante 2011 y 2012?"
        )
    },
    {
        "Tipo": "Secundaria 2",
        "Pregunta": (
            "¿Qué variables meteorológicas presentan "
            "mayor asociación estadística con la demanda?"
        )
    },
    {
        "Tipo": "Secundaria 3",
        "Pregunta": (
            "¿Qué problemas de calidad y valores atípicos "
            "deben considerarse antes del modelado?"
        )
    }
])

# ==========================================================
# 2. OBJETIVOS DE INVESTIGACIÓN
# ==========================================================

objetivo_general_fase1 = """
Caracterizar la demanda diaria de bicicletas compartidas
durante 2011 y 2012 mediante técnicas de análisis
exploratorio, estadística descriptiva y evaluación de
calidad de datos, con el propósito de establecer una base
analítica para las fases posteriores de modelado predictivo.
"""

df_objetivos_fase1 = pd.DataFrame([
    {
        "Objetivo": "Específico 1",
        "Descripcion": (
            "Caracterizar las 731 observaciones y las "
            "16 variables originales mediante estadísticas "
            "descriptivas, frecuencias y visualizaciones "
            "durante la Fase 1."
        ),
        "Indicador": "16 variables caracterizadas",
        "Plazo": "Finalización de la Fase 1"
    },
    {
        "Objetivo": "Específico 2",
        "Descripcion": (
            "Cuantificar la asociación entre las variables "
            "meteorológicas y la demanda mediante Pearson, "
            "Spearman y pruebas exploratorias Chi-cuadrado."
        ),
        "Indicador": (
            "2 matrices de correlación y 4 pruebas "
            "Chi-cuadrado"
        ),
        "Plazo": "Finalización de la Fase 1"
    },
    {
        "Objetivo": "Específico 3",
        "Descripcion": (
            "Evaluar la calidad de las 731 observaciones "
            "mediante la identificación de valores "
            "faltantes, duplicados, inconsistencias y "
            "outliers utilizando IQR y Z-score."
        ),
        "Indicador": (
            "100 % de los registros evaluados y "
            "2 métodos de detección de outliers"
        ),
        "Plazo": "Antes del modelado predictivo"
    }
])

# ==========================================================
# 3. RECUPERAR DOCUMENTACIÓN ORIGINAL DEL DATASET
# ==========================================================

with zipfile.ZipFile(RUTA_ZIP, "r") as archivo_zip:

    archivos_readme = [
        nombre
        for nombre in archivo_zip.namelist()
        if "readme" in nombre.lower()
    ]

    assert archivos_readme, (
        "No se encontró la documentación original."
    )

    readme_dataset = archivo_zip.read(
        archivos_readme[0]
    ).decode("utf-8", errors="replace")

print("=== DOCUMENTACIÓN ORIGINAL DEL DATASET ===")
print(readme_dataset)

# ==========================================================
# 4. FICHA TÉCNICA
# ==========================================================

df_ficha_tecnica = pd.DataFrame([
    ("Nombre", "Bike Sharing Dataset"),
    ("Repositorio", "UCI Machine Learning Repository"),
    ("Autor", "Hadi Fanaee-T"),
    ("DOI", "10.24432/C5W894"),
    (
        "URL del repositorio",
        "https://archive.ics.uci.edu/dataset/275/"
        "bike+sharing+dataset"
    ),
    ("URL de descarga", URL_DATASET),
    ("Archivo analizado", "day.csv"),
    ("Formato", "CSV"),
    ("Registros", str(len(df))),
    ("Variables", str(df.shape[1])),
    ("Granularidad", "Diaria"),
    (
        "Periodo inicial",
        str(df["dteday"].min().date())
    ),
    (
        "Periodo final",
        str(df["dteday"].max().date())
    ),
    ("Variable objetivo", "cnt"),
    (
        "Fecha de extracción",
        "Pendiente de confirmar fecha original"
    ),
    (
        "Licencia",
        "Pendiente de verificación en el repositorio UCI"
    ),
    ("Valores faltantes", str(df.isna().sum().sum())),
    ("Duplicados", str(df.duplicated().sum())),
    (
        "Procedencia del análisis",
        "EDA reconstruido desde el dataset original"
    )
], columns=["Campo", "Valor"])

# ==========================================================
# 5. DICCIONARIO DE DATOS
# ==========================================================

def rango_observado(variable):

    return (
        f"{df[variable].min()} — "
        f"{df[variable].max()}"
    )


df_diccionario_fase1 = pd.DataFrame([
    (
        "instant",
        "Identificador",
        "Índice único del registro",
        rango_observado("instant")
    ),
    (
        "dteday",
        "Temporal",
        "Fecha de observación",
        "2011-01-01 a 2012-12-31"
    ),
    (
        "season",
        "Cualitativa nominal",
        "Estación del año codificada",
        "1, 2, 3, 4"
    ),
    (
        "yr",
        "Cualitativa ordinal",
        "Año de observación",
        "0: 2011; 1: 2012"
    ),
    (
        "mnth",
        "Cualitativa ordinal cíclica",
        "Mes del año",
        "1 a 12"
    ),
    (
        "holiday",
        "Cualitativa nominal binaria",
        "Indicador de día festivo",
        "0: No; 1: Sí"
    ),
    (
        "weekday",
        "Cualitativa nominal cíclica",
        "Día de la semana",
        "0 a 6"
    ),
    (
        "workingday",
        "Cualitativa nominal binaria",
        "Día laboral, excluye festivos y fines de semana",
        "0: No; 1: Sí"
    ),
    (
        "weathersit",
        "Cualitativa ordinal",
        "Categoría de condiciones meteorológicas",
        "1 a 4; observadas: 1, 2 y 3"
    ),
    (
        "temp",
        "Cuantitativa continua",
        "Temperatura normalizada",
        rango_observado("temp")
    ),
    (
        "atemp",
        "Cuantitativa continua",
        "Sensación térmica normalizada",
        rango_observado("atemp")
    ),
    (
        "hum",
        "Cuantitativa continua",
        "Humedad relativa normalizada",
        rango_observado("hum")
    ),
    (
        "windspeed",
        "Cuantitativa continua",
        "Velocidad del viento normalizada",
        rango_observado("windspeed")
    ),
    (
        "casual",
        "Cuantitativa discreta",
        "Alquileres de usuarios casuales",
        rango_observado("casual")
    ),
    (
        "registered",
        "Cuantitativa discreta",
        "Alquileres de usuarios registrados",
        rango_observado("registered")
    ),
    (
        "cnt",
        "Cuantitativa discreta",
        "Total diario de alquileres; variable objetivo",
        rango_observado("cnt")
    )
], columns=[
    "Variable",
    "Tipo",
    "Descripcion",
    "Rango_o_valores"
])

# ==========================================================
# 6. DETECCIÓN DE OUTLIERS MEDIANTE Z-SCORE
# ==========================================================

# Se utiliza un umbral convencional de |Z| > 3.
# El análisis es descriptivo y no modifica el dataset.

UMBRAL_Z = 3

resultados_zscore = []
detalle_zscore = []

for variable in variables_cuantitativas:

    valores = df[variable].astype(float)

    if valores.std(ddof=0) == 0:

        puntuaciones = np.zeros(len(valores))

    else:

        puntuaciones = zscore(
            valores,
            ddof=0,
            nan_policy="omit"
        )

    mascara = np.abs(puntuaciones) > UMBRAL_Z

    cantidad = int(mascara.sum())

    resultados_zscore.append({
        "Variable": variable,
        "Outliers_Zscore": cantidad,
        "Porcentaje_Zscore": (
            cantidad / len(df) * 100
        ),
        "Umbral": UMBRAL_Z
    })

    for posicion in np.flatnonzero(mascara):

        detalle_zscore.append({
            "Variable": variable,
            "Fecha": df["dteday"].iloc[posicion],
            "Valor": valores.iloc[posicion],
            "Zscore": float(puntuaciones[posicion])
        })

df_zscore_fase1 = pd.DataFrame(resultados_zscore)

df_detalle_zscore = pd.DataFrame(
    detalle_zscore,
    columns=["Variable", "Fecha", "Valor", "Zscore"]
)

# ==========================================================
# 7. COMPARACIÓN IQR VS Z-SCORE
# ==========================================================

df_comparacion_outliers = (
    df_outliers_eda[
        ["Variable", "Outliers", "Porcentaje"]
    ]
    .rename(columns={
        "Outliers": "Outliers_IQR",
        "Porcentaje": "Porcentaje_IQR"
    })
    .merge(
        df_zscore_fase1,
        on="Variable",
        how="inner",
        validate="one_to_one"
    )
)

df_outliers_largo = df_comparacion_outliers.melt(
    id_vars="Variable",
    value_vars=[
        "Outliers_IQR",
        "Outliers_Zscore"
    ],
    var_name="Metodo",
    value_name="Cantidad"
)

df_outliers_largo["Metodo"] = (
    df_outliers_largo["Metodo"].map({
        "Outliers_IQR": "IQR",
        "Outliers_Zscore": "Z-score"
    })
)

fig_comparacion_outliers = px.bar(
    df_outliers_largo,
    x="Variable",
    y="Cantidad",
    color="Metodo",
    barmode="group",
    text="Cantidad",
    title="Comparación de outliers: IQR vs Z-score",
    color_discrete_map={
        "IQR": "#2563EB",
        "Z-score": "#0D9488"
    },
    template="plotly_white"
)

fig_comparacion_outliers.update_layout(
    height=450,
    xaxis_title="Variable",
    yaxis_title="Cantidad de valores atípicos",
    legend_title_text="Método"
)

fig_comparacion_outliers.update_traces(
    textposition="outside"
)

# ==========================================================
# 8. EXPORTAR COMPLEMENTOS ACADÉMICOS
# ==========================================================

complementos_fase1 = {
    "preguntas_investigacion": preguntas_fase1,
    "objetivos_smart": df_objetivos_fase1,
    "ficha_tecnica": df_ficha_tecnica,
    "diccionario_datos": df_diccionario_fase1,
    "outliers_zscore": df_zscore_fase1,
    "detalle_zscore": df_detalle_zscore,
    "comparacion_outliers": df_comparacion_outliers
}

for nombre, tabla in complementos_fase1.items():

    tabla.to_csv(
        FASE1 / f"{nombre}.csv",
        index=False,
        encoding="utf-8-sig"
    )

(FASE1 / "formulacion_problema.txt").write_text(
    contexto_fase1,
    encoding="utf-8"
)

(FASE1 / "objetivo_general.txt").write_text(
    objetivo_general_fase1,
    encoding="utf-8"
)

(FASE1 / "documentacion_original.txt").write_text(
    readme_dataset,
    encoding="utf-8"
)

fig_comparacion_outliers.write_html(
    CARPETA_GRAFICAS_F1 / "comparacion_outliers.html"
)

fig_comparacion_outliers.write_json(
    CARPETA_GRAFICAS_F1 / "comparacion_outliers.json"
)

# ==========================================================
# 9. VALIDACIONES
# ==========================================================

assert len(df_diccionario_fase1) == 16

assert set(df_diccionario_fase1["Variable"]) == set(df.columns)

assert len(preguntas_fase1) == 4

assert len(df_objetivos_fase1) == 3

assert len(df_zscore_fase1) == len(variables_cuantitativas)

assert len(df_comparacion_outliers) == len(variables_cuantitativas)

assert len(df) == 731

assert int(df["cnt"].sum()) == total_alquileres

assert all(
    (FASE1 / f"{nombre}.csv").exists()
    for nombre in complementos_fase1
)

# ==========================================================
# 10. PRESENTACIÓN DE RESULTADOS
# ==========================================================

print("=" * 60)
print("FASE 1 - COMPLEMENTOS ACADÉMICOS")
print("=" * 60)

print("\n=== FICHA TÉCNICA ===")
print(df_ficha_tecnica.to_string(index=False))

print("\n=== DICCIONARIO DE DATOS ===")
print(df_diccionario_fase1.to_string(index=False))

print("\n=== COMPARACIÓN IQR VS Z-SCORE ===")
print(
    df_comparacion_outliers.round(4)
    .to_string(index=False)
)

print("\n=== DETALLE DE OUTLIERS Z-SCORE ===")
print(df_detalle_zscore.to_string(index=False))

print("\n=== VALIDACIONES ===")
print("Preguntas de investigación: 4 - OK")
print("Objetivo general: 1 - OK")
print("Objetivos específicos SMART: 3 - OK")
print("Diccionario de variables: 16/16 - OK")
print("Detección mediante IQR: OK")
print("Detección mediante Z-score: OK")
print("Integridad del dataset: OK")
print("Archivos CSV generados: 7")
print("Modelos reentrenados: 0")

print("\nPendiente: verificar licencia y fecha de extracción.")
print("Pendiente: integrar complementos en la interfaz Dash.")

fig_comparacion_outliers.show()

In [ ]:
# ==========================================================
# DASHBOARD GLOBAL - CELDA 19A
# Verificación de metadatos y categorías estacionales
# ==========================================================

from datetime import datetime, timezone
from pathlib import Path

# ==========================================================
# 1. VERIFICAR CODIFICACIÓN DE ESTACIONES
# ==========================================================

df_verificacion_estaciones = (
    df.assign(Mes=df["dteday"].dt.month)
    .groupby(["season", "Mes"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=range(1, 13), fill_value=0)
)

df_periodos_estaciones = (
    df.groupby(["season", "yr"])
    .agg(
        Fecha_inicial=("dteday", "min"),
        Fecha_final=("dteday", "max"),
        Registros=("dteday", "size")
    )
    .reset_index()
)

print("=" * 60)
print("VERIFICACIÓN DE METADATOS")
print("=" * 60)

print("\n=== DISTRIBUCIÓN DE ESTACIONES POR MES ===")
print(df_verificacion_estaciones.to_string())

print("\n=== PERIODOS POR ESTACIÓN Y AÑO ===")
print(df_periodos_estaciones.to_string(index=False))

# ==========================================================
# 2. VERIFICAR CODIFICACIÓN DE DÍAS DE LA SEMANA
# ==========================================================

dias_semana = {
    0: "Lunes",
    1: "Martes",
    2: "Miércoles",
    3: "Jueves",
    4: "Viernes",
    5: "Sábado",
    6: "Domingo"
}

df_verificacion_dias = (
    df.assign(
        Dia_real=df["dteday"].dt.dayofweek.map(dias_semana)
    )[["weekday", "Dia_real"]]
    .drop_duplicates()
    .sort_values("weekday")
    .reset_index(drop=True)
)

print("\n=== CODIFICACIÓN DE DÍAS ===")
print(df_verificacion_dias.to_string(index=False))

# ==========================================================
# 3. VERIFICAR FECHAS DE ARCHIVOS
# ==========================================================

print("\n=== FECHAS DE LOS ARCHIVOS ===")

for ruta in [RUTA_ZIP, RUTA_CSV]:

    ruta = Path(ruta)

    fecha_modificacion = datetime.fromtimestamp(
        ruta.stat().st_mtime,
        tz=timezone.utc
    )

    print("Archivo:", ruta.name)
    print("Última modificación UTC:",
          fecha_modificacion.isoformat())

# ==========================================================
# 4. VERIFICAR LICENCIA EN REPOSITORIO UCI
# ==========================================================

try:

    import requests

    respuesta = requests.get(
        "https://archive.ics.uci.edu/api/dataset?id=275",
        timeout=30
    )

    respuesta.raise_for_status()

    metadata_uci = respuesta.json().get("data", {})

    print("\n=== METADATOS OFICIALES UCI ===")

    for campo, valor in metadata_uci.items():

        if any(
            palabra in campo.lower()
            for palabra in [
                "license",
                "licence",
                "doi",
                "date",
                "year",
                "name"
            ]
        ):

            print(f"{campo}: {valor}")

except Exception as error:

    print("\nNo fue posible consultar la API de UCI.")
    print("Detalle:", str(error))

# ==========================================================
# 5. REFERENCIAS BIBLIOGRÁFICAS
# ==========================================================

referencia_dataset = (
    "Fanaee-T, H. (2013). Bike Sharing Dataset "
    "[Dataset]. UCI Machine Learning Repository. "
    "https://doi.org/10.24432/C5W894"
)

referencia_articulo = (
    "Fanaee-T, H., & Gama, J. (2013). "
    "Event labeling combining ensemble detectors "
    "and background knowledge. "
    "Progress in Artificial Intelligence. "
    "https://doi.org/10.1007/s13748-013-0040-3"
)

print("\n=== REFERENCIAS BIBLIOGRÁFICAS ===")
print(referencia_dataset)
print(referencia_articulo)

# ==========================================================
# 6. VALIDACIONES
# ==========================================================

assert df_verificacion_estaciones.to_numpy().sum() == 731
assert len(df_verificacion_dias) == 7
assert len(df) == 731
assert int(df["cnt"].sum()) == total_alquileres

print("\n=== VALIDACIONES ===")
print("Distribución estacional: OK")
print("Codificación semanal: OK")
print("Integridad del dataset: OK")
print("Modelos reentrenados: 0")
print("\nVerificación de metadatos finalizada.")

In [ ]:
# ==========================================================
# DASHBOARD GLOBAL - CELDA 19B
# Integración académica final de la Fase 1
# ==========================================================

from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from dash import dash_table

# ==========================================================
# 1. ACTUALIZAR FICHA TÉCNICA
# ==========================================================

fecha_archivo = datetime.fromtimestamp(
    RUTA_ZIP.stat().st_mtime,
    tz=timezone.utc
).astimezone(ZoneInfo("America/Bogota"))

df_ficha_tecnica.loc[
    df_ficha_tecnica["Campo"] == "Fecha de extracción",
    "Valor"
] = (
    fecha_archivo.strftime("%Y-%m-%d %H:%M:%S")
    + " Colombia (fecha del ZIP auxiliar reconstruido)"
)

df_ficha_tecnica.loc[
    df_ficha_tecnica["Campo"] == "Licencia",
    "Valor"
] = (
    "Pendiente de confirmación oficial. "
    "El README exige citar la publicación original."
)

campos_adicionales = pd.DataFrame([
    (
        "Año de creación",
        "2013"
    ),
    (
        "Última actualización reportada por UCI",
        "2024-03-10"
    ),
    (
        "Sistema de bicicletas",
        "Capital Bikeshare"
    ),
    (
        "Ubicación",
        "Washington D.C., Estados Unidos"
    ),
    (
        "Referencia del dataset",
        referencia_dataset
    ),
    (
        "Referencia científica",
        referencia_articulo
    )
], columns=["Campo", "Valor"])

# Evitar duplicados si la celda se ejecuta nuevamente.

df_ficha_tecnica = pd.concat([
    df_ficha_tecnica,
    campos_adicionales
], ignore_index=True).drop_duplicates(
    subset="Campo",
    keep="last"
)

# ==========================================================
# 2. ACTUALIZAR DICCIONARIO DE VARIABLES
# ==========================================================

df_diccionario_fase1.loc[
    df_diccionario_fase1["Variable"] == "season",
    "Rango_o_valores"
] = (
    "1: Invierno; 2: Primavera; "
    "3: Verano; 4: Otoño"
)

df_diccionario_fase1.loc[
    df_diccionario_fase1["Variable"] == "weekday",
    "Rango_o_valores"
] = (
    "0: Domingo; 1: Lunes; 2: Martes; "
    "3: Miércoles; 4: Jueves; "
    "5: Viernes; 6: Sábado"
)

df_diccionario_fase1.loc[
    df_diccionario_fase1["Variable"] == "weathersit",
    "Rango_o_valores"
] = (
    "1: Despejado o parcialmente nublado; "
    "2: Niebla o nublado; "
    "3: Precipitación ligera o tormenta; "
    "4: Condiciones severas"
)

nota_estaciones = (
    "El README original presenta una inconsistencia "
    "en la descripción de las estaciones. La codificación "
    "utilizada en este dashboard fue contrastada con las "
    "fechas de las 731 observaciones, confirmando que "
    "1 corresponde a invierno, 2 a primavera, "
    "3 a verano y 4 a otoño."
)

nota_outliers = (
    "El método IQR identificó 59 valores atípicos "
    "y Z-score identificó 12. Estas cantidades "
    "corresponden a detecciones por variable y no "
    "necesariamente a registros únicos. "
    "No se eliminaron observaciones durante el EDA."
)

# ==========================================================
# 3. ACTUALIZAR ARCHIVOS DE EVIDENCIA
# ==========================================================

complementos_fase1["ficha_tecnica"] = (
    df_ficha_tecnica
)

complementos_fase1["diccionario_datos"] = (
    df_diccionario_fase1
)

for nombre, tabla in complementos_fase1.items():

    tabla.to_csv(
        FASE1 / f"{nombre}.csv",
        index=False,
        encoding="utf-8-sig"
    )

df_verificacion_estaciones.to_csv(
    FASE1 / "verificacion_estaciones.csv",
    encoding="utf-8-sig"
)

df_verificacion_dias.to_csv(
    FASE1 / "verificacion_dias_semana.csv",
    index=False,
    encoding="utf-8-sig"
)

(FASE1 / "nota_metodologica_estaciones.txt").write_text(
    nota_estaciones,
    encoding="utf-8"
)

(FASE1 / "referencias_bibliograficas.txt").write_text(
    referencia_dataset + "\n\n" + referencia_articulo,
    encoding="utf-8"
)

# ==========================================================
# 4. FUNCIÓN DE TABLAS ACADÉMICAS
# ==========================================================

def tabla_complementaria_f1(tabla, identificador):

    datos = tabla.copy()

    # Convertir fechas a texto para serialización.

    for columna in datos.columns:

        if pd.api.types.is_datetime64_any_dtype(
            datos[columna]
        ):

            datos[columna] = (
                datos[columna].astype(str)
            )

    return dash_table.DataTable(

        id=f"academico-f1-{identificador}",

        columns=[
            {
                "name": columna.replace("_", " "),
                "id": columna
            }
            for columna in datos.columns
        ],

        data=datos.round(4).to_dict("records"),

        page_size=10,
        sort_action="native",
        filter_action="native",

        style_table={
            "overflowX": "auto"
        },

        style_header={
            "backgroundColor": "#142C49",
            "color": "white",
            "fontWeight": "bold",
            "whiteSpace": "normal"
        },

        style_cell={
            "fontFamily": "Arial",
            "fontSize": "13px",
            "padding": "12px",
            "textAlign": "left",
            "whiteSpace": "normal",
            "height": "auto",
            "minWidth": "100px",
            "maxWidth": "550px"
        },

        style_data_conditional=[
            {
                "if": {"row_index": "odd"},
                "backgroundColor": "#F8FAFC"
            }
        ]
    )


# ==========================================================
# 5. CONSTRUIR SECCIONES ACADÉMICAS
# ==========================================================

def panel_academico_fase1():

    return html.Div([

        html.H3(
            "Complementos académicos",
            className="mt-4 mb-3"
        ),

        dbc.Alert(
            "Esta sección complementa el análisis "
            "exploratorio reconstruido desde el dataset "
            "original. No se dispone del notebook "
            "original de la Fase 1.",
            color="info"
        ),

        dbc.Accordion([

            # ----------------------------------------------
            # Formulación del problema
            # ----------------------------------------------

            dbc.AccordionItem([

                html.P(
                    contexto_fase1.strip(),
                    style={
                        "whiteSpace": "pre-line"
                    }
                ),

                html.H5(
                    "Preguntas de investigación"
                ),

                tabla_complementaria_f1(
                    preguntas_fase1,
                    "preguntas"
                )

            ], title="1. Formulación del problema"),

            # ----------------------------------------------
            # Objetivos SMART
            # ----------------------------------------------

            dbc.AccordionItem([

                html.H5("Objetivo general"),

                html.P(
                    objetivo_general_fase1.strip()
                ),

                html.H5("Objetivos específicos SMART"),

                tabla_complementaria_f1(
                    df_objetivos_fase1,
                    "objetivos"
                )

            ], title="2. Objetivos de investigación"),

            # ----------------------------------------------
            # Ficha técnica
            # ----------------------------------------------

            dbc.AccordionItem([

                tabla_complementaria_f1(
                    df_ficha_tecnica,
                    "ficha"
                )

            ], title="3. Ficha técnica del dataset"),

            # ----------------------------------------------
            # Diccionario de variables
            # ----------------------------------------------

            dbc.AccordionItem([

                tabla_complementaria_f1(
                    df_diccionario_fase1,
                    "diccionario"
                ),

                dbc.Alert(
                    nota_estaciones,
                    color="warning",
                    className="mt-3"
                )

            ], title="4. Diccionario de datos"),

            # ----------------------------------------------
            # Comparación de valores atípicos
            # ----------------------------------------------

            dbc.AccordionItem([

                html.P(nota_outliers),

                tarjeta_grafica(
                    fig_comparacion_outliers,
                    "fase1-comparacion-iqr-zscore"
                ),

                html.H5("Comparación de métodos"),

                tabla_complementaria_f1(
                    df_comparacion_outliers,
                    "comparacion-outliers"
                ),

                html.H5(
                    "Resumen Z-score",
                    className="mt-4"
                ),

                tabla_complementaria_f1(
                    df_zscore_fase1,
                    "resumen-zscore"
                ),

                html.H5(
                    "Detalle de observaciones detectadas",
                    className="mt-4"
                ),

                tabla_complementaria_f1(
                    df_detalle_zscore,
                    "detalle-zscore"
                )

            ], title="5. Análisis complementario de outliers"),

            # ----------------------------------------------
            # Referencias
            # ----------------------------------------------

            dbc.AccordionItem([

                html.P(referencia_dataset),

                html.P(referencia_articulo),

                html.A(
                    "Consultar repositorio original UCI",
                    href=(
                        "https://archive.ics.uci.edu/"
                        "dataset/275/bike+sharing+dataset"
                    ),
                    target="_blank",
                    rel="noopener noreferrer"
                )

            ], title="6. Referencias bibliográficas")

        ],
            start_collapsed=True,
            always_open=True
        )

    ], className="mb-4")


# ==========================================================
# 6. EXTENDER LA VISTA ORIGINAL DE LA FASE 1
# ==========================================================

# Conservar la función original.
# Esta condición evita extensiones duplicadas.

if "vista_fase1_original" not in globals():

    vista_fase1_original = vistas_dashboard["fase1"]


def vista_fase1_integrada():

    return html.Div([

        vista_fase1_original(),

        panel_academico_fase1()

    ])


vistas_dashboard["fase1"] = vista_fase1_integrada
# ==========================================================
# 7. EVITAR DUPLICACIÓN DE FIGURAS
# ==========================================================

# La gráfica de comparación IQR vs. Z-score
# ya está incluida en el panel académico de Fase 1.
# Se elimina del diccionario de gráficas generales
# para evitar duplicaciones.

figuras_fase1.pop("comparacion_outliers", None)

assert len(figuras_fase1) == 11

print("Gráficas originales de Fase 1: 11")
print("Gráfica complementaria IQR vs. Z-score: 1")
print("Total de gráficas de Fase 1: 12")
# ==========================================================
# 8. VALIDAR TODAS LAS VISTAS ACTUALIZADAS
# ==========================================================

inventario_final = []

for nombre, funcion_vista in vistas_dashboard.items():

    vista = funcion_vista()

    componentes = list(
        recorrer_componentes(vista)
    )

    graficas = sum(
        isinstance(c, dcc.Graph)
        for c in componentes
    )

    tablas = sum(
        isinstance(c, dash_table.DataTable)
        for c in componentes
    )

    imagenes = sum(
        isinstance(c, html.Img)
        for c in componentes
    )

    ids = [
        str(c.id)
        for c in componentes
        if getattr(c, "id", None) is not None
    ]

    assert len(ids) == len(set(ids)), (
        f"Identificadores duplicados en {nombre}"
    )

    inventario_final.append({
        "Vista": nombre,
        "Componentes": len(componentes),
        "Graficas": graficas,
        "Tablas": tablas,
        "Imagenes": imagenes,
        "Estado": "OK"
    })

df_inventario_final = pd.DataFrame(
    inventario_final
)

# ==========================================================
# 9. VALIDACIONES FINALES
# ==========================================================

assert len(df_inventario_final) == 5

assert df_inventario_final["Graficas"].sum() == 47

assert df_inventario_final["Tablas"].sum() == 41

assert df_inventario_final["Imagenes"].sum() == 25

assert len(df_diccionario_fase1) == 16

assert len(app.callback_map) == 2

assert len(df) == 731

assert int(df["cnt"].sum()) == total_alquileres

df_inventario_final.to_csv(
    RESULTADOS / "inventario_final_dashboard.csv",
    index=False,
    encoding="utf-8-sig"
)

# ==========================================================
# 10. PRESENTACIÓN DE RESULTADOS
# ==========================================================

print("=" * 60)
print("DASHBOARD GLOBAL - INTEGRACIÓN ACADÉMICA FINAL")
print("=" * 60)

print("\n=== INVENTARIO FINAL ===")
print(df_inventario_final.to_string(index=False))

print("\n=== COMPONENTES INTEGRADOS ===")

print("Pestañas:", len(df_inventario_final))

print(
    "Gráficas interactivas:",
    df_inventario_final["Graficas"].sum()
)

print(
    "Tablas interactivas:",
    df_inventario_final["Tablas"].sum()
)

print(
    "Imágenes originales:",
    df_inventario_final["Imagenes"].sum()
)

print(
    "Componentes de interfaz:",
    df_inventario_final["Componentes"].sum()
)

print("\n=== VALIDACIONES ===")
print("Formulación del problema: OK")
print("Objetivos SMART: OK")
print("Ficha técnica actualizada: OK")
print("Diccionario de variables: OK")
print("Codificación estacional verificada: OK")
print("Comparación IQR y Z-score: OK")
print("Referencias bibliográficas: OK")
print("Identificadores únicos: OK")
print("Integridad del dataset: OK")
print("Modelos reentrenados: 0")

print("\nIntegración académica finalizada.")

In [ ]:
# ==========================================================
# DASHBOARD GLOBAL - CELDA 20
# Actualización, validación y exportación final
# ==========================================================

import shutil
import json
import hashlib
import platform
from importlib.metadata import version
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

# ==========================================================
# 1. CONFIRMAR LICENCIA DEL DATASET
# ==========================================================

df_ficha_tecnica.loc[
    df_ficha_tecnica["Campo"] == "Licencia",
    "Valor"
] = "Creative Commons Attribution 4.0 International (CC BY 4.0)"

df_ficha_tecnica = pd.concat([
    df_ficha_tecnica,
    pd.DataFrame([
        {
            "Campo": "Fuente oficial de licencia",
            "Valor": (
                "https://archive.ics.uci.edu/"
                "dataset/275/bike-sharing-dataset"
            )
        },
        {
            "Campo": "URL de licencia",
            "Valor": (
                "https://creativecommons.org/licenses/by/4.0/"
            )
        }
    ])
], ignore_index=True).drop_duplicates(
    subset="Campo",
    keep="last"
)

complementos_fase1["ficha_tecnica"] = df_ficha_tecnica

df_ficha_tecnica.to_csv(
    FASE1 / "ficha_tecnica.csv",
    index=False,
    encoding="utf-8-sig"
)

# ==========================================================
# 2. ACTUALIZAR INVENTARIO EJECUTIVO
# ==========================================================

inventario_fases.loc[
    inventario_fases["Fase"] == "Fase 1",
    "Graficas_interactivas"
] = 12

inventario_fases.loc[
    inventario_fases["Fase"] == "Fase 1",
    "Estado"
] = "Consolidado y complementado"

inventario_fases.to_csv(
    RESULTADOS / "inventario_fases.csv",
    index=False,
    encoding="utf-8-sig"
)

figuras_resumen["evidencias_por_fase"] = px.bar(
    inventario_fases,
    x="Fase",
    y=[
        "Graficas_interactivas",
        "Imagenes_originales"
    ],
    barmode="group",
    title="Evidencias integradas por fase",
    labels={
        "value": "Cantidad",
        "variable": "Tipo de evidencia"
    },
    color_discrete_sequence=[
        "#2563EB",
        "#0D9488"
    ],
    template="plotly_white"
)

figuras_resumen[
    "evidencias_por_fase"
].update_layout(
    height=450,
    legend_title_text="Tipo de evidencia"
)

# Actualizar las copias exportadas de esta gráfica.

for ruta in RESULTADOS.rglob(
    "evidencias_por_fase.html"
):

    figuras_resumen[
        "evidencias_por_fase"
    ].write_html(ruta)

for ruta in RESULTADOS.rglob(
    "evidencias_por_fase.json"
):

    figuras_resumen[
        "evidencias_por_fase"
    ].write_json(ruta)

# ==========================================================
# 3. ACTUALIZAR INVENTARIO DE NAVEGACIÓN
# ==========================================================

df_inventario_vistas = df_inventario_final.copy()

df_inventario_vistas.to_csv(
    RESULTADOS / "inventario_vistas_dashboard.csv",
    index=False,
    encoding="utf-8-sig"
)

resultados_navegacion_final = []

for nombre, funcion in vistas_dashboard.items():

    componentes = list(
        recorrer_componentes(funcion())
    )

    graficas = sum(
        isinstance(c, dcc.Graph)
        for c in componentes
    )

    esperadas = int(
        df_inventario_final.loc[
            df_inventario_final["Vista"] == nombre,
            "Graficas"
        ].iloc[0]
    )

    assert graficas == esperadas

    resultados_navegacion_final.append({
        "Pestana": nombre,
        "Graficas_esperadas": esperadas,
        "Graficas_recuperadas": graficas,
        "Estado": "OK"
    })

df_pruebas_navegacion = pd.DataFrame(
    resultados_navegacion_final
)

df_pruebas_navegacion.to_csv(
    CARPETA_VALIDACIONES / "pruebas_navegacion.csv",
    index=False,
    encoding="utf-8-sig"
)

# ==========================================================
# 4. ACTUALIZAR LAYOUT DEL DASHBOARD
# ==========================================================

app.layout = html.Div([

    encabezado_dashboard,

    dbc.Container([

        navegacion_dashboard,

        dcc.Loading(
            html.Div(
                id="contenido-dashboard",
                children=vista_resumen()
            ),
            type="circle"
        ),

        pie_dashboard

    ], fluid=True, className="px-3 px-lg-4")

], style={
    "backgroundColor": ESTILO_FONDO,
    "minHeight": "100vh",
    "fontFamily": "Arial, sans-serif",
    "color": ESTILO_TEXTO
})

# ==========================================================
# 5. DOCUMENTAR ENTORNO DE EJECUCIÓN
# ==========================================================

paquetes = [
    "pandas",
    "numpy",
    "scipy",
    "scikit-learn",
    "plotly",
    "dash",
    "dash-bootstrap-components",
    "requests"
]

versiones = {
    paquete: version(paquete)
    for paquete in paquetes
}

entorno_final = {
    "Python": platform.python_version(),
    "Sistema_operativo": platform.platform(),
    "Fecha_exportacion": datetime.now(
        ZoneInfo("America/Bogota")
    ).isoformat(),
    "Semilla": SEMILLA,
    "Bibliotecas": versiones
}

(RESULTADOS / "entorno_ejecucion.json").write_text(
    json.dumps(
        entorno_final,
        indent=4,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

(BASE_DASHBOARD / "requirements.txt").write_text(
    "\n".join(
        f"{paquete}=={numero_version}"
        for paquete, numero_version
        in versiones.items()
    ) + "\n",
    encoding="utf-8"
)

# ==========================================================
# 6. VERIFICAR IDENTIDAD DEL DATASET
# ==========================================================

hash_dataset = hashlib.sha256(
    RUTA_CSV.read_bytes()
).hexdigest()

hash_esperado = (
    "a6bcf826782d3c0fbfdcbeead17cd0884"
    "185a0dafe8ff10cd48a874ee7ba18be"
)

assert hash_dataset == hash_esperado

identidad_dataset = {
    "Nombre": "Bike Sharing Dataset",
    "Archivo": "day.csv",
    "DOI": "10.24432/C5W894",
    "Licencia": "CC BY 4.0",
    "Registros": len(df),
    "Variables": df.shape[1],
    "SHA256": hash_dataset,
    "URL": URL_DATASET
}

(RESULTADOS / "identidad_dataset.json").write_text(
    json.dumps(
        identidad_dataset,
        indent=4,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ==========================================================
# 7. GENERAR DOCUMENTACIÓN DE REPRODUCIBILIDAD
# ==========================================================

readme_final = """
# BIKE SHARING ANALYTICS

## Dashboard global - Proyecto Integrador

Universidad de La Salle
Maestría en Inteligencia Artificial
Asignatura: Ciencia de Datos
Estudiante: Adriana Patricia Fernández Infante

## Descripción

Dashboard interactivo desarrollado con Python, Dash,
Plotly y Bootstrap para integrar las cuatro fases
del Proyecto Integrador de Ciencia de Datos.

## Componentes

- 5 pestañas de navegación.
- 47 gráficas interactivas.
- 41 tablas interactivas.
- 25 imágenes originales.
- 3 filtros dinámicos.
- 2 callbacks registrados.
- 23 pruebas funcionales automatizadas.

## Dataset

Bike Sharing Dataset - UCI Machine Learning Repository.
Archivo utilizado: day.csv.
731 observaciones diarias.
16 variables originales.
Periodo: 2011-2012.
Licencia: Creative Commons Attribution 4.0 International.

DOI: https://doi.org/10.24432/C5W894

## Reproducción en Google Colab

1. Abrir el notebook principal del dashboard.
2. Conectar un entorno de ejecución.
3. Ejecutar las celdas en orden.
4. Cuando la celda 4 lo solicite, cargar los archivos ZIP
   originales incluidos en evidencias/archivos_originales.
5. Ejecutar las celdas de consolidación e integración.
6. Después de la celda 20, ejecutar nuevamente la celda 17
   para visualizar la versión definitiva del dashboard.

## Archivos requeridos por el notebook

- fase 2 y fase 3.zip
- Evidencias_Fase_4_Bicicletas.zip

## Procedencia de resultados

Fase 1:
EDA reconstruido desde el dataset original debido a
la ausencia del notebook inicial.

Fases 2 y 3:
Resultados recuperados de los notebooks originales.

Fase 4:
Resultados recuperados desde las evidencias originales.

No se reentrenaron modelos durante la consolidación.

## Consideraciones metodológicas

Las métricas de clasificación de las fases 3 y 4
corresponden a diferentes protocolos de partición.

No deben interpretarse como una comparación directa
del desempeño de los modelos.

El umbral de clasificación de Fase 4 fue heredado de
Fase 3, con posible solapamiento de observaciones.

## Disponibilidad de la aplicación

El servidor Dash ejecutado desde Google Colab es temporal.
Para volver a visualizarlo, se debe ejecutar el notebook.

Las gráficas HTML exportadas permiten consultar
visualizaciones individuales fuera de Colab.

## Referencias
"""

readme_final += (
    "\n" + referencia_dataset
    + "\n\n" + referencia_articulo
    + "\n"
)

(BASE_DASHBOARD / "README.md").write_text(
    readme_final.strip(),
    encoding="utf-8"
)

# ==========================================================
# 8. VALIDACIONES FINALES DE LA APLICACIÓN
# ==========================================================

assert len(vistas_dashboard) == 5

assert df_inventario_final["Graficas"].sum() == 47

assert df_inventario_final["Tablas"].sum() == 41

assert df_inventario_final["Imagenes"].sum() == 25

assert len(app.callback_map) == 2

assert len(df_pruebas_filtros) == 18

assert len(df_pruebas_navegacion) == 5

assert int(df["cnt"].sum()) == 3292679

assert inventario_fases[
    "Graficas_interactivas"
].sum() == 39

assert len(figuras_resumen) == 8

with app.server.test_client() as cliente:

    estados_http = {
        ruta: cliente.get(ruta).status_code
        for ruta in [
            "/",
            "/_dash-layout",
            "/_dash-dependencies"
        ]
    }

assert all(
    estado == 200
    for estado in estados_http.values()
)

# ==========================================================
# 9. EXPORTAR INVENTARIO Y VALIDACIONES
# ==========================================================

validacion_final = {
    "Pestanas": 5,
    "Graficas_interactivas": 47,
    "Tablas_interactivas": 41,
    "Imagenes_originales": 25,
    "Componentes_interfaz": int(
        df_inventario_final["Componentes"].sum()
    ),
    "Callbacks": len(app.callback_map),
    "Pruebas_filtros": len(df_pruebas_filtros),
    "Pruebas_navegacion": len(df_pruebas_navegacion),
    "Modelos_reentrenados": 0,
    "Identidad_dataset": "OK",
    "Licencia": "CC BY 4.0",
    "Endpoints": estados_http
}

(RESULTADOS / "validacion_final_dashboard.json").write_text(
    json.dumps(
        validacion_final,
        indent=4,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ==========================================================

print("Dashboard integrado correctamente.")
print("Resultados:", RESULTADOS)
print("Empaquetado y descarga manual omitidos.")



## Validación definitiva del dashboard

Se verifican los componentes integrados y los
endpoints de Dash sin iniciar un servidor persistente.

El dashboard recupera evidencias académicas originales.
No vuelve a entrenar los modelos de las fases anteriores.


In [ ]:

assert len(vistas_dashboard) == 5

assert df_inventario_final["Graficas"].sum() == 47

assert df_inventario_final["Tablas"].sum() == 41

assert df_inventario_final["Imagenes"].sum() == 25

assert len(app.callback_map) == 2

assert len(df_pruebas_filtros) == 18

assert len(df_pruebas_navegacion) == 5

endpoints = {}

with app.server.test_client() as cliente:

    for endpoint in [
        "/",
        "/_dash-layout",
        "/_dash-dependencies"
    ]:

        respuesta = cliente.get(endpoint)

        endpoints[endpoint] = respuesta.status_code

        assert respuesta.status_code == 200

resumen_dashboard = {
    "pestanas": len(vistas_dashboard),
    "graficas": int(
        df_inventario_final["Graficas"].sum()
    ),
    "tablas": int(
        df_inventario_final["Tablas"].sum()
    ),
    "imagenes": int(
        df_inventario_final["Imagenes"].sum()
    ),
    "callbacks": len(app.callback_map),
    "pruebas_filtros": len(df_pruebas_filtros),
    "pruebas_navegacion": len(df_pruebas_navegacion),
    "endpoints": endpoints,
    "modelos_reentrenados": 0
}

import json

(RESULTADOS / "validacion_dashboard.json").write_text(
    json.dumps(
        resumen_dashboard,
        indent=4,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

for clave, valor in resumen_dashboard.items():

    print(f"{clave}: {valor}")

print("Validación del dashboard: OK")



## Visualización interactiva

Para iniciar el dashboard, cambiar MOSTRAR_DASHBOARD
a True y ejecutar la siguiente celda.

Se mantiene desactivado por defecto para permitir
la ejecución automatizada del notebook.


In [ ]:

MOSTRAR_DASHBOARD = False

if MOSTRAR_DASHBOARD:

    app.run(
        jupyter_mode="inline",
        jupyter_height=1200,
        port=8050,
        debug=False
    )

else:

    print("Dashboard preparado para visualización.")
    print("Cambiar MOSTRAR_DASHBOARD a True para iniciar.")
